# NBA_ML_project

This is the final class-facing notebook for the GitHub submission package. It uses NBA seasons 2021-2025 and follows the supervised-learning workflow from the course while preserving time-aware fantasy-basketball modeling rules.


## 1. Objective And Target Definition

Goal: predict player-game fantasy basketball points before the game is played.

Target:

`fantasy_points = points + 1.2 * reboundsTotal + 1.5 * assists + 3 * steals + 3 * blocks - turnovers`

Prediction grain: one player in one regular-season NBA game.

Leakage rule: current-game box-score outcomes are not valid features. They may only be used to create the target or shifted historical rolling features. Diagnostic leakage models are clearly labeled and are not candidates for final recommendation.


## 2. Imports, Paths, And Constants

This setup cell defines the final run folder, downloads the Kaggle dataset through `kagglehub`, sets the 2021-2025 season scope, and fixes random seeds/thread counts for reproducibility.


In [1]:
import os
from pathlib import Path
from time import perf_counter
from dataclasses import dataclass
from io import StringIO

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import kagglehub
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.decomposition import PCA

KAGGLE_DATASET = "eoinamoore/historical-nba-data-and-player-box-scores"

# Input: start Path; Output: project root Path. Walks upward from the current folder to find the project root using pyproject.toml.
# The raw data is downloaded separately from Kaggle so new users do not need to prepare a local raw-data folder.
def find_project_root(start: Path) -> Path:
    # Find the NBA_ML_PROJ_2 folder even when Jupyter starts inside runs/final/notebooks.
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
        if candidate.name == "NBA_ML_PROJ_2":
            return candidate
    raise FileNotFoundError("Could not find the project root. Start Jupyter from the NBA_ML_PROJ_2 project folder.")

PROJECT_ROOT = find_project_root(Path.cwd())
RUN_DIR = PROJECT_ROOT / "runs" / "final"
KAGGLE_CACHE_DIR = RUN_DIR / "kagglehub_cache"
os.environ.setdefault("KAGGLEHUB_CACHE", str(KAGGLE_CACHE_DIR))
KAGGLE_DATA_DIR = Path(kagglehub.dataset_download(KAGGLE_DATASET))
RAW_DIR = KAGGLE_DATA_DIR
PROCESSED_DIR = RUN_DIR / "data" / "processed"
REPORTS_DIR = RUN_DIR / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
STEP_SUMMARIES_DIR = REPORTS_DIR / "step_summaries"
NOTEBOOKS_DIR = RUN_DIR / "notebooks"
PRESENTATION_DIR = RUN_DIR / "presentation"

for directory in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, STEP_SUMMARIES_DIR, NOTEBOOKS_DIR, PRESENTATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

for folder in [TABLES_DIR, FIGURES_DIR, STEP_SUMMARIES_DIR]:
    for pattern in ["*starter_minutes_leakage_diagnostic*", "*replication_11_leakage_diagnostic*"]:
        for obsolete_path in folder.glob(pattern):
            if obsolete_path.is_file():
                obsolete_path.unlink()

TARGET_COMPONENTS = ["points", "reboundsTotal", "assists", "steals", "blocks", "turnovers"]
TARGET_WEIGHTS = {"points": 1.0, "reboundsTotal": 1.2, "assists": 1.5, "steals": 3.0, "blocks": 3.0, "turnovers": -1.0}
LAST4_SEASON_STARTS = [2021, 2022, 2023, 2024, 2025]
SPLIT_COL = "split_2021_train_2024_validation_2025_test"
TARGET_COL = "fantasy_points"
MODEL_RANDOM_STATE = 42

print("Season starts:", LAST4_SEASON_STARTS)
print("Split column:", SPLIT_COL)
print("Run folder:", RUN_DIR)
print("Kaggle cache folder:", KAGGLE_CACHE_DIR)
print("Kaggle dataset:", KAGGLE_DATASET)
print("Raw data folder:", RAW_DIR)


C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0.00/1.03G [00:00<?, ?B/s]

  0%|          | 1.00M/1.03G [00:00<15:58, 1.16MB/s]

  0%|          | 2.00M/1.03G [00:01<07:51, 2.35MB/s]

  0%|          | 4.00M/1.03G [00:01<03:39, 5.03MB/s]

...

100%|█████████▉| 1.03G/1.03G [00:39<00:00, 29.3MB/s]

100%|█████████▉| 1.03G/1.03G [00:39<00:00, 29.9MB/s]

100%|██████████| 1.03G/1.03G [00:39<00:00, 27.8MB/s]

Extracting files...


Season starts: [2021, 2022, 2023, 2024, 2025]
Split column: split_2021_train_2024_validation_2025_test
Run folder: C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\runs\final
Kaggle cache folder: C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\runs\final\kagglehub_cache
Kaggle dataset: eoinamoore/historical-nba-data-and-player-box-scores
Raw data folder: C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\runs\final\kagglehub_cache\datasets\eoinamoore\historical-nba-data-and-player-box-scores\versions\479


## 3. Raw Data Collection And Inventory

The raw data is downloaded directly from Kaggle with `kagglehub`. This removes the manual requirement to place CSV files in a local raw-data folder before running the notebook.

Dataset: `eoinamoore/historical-nba-data-and-player-box-scores`

This section inventories the downloaded tables, columns, row counts, key-like columns, missingness, and which raw tables can support the target.


In [2]:
from io import StringIO
from pathlib import Path

import pandas as pd



KEY_COLUMN_CANDIDATES = {
    "game": {"gameId", "game_id", "GAME_ID", "GameID"},
    "player": {"personId", "playerId", "player_id", "PLAYER_ID", "PersonID"},
    "team": {"teamId", "team_id", "TEAM_ID", "TeamID"},
    "opponent": {"opponentTeamId", "opponentteamId", "oppTeamId", "opponent_team_id"},
    "date": {"gameDate", "game_date", "gameDateTimeEst", "GAME_DATE", "date"},
}


SUPPORTED_RAW_SUFFIXES = {".csv", ".parquet"}

# Input: raw_dir Path; Output: sorted list of supported raw table file Paths. Scans the Kaggle download folder for CSV or Parquet files in deterministic order.
def list_raw_table_files(raw_dir: Path = RAW_DIR) -> list[Path]:
    # Return supported raw table files in deterministic order.
    if not raw_dir.exists():
        return []
    return sorted(path for path in raw_dir.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_RAW_SUFFIXES)


def list_raw_csv_files(raw_dir: Path = RAW_DIR) -> list[Path]:
    # Return raw CSV files in deterministic order.
    return [path for path in list_raw_table_files(raw_dir) if path.suffix.lower() == ".csv"]


def table_name_from_path(path: Path) -> str:
    # Convert a raw table path into a stable table name.
    return path.stem


def resolve_raw_table_path(table_name_or_path: str | Path, raw_dir: Path = RAW_DIR) -> Path:
    # Resolve a table name, file name, or path to a raw table path.
    candidate = Path(table_name_or_path)
    if candidate.exists():
        return candidate

    raw_candidate = raw_dir / str(table_name_or_path)
    if raw_candidate.exists():
        return raw_candidate

    name = str(table_name_or_path)
    if Path(name).suffix.lower() not in SUPPORTED_RAW_SUFFIXES:
        for suffix in sorted(SUPPORTED_RAW_SUFFIXES):
            raw_candidate = raw_dir / f"{name}{suffix}"
            if raw_candidate.exists():
                return raw_candidate

    normalized = Path(name).stem.lower()
    for path in list_raw_table_files(raw_dir):
        if path.stem.lower() == normalized or path.name.lower() == name.lower():
            return path

    available = ", ".join(path.name for path in list_raw_table_files(raw_dir)) or "none"
    raise FileNotFoundError(f"Raw table '{table_name_or_path}' was not found in {raw_dir}. Available raw tables: {available}")


def _read_table(path: Path, columns: list[str] | None = None, nrows: int | None = None) -> pd.DataFrame:
    # Read a CSV or Parquet raw table with optional column and row limits.
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, usecols=columns, nrows=nrows, low_memory=False)
    if suffix == ".parquet":
        if nrows is None:
            return pd.read_parquet(path, columns=columns)

        import pyarrow.parquet as pq

        parquet_file = pq.ParquetFile(path)
        first_batch = next(parquet_file.iter_batches(batch_size=nrows, columns=columns), None)
        if first_batch is None:
            return pd.DataFrame(columns=columns or [])
        return first_batch.to_pandas()
    raise ValueError(f"Unsupported raw table suffix: {path.suffix}")


def _raw_table_metadata(path: Path) -> tuple[list[str] | None, int | None, str | None]:
    # Return file-level metadata for a raw table path without loading the full table.
    try:
        if path.suffix.lower() == ".csv":
            header = pd.read_csv(path, nrows=0)
            return list(header.columns), count_csv_rows(path), None

        import pyarrow.parquet as pq

        parquet_file = pq.ParquetFile(path)
        return parquet_file.schema.names, parquet_file.metadata.num_rows, None
    except Exception as exc:
        return None, None, f"{type(exc).__name__}: {exc}"


def load_raw_table(
    table_name_or_path: str | Path,
    columns: list[str] | None = None,
    nrows: int | None = None,
    raw_dir: Path = RAW_DIR,
) -> pd.DataFrame:
    # Load one raw table without mutating it.
    path = resolve_raw_table_path(table_name_or_path, raw_dir=raw_dir)
    return _read_table(path, columns=columns, nrows=nrows)


def count_csv_rows(path: Path) -> int:
    # Count data rows in a CSV file, excluding the header.
    with path.open("rb") as file:
        return max(sum(1 for _ in file) - 1, 0)


def detect_suspected_keys(columns: list[str] | pd.Index) -> str:
    # Return key-like columns found in a table.
    return ", ".join(f"{role}:{column}" for role, column in suspected_key_columns(columns))


def suspected_key_columns(columns: list[str] | pd.Index) -> list[tuple[str, str]]:
    # Return key-like columns as (role, column) pairs.
    column_set = set(map(str, columns))
    found = []
    for role, candidates in KEY_COLUMN_CANDIDATES.items():
        matches = sorted(column_set.intersection(candidates))
        for match in matches:
            found.append((role, match))
    return found


def raw_table_inventory(raw_dir: Path = RAW_DIR, count_rows: bool = False) -> pd.DataFrame:
    # Summarize every supported table currently available in the Kaggle download folder.
    rows = []
    for path in list_raw_table_files(raw_dir):
        columns, detected_rows, load_error = _raw_table_metadata(path)
        row_count = detected_rows if count_rows or path.suffix.lower() == ".parquet" else pd.NA

        rows.append(
            {
                "table_name": table_name_from_path(path),
                "file_name": path.name,
                "file_type": path.suffix.lower().lstrip("."),
                "exists": True,
                "row_count": row_count,
                "column_count": len(columns) if columns is not None else pd.NA,
                "size_mb": round(path.stat().st_size / 1024 / 1024, 3),
                "suspected_keys": detect_suspected_keys(columns or []),
                "sample_columns": ", ".join(map(str, (columns or [])[:20])),
                "load_error": load_error or "",
            }
        )

    columns = [
        "table_name",
        "file_name",
        "file_type",
        "exists",
        "row_count",
        "column_count",
        "size_mb",
        "suspected_keys",
        "sample_columns",
        "load_error",
    ]
    return pd.DataFrame(rows, columns=columns)


def dataframe_info_summary(df: pd.DataFrame) -> str:
    # Return the text form of DataFrame.info() for notebook display.
    buffer = StringIO()
    df.info(buf=buffer)
    return buffer.getvalue()


def _profile_value(value) -> object:
    # Return a hashable representation for raw profiling output.
    try:
        hash(value)
    except TypeError:
        if hasattr(value, "tolist"):
            value = value.tolist()
        return repr(value)
    return value


def _safe_unique_count(series: pd.Series) -> int:
    # Count unique values, falling back for array/list-like raw values.
    try:
        return int(series.nunique(dropna=True))
    except TypeError:
        return int(series.dropna().map(_profile_value).nunique(dropna=True))


def profile_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Return a notebook-friendly per-column profile for one DataFrame.
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing_count_profiled": df.isna().sum(),
            "missing_pct_profiled": df.isna().mean().round(4),
            "unique_count_profiled": [_safe_unique_count(df[column]) for column in df.columns],
        },
        index=df.columns,
    )


def table_column_summary(raw_dir: Path = RAW_DIR, sample_rows: int | None = None) -> pd.DataFrame:
    # Create a column-level profile for every supported raw table.
    #
    # When sample_rows is None, the full file is loaded and missing/unique counts
    # are exact. Set sample_rows to an integer if a raw file is too large to load.
    rows = []
    for path in list_raw_table_files(raw_dir):
        _, total_rows, metadata_error = _raw_table_metadata(path)
        if metadata_error:
            rows.append(
                {
                    "table_name": table_name_from_path(path),
                    "file_name": path.name,
                    "column": "",
                    "dtype": "",
                    "row_count": pd.NA,
                    "profiled_rows": 0,
                    "missing_count": pd.NA,
                    "missing_pct_profiled": pd.NA,
                    "unique_count_profiled": pd.NA,
                    "is_suspected_key": False,
                    "sample_values": "",
                    "load_error": metadata_error,
                }
            )
            continue

        try:
            df = _read_table(path, nrows=sample_rows)
        except Exception as exc:
            rows.append(
                {
                    "table_name": table_name_from_path(path),
                    "file_name": path.name,
                    "column": "",
                    "dtype": "",
                    "row_count": pd.NA,
                    "profiled_rows": 0,
                    "missing_count": pd.NA,
                    "missing_pct_profiled": pd.NA,
                    "unique_count_profiled": pd.NA,
                    "is_suspected_key": False,
                    "sample_values": "",
                    "load_error": f"{type(exc).__name__}: {exc}",
                }
            )
            continue
        suspected_keys = {column for _, column in suspected_key_columns(df.columns)}
        profile = profile_dataframe_columns(df)

        for column in df.columns:
            series = df[column]
            non_null = series.dropna()
            sample_values = non_null.astype(str).head(5).tolist()
            rows.append(
                {
                    "table_name": table_name_from_path(path),
                    "file_name": path.name,
                    "column": column,
                    "dtype": profile.loc[column, "dtype"],
                    "row_count": total_rows,
                    "profiled_rows": len(df),
                    "missing_count": int(profile.loc[column, "missing_count_profiled"]),
                    "missing_pct_profiled": float(profile.loc[column, "missing_pct_profiled"]),
                    "unique_count_profiled": int(profile.loc[column, "unique_count_profiled"]),
                    "is_suspected_key": column in suspected_keys,
                    "sample_values": " | ".join(sample_values),
                    "load_error": "",
                }
            )

    columns = [
        "table_name",
        "file_name",
        "column",
        "dtype",
        "row_count",
        "profiled_rows",
        "missing_count",
        "missing_pct_profiled",
        "unique_count_profiled",
        "is_suspected_key",
        "sample_values",
        "load_error",
    ]
    return pd.DataFrame(rows, columns=columns)


def candidate_base_tables(column_summary: pd.DataFrame) -> pd.DataFrame:
    # Find raw tables that could support one row per player-game target creation.
    if column_summary.empty:
        return pd.DataFrame(
            columns=[
                "table_name",
                "has_game_id",
                "has_player_id",
                "has_target_components",
                "target_components_present",
                "target_components_missing",
            ]
        )

    rows = []
    for table_name, group in column_summary.groupby("table_name", sort=True):
        columns = set(group["column"].astype(str))
        target_present = [column for column in TARGET_COMPONENTS if column in columns]
        target_missing = [column for column in TARGET_COMPONENTS if column not in columns]
        rows.append(
            {
                "table_name": table_name,
                "has_game_id": bool({"gameId", "game_id", "GAME_ID"}.intersection(columns)),
                "has_player_id": bool({"personId", "playerId", "PLAYER_ID", "PersonID"}.intersection(columns)),
                "has_target_components": not target_missing,
                "target_components_present": ", ".join(target_present),
                "target_components_missing": ", ".join(target_missing),
            }
        )
    return pd.DataFrame(rows)


## 3.1 Raw Data Inventory Execution

This cell runs the raw-data inventory process for all source files downloaded from Kaggle.

It creates three summary tables:

- `inventory`: a table-level summary of the raw files, including file name, file type, size, column count, suspected key columns, and loading errors if any.
- `column_summary`: a column-level profile for each raw table, based on up to 100,000 sampled rows per file. It summarizes data types, missing values, unique counts, suspected key columns, and sample values.
- `base_candidates`: a target-readiness summary that identifies which raw tables contain player/game identifiers and the required target components for calculating `fantasy_points`.

The generated summaries are saved as CSV files under `runs/final/reports/tables`, and the notebook displays the raw-file inventory and the most relevant candidate base tables for target creation.


In [3]:
inventory = raw_table_inventory(raw_dir=RAW_DIR)
column_summary = table_column_summary(raw_dir=RAW_DIR, sample_rows=100_000)
base_candidates = candidate_base_tables(column_summary)

inventory.to_csv(TABLES_DIR / "raw_table_inventory.csv", index=False)
column_summary.to_csv(TABLES_DIR / "table_column_summary.csv", index=False)
base_candidates.to_csv(TABLES_DIR / "candidate_base_tables.csv", index=False)

display(inventory)
display(base_candidates.sort_values(["has_target_components", "has_game_id", "has_player_id"], ascending=False))


,table_name,file_name,file_type,exists,row_count,column_count,size_mb,suspected_keys,sample_columns,load_error
0,Games,Games.csv,csv,True,<NA>,23,10.684,"game:gameId, date:gameDate, date:gameDateTimeEst","gameId, gameDateTimeEst, hometeamCity, hometea...",
1,LeagueSchedule24_25,LeagueSchedule24_25.csv,csv,True,<NA>,15,0.140,"game:gameId, date:gameDateTimeEst","gameId, gameDateTimeEst, gameDay, arenaCity, a...",
2,LeagueSchedule25_26,LeagueSchedule25_26.csv,csv,True,<NA>,17,0.179,"game:gameId, date:gameDateTimeEst","gameId, gameDateTimeEst, gameDay, homeTeamId, ...",
3,PlayByPlay,PlayByPlay.parquet,parquet,True,18713301,89,889.259,"game:gameId, player:personId, team:teamId, opp...","clock, actionType, description, playerFullName...",
4,Players,Players.csv,csv,True,<NA>,20,0.500,player:personId,"personId, firstName, lastName, birthDate, scho...",
5,PlayerStatistics,PlayerStatistics.csv,csv,True,<NA>,40,371.467,"game:gameId, player:personId, opponent:opponen...","firstName, lastName, personId, gameId, gameDat...",
6,PlayerStatisticsExtended,PlayerStatisticsExtended.csv,csv,True,<NA>,110,432.047,"game:gameId, player:personId, opponent:opponen...","firstName, lastName, personId, gameId, gameDat...",
7,TeamHistories,TeamHistories.csv,csv,True,<NA>,7,0.007,team:teamId,"teamId, teamCity, teamName, teamAbbrev, season...",
8,TeamStatistics,TeamStatistics.csv,csv,True,<NA>,59,34.339,"game:gameId, team:teamId, opponent:opponentTea...","gameId, gameDateTimeEst, teamCity, teamName, t...",
9,TeamStatisticsExtended,TeamStatisticsExtended.csv,csv,True,<NA>,104,36.396,"game:gameId, team:teamId, opponent:opponentTea...","gameId, gameDateTimeEst, gameType, gameLabel, ...",


,table_name,has_game_id,has_player_id,has_target_components,target_components_present,target_components_missing
4,PlayerStatistics,True,True,True,"points, reboundsTotal, assists, steals, blocks...",
5,PlayerStatisticsExtended,True,True,True,"points, reboundsTotal, assists, steals, blocks...",
3,PlayByPlay,True,True,False,,"points, reboundsTotal, assists, steals, blocks..."
0,Games,True,False,False,,"points, reboundsTotal, assists, steals, blocks..."
1,LeagueSchedule24_25,True,False,False,,"points, reboundsTotal, assists, steals, blocks..."
2,LeagueSchedule25_26,True,False,False,,"points, reboundsTotal, assists, steals, blocks..."
8,TeamStatistics,True,False,False,"reboundsTotal, assists, steals, blocks, turnovers",points
9,TeamStatisticsExtended,True,False,False,"reboundsTotal, assists, steals, blocks, turnovers",points
6,Players,False,True,False,,"points, reboundsTotal, assists, steals, blocks..."
7,TeamHistories,False,False,False,,"points, reboundsTotal, assists, steals, blocks..."


## 4. Data Preparation And Table Unification

This section builds one player-game source table by joining player box-score rows, player profile fields, own-team context, and opponent-team context.

The main 2021 adjustment is here: missing 2021 `gameType` values in `TeamStatisticsExtended` are filled from `Games.csv` before filtering to regular season.


In [4]:
import numpy as np
import pandas as pd


ROLLING_WINDOWS = [3, 5, 10, 30]

PLAYER_ROLLING_PLAN = {
    "fantasy_points": ROLLING_WINDOWS,
    "current_is_starter": [5, 10, 30],
    "starter_minutes": ROLLING_WINDOWS,
    "bench_minutes": [5, 10, 30],
    "numMinutes": ROLLING_WINDOWS,
    "points": [5, 10, 30],
    "reboundsTotal": [5, 10, 30],
    "assists": [5, 10, 30],
    "steals": [5, 10, 30],
    "blocks": [5, 10, 30],
    "turnovers": [5, 10, 30],
    "usagePercentage": ROLLING_WINDOWS,
    "trueShootingPercentage": [5, 10, 30],
    "effectiveFieldGoalPercentage": [5, 10, 30],
    "fieldGoalsPercentage": [5, 10, 30],
    "threePointersPercentage": [5, 10, 30],
    "freeThrowsPercentage": [5, 10, 30],
    "playerImpactEstimate": [5, 10, 30],
    "offensiveRating": [5, 10, 30],
    "defensiveRating": [5, 10, 30],
    "estimatedDefensiveRating": [5, 10, 30],
    "pace": [5, 10, 30],
}

TEAM_ROLLING_PLAN = {
    "team_win": [5, 10, 30],
    "team_score": [5, 10, 30],
    "team_points_allowed": [5, 10, 30],
    "team_pace": [5, 10, 30],
    "team_offensiveRating": [5, 10, 30],
    "team_defensiveRating": [5, 10, 30],
    "team_estimatedDefensiveRating": [5, 10, 30],
    "team_fieldGoalsPercentage": [5, 10, 30],
    "team_threePointersPercentage": [5, 10, 30],
    "team_effectiveFieldGoalPercentage": [5, 10, 30],
    "team_trueShootingPercentage": [5, 10, 30],
    "team_assists": [5, 10, 30],
    "team_reboundsTotal": [5, 10, 30],
    "team_steals": [5, 10, 30],
    "team_blocks": [5, 10, 30],
    "team_turnovers": [5, 10, 30],
    "team_pointsInThePaint": [5, 10, 30],
    "team_pointsFastBreak": [5, 10, 30],
    "team_pointsSecondChance": [5, 10, 30],
    "team_pointsFromTurnovers": [5, 10, 30],
}

OPPONENT_ROLLING_PLAN = {
    "opp_win": [5, 10, 30],
    "opp_points_allowed": [5, 10, 30],
    "opp_pace": [5, 10, 30],
    "opp_defensiveRating": [5, 10, 30],
    "opp_estimatedDefensiveRating": [5, 10, 30],
    "opp_fieldGoalsPercentage_allowed": [5, 10, 30],
    "opp_effectiveFieldGoalPercentage_allowed": [5, 10, 30],
    "opp_trueShootingPercentage_allowed": [5, 10, 30],
    "opp_rebounds_allowed": [5, 10, 30],
    "opp_assists_allowed": [5, 10, 30],
    "opp_steals": [5, 10, 30],
    "opp_blocks": [5, 10, 30],
    "opp_turnovers_forced": [5, 10, 30],
    "opp_fp_allowed": [5, 10, 30],
    "opp_pointsInThePaint_allowed": [5, 10, 30],
    "opp_pointsFastBreak_allowed": [5, 10, 30],
    "opp_pointsSecondChance_allowed": [5, 10, 30],
    "opp_pointsFromTurnovers_allowed": [5, 10, 30],
}

OPPONENT_POSITION_ROLLING_PLAN = {
    "opp_pos_fp_allowed": ROLLING_WINDOWS,
    "opp_pos_fp_allowed_per_player": ROLLING_WINDOWS,
    "opp_pos_minutes_allowed": [5, 10, 30],
    "opp_pos_player_count": [5, 10, 30],
    "opp_pos_starter_count": [5, 10, 30],
}


def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    # Divide two values while returning NaN when the denominator is zero or missing.
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator


def add_lagged_rolling_features(
    df: pd.DataFrame,
    group_col: str,
    date_col: str,
    value_cols: list[str],
    windows: list[int],
    min_periods: int = 1,
) -> pd.DataFrame:
    # Add rolling means that use only rows before the current row.
    result = df.sort_values([group_col, date_col]).copy()

    for value_col in value_cols:
        if value_col not in result.columns:
            continue
        for window in windows:
            result[f"{value_col}_roll_{window}"] = result.groupby(group_col)[value_col].transform(
                lambda values: values.shift(1).rolling(window=window, min_periods=min_periods).mean()
            )

    return result.sort_index()


def add_lagged_rolling_summary_features(
    df: pd.DataFrame,
    group_col: str,
    date_col: str,
    value_cols: list[str],
    windows: list[int] | None = None,
    min_periods: int = 1,
) -> pd.DataFrame:
    # Add shifted rolling mean, momentum, and z-score style features.
    #
    # For every value column, this creates:
    # - `{col}_roll_5`, `{col}_roll_10`, `{col}_roll_30`
    # - `{col}_std_30`
    # - `{col}_momentum_5_vs_10`, `{col}_momentum_10_vs_30`
    # - `{col}_z_5_vs_30`, `{col}_z_10_vs_30`
    #
    # All source values are shifted by one row inside each group before rolling.
    windows = windows or ROLLING_WINDOWS
    result = df.sort_values([group_col, date_col]).copy()

    for value_col in value_cols:
        if value_col not in result.columns:
            continue

        source = pd.to_numeric(result[value_col], errors="coerce")
        shifted = source.groupby(result[group_col]).shift(1)

        for window in windows:
            result[f"{value_col}_roll_{window}"] = shifted.groupby(result[group_col]).transform(
                lambda values: values.rolling(window=window, min_periods=min_periods).mean()
            )

        long_window = max(windows)
        result[f"{value_col}_std_{long_window}"] = shifted.groupby(result[group_col]).transform(
            lambda values: values.rolling(window=long_window, min_periods=2).std()
        )

        if {5, 10}.issubset(windows):
            result[f"{value_col}_momentum_5_vs_10"] = result[f"{value_col}_roll_5"] - result[f"{value_col}_roll_10"]
        if {3, 10}.issubset(windows):
            result[f"{value_col}_momentum_3_vs_10"] = result[f"{value_col}_roll_3"] - result[f"{value_col}_roll_10"]
        if {10, 30}.issubset(windows):
            result[f"{value_col}_momentum_10_vs_30"] = result[f"{value_col}_roll_10"] - result[f"{value_col}_roll_30"]
        if {5, 30}.issubset(windows):
            result[f"{value_col}_z_5_vs_30"] = _safe_divide(
                result[f"{value_col}_roll_5"] - result[f"{value_col}_roll_30"],
                result[f"{value_col}_std_{long_window}"],
            )
        if {10, 30}.issubset(windows):
            result[f"{value_col}_z_10_vs_30"] = _safe_divide(
                result[f"{value_col}_roll_10"] - result[f"{value_col}_roll_30"],
                result[f"{value_col}_std_{long_window}"],
            )

    return result.sort_index()


def add_lagged_rolling_plan(
    df: pd.DataFrame,
    group_col: str,
    date_col: str,
    rolling_plan: dict[str, list[int]],
    min_periods: int = 1,
) -> pd.DataFrame:
    # Apply a column-to-window rolling plan with shift(1) leakage protection.
    result = df.copy()
    result[date_col] = pd.to_datetime(result[date_col], errors="coerce")

    for value_col, windows in rolling_plan.items():
        result = add_lagged_rolling_summary_features(
            result,
            group_col=group_col,
            date_col=date_col,
            value_cols=[value_col],
            windows=windows,
            min_periods=min_periods,
        )

    return result


def add_player_rolling_features(
    df: pd.DataFrame,
    player_col: str = "personId",
    date_col: str = "game_date",
    rolling_plan: dict[str, list[int]] | None = None,
) -> pd.DataFrame:
    # Add player-level shifted rolling features.
    return add_lagged_rolling_plan(
        df=df,
        group_col=player_col,
        date_col=date_col,
        rolling_plan=rolling_plan or PLAYER_ROLLING_PLAN,
    )


def candidate_feature_columns(df: pd.DataFrame) -> list[str]:
    # Return numeric engineered feature columns and avoid raw same-game stats.
    numeric_cols = df.select_dtypes(include="number").columns
    allowed_tokens = ("_roll_", "_rate_", "_momentum_", "_z_", "_std_", "days_since_", "is_")
    return sorted(column for column in numeric_cols if any(token in column for token in allowed_tokens))


In [5]:
from pathlib import Path

import numpy as np
import pandas as pd



TARGET_COMPONENTS = ["points", "reboundsTotal", "assists", "steals", "blocks", "turnovers"]
TARGET_WEIGHTS = {
    "points": 1.0,
    "reboundsTotal": 1.2,
    "assists": 1.5,
    "steals": 3.0,
    "blocks": 3.0,
    "turnovers": -1.0,
}

ID_COLUMNS = {
    "gameId",
    "game_id",
    "personId",
    "playerId",
    "teamId",
    "team_id",
    "opponentTeamId",
    "season",
    "gameDate",
    "game_date",
    "gameDateTimeEst",
}

SAME_GAME_STAT_HINTS = {
    "points",
    "reboundsTotal",
    "assists",
    "steals",
    "blocks",
    "turnovers",
    "numMinutes",
    "minutes",
    "usgPct",
    "tsPct",
    "pie",
    "pace",
    "offRating",
    "defRating",
    "netRating",
    "rebPct",
    "astPct",
    "efgPct",
    "teamScore",
    "opponentScore",
    "plusMinusPoints",
}

LAST4_SEASON_STARTS = [2021, 2022, 2023, 2024, 2025]

PLAYER_BASE_COLUMNS = [
    "firstName",
    "lastName",
    "personId",
    "gameId",
    "gameDateTimeEst",
    "gameType",
    "win",
    "home",
    "playerteamId",
    "playerteamCity",
    "playerteamName",
    "opponentteamId",
    "opponentteamCity",
    "opponentteamName",
    "comment",
    "startingPosition",
    "numMinutes",
    "points",
    "assists",
    "reboundsTotal",
    "reboundsOffensive",
    "reboundsDefensive",
    "fieldGoalsMade",
    "fieldGoalsAttempted",
    "fieldGoalsPercentage",
    "threePointersMade",
    "threePointersAttempted",
    "threePointersPercentage",
    "freeThrowsMade",
    "freeThrowsAttempted",
    "freeThrowsPercentage",
    "steals",
    "blocks",
    "blocksAgainst",
    "turnovers",
    "foulsPersonal",
    "plusMinusPoints",
    "estimatedOffensiveRating",
    "offensiveRating",
    "estimatedDefensiveRating",
    "defensiveRating",
    "estimatedNetRating",
    "netRating",
    "assistPercentage",
    "assistToTurnoverRatio",
    "assistRatio",
    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    "reboundPercentage",
    "teamTurnoverPercentage",
    "effectiveFieldGoalPercentage",
    "trueShootingPercentage",
    "usagePercentage",
    "estimatedPace",
    "pace",
    "playerImpactEstimate",
    "possessions",
    "pointsOffTurnovers",
    "pointsSecondChance",
    "pointsFastBreak",
    "pointsInPaint",
]

PLAYER_PROFILE_COLUMNS = [
    "personId",
    "heightInches",
    "bodyWeightLbs",
    "guard",
    "forward",
    "center",
    "draftYear",
    "draftRound",
    "draftNumber",
    "school",
    "country",
]

TEAM_CONTEXT_COLUMNS = [
    "gameId",
    "gameDateTimeEst",
    "gameType",
    "teamId",
    "teamCity",
    "teamName",
    "opponentTeamId",
    "opponentTeamCity",
    "opponentTeamName",
    "home",
    "win",
    "teamScore",
    "opponentScore",
    "numMinutes",
    "assists",
    "steals",
    "blocks",
    "blocksAgainst",
    "fieldGoalsPercentage",
    "threePointersPercentage",
    "freeThrowsPercentage",
    "reboundsOffensive",
    "reboundsDefensive",
    "reboundsTotal",
    "turnovers",
    "plusMinusPoints",
    "benchPoints",
    "pointsFastBreak",
    "pointsFromTurnovers",
    "pointsInThePaint",
    "pointsSecondChance",
    "seasonWins",
    "seasonLosses",
    "estimatedOffensiveRating",
    "offensiveRating",
    "estimatedDefensiveRating",
    "defensiveRating",
    "estimatedNetRating",
    "netRating",
    "assistPercentage",
    "assistToTurnoverRatio",
    "assistRatio",
    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    "reboundPercentage",
    "teamTurnoverPercentage",
    "effectiveFieldGoalPercentage",
    "trueShootingPercentage",
    "estimatedPace",
    "pace",
    "possessions",
    "playerImpactEstimate",
    "opponentPointsOffTurnovers",
    "opponentPointsSecondChance",
    "opponentPointsFastBreak",
    "opponentPointsInPaint",
    "opponentEffectiveFieldGoalPercentage",
    "opponentFreeThrowAttemptRate",
    "opponentTurnoverPercentage",
    "opponentOffensiveReboundPercentage",
]

PLAYER_MODEL_ROLLING_SOURCES = [
    "fantasy_points",
    "current_is_starter",
    "starter_minutes",
    "bench_minutes",
    "numMinutes",
    "points",
    "reboundsTotal",
    "assists",
    "steals",
    "blocks",
    "turnovers",
    "usagePercentage",
    "trueShootingPercentage",
    "effectiveFieldGoalPercentage",
    "fieldGoalsPercentage",
    "threePointersPercentage",
    "freeThrowsPercentage",
    "playerImpactEstimate",
    "offensiveRating",
    "defensiveRating",
    "estimatedDefensiveRating",
    "pace",
]

DIRECT_MODEL_COLUMNS = [
    "gameId",
    "personId",
    "game_date",
    "season_start",
    "firstName",
    "lastName",
    "player_name",
    "playerteamId",
    "playerteamCity",
    "playerteamName",
    "opponentteamId",
    "opponentteamCity",
    "opponentteamName",
    "home",
    "current_is_starter",
    "starter_minutes",
    "bench_minutes",
    "startingPosition",
    "heightInches",
    "bodyWeightLbs",
    "guard",
    "forward",
    "center",
    "primary_position_group",
    "years_since_draft",
    "fantasy_points",
]


def _available_columns(requested: list[str], actual: list[str]) -> list[str]:
    # Return requested columns that actually exist in the source schema.
    actual_set = set(actual)
    return [column for column in requested if column in actual_set]


def _normalize_id(series: pd.Series) -> pd.Series:
    # Normalize numeric-looking IDs to nullable strings for stable merges.
    numeric = pd.to_numeric(series, errors="coerce")
    normalized = numeric.astype("Int64").astype("string")
    fallback = series.astype("string").str.strip()
    return normalized.fillna(fallback)


def add_game_date_and_season(df: pd.DataFrame, date_col: str = "gameDateTimeEst") -> pd.DataFrame:
    # Add parsed game_date and NBA season_start.
    result = df.copy()
    result["game_date"] = pd.to_datetime(result[date_col], errors="coerce")
    result["season_start"] = np.where(result["game_date"].dt.month >= 7, result["game_date"].dt.year, result["game_date"].dt.year - 1)
    result["season_start"] = result["season_start"].astype("Int64")
    return result


def filter_regular_last4(df: pd.DataFrame, season_starts: list[int] | None = None) -> pd.DataFrame:
    # Keep regular-season rows from the selected NBA season starts.
    season_starts = season_starts or LAST4_SEASON_STARTS
    result = add_game_date_and_season(df)
    return result[result["gameType"].eq("Regular Season") & result["season_start"].isin(season_starts)].copy()


def repair_team_game_type_from_games(team: pd.DataFrame) -> pd.DataFrame:
    # Fill missing team-game gameType values from Games.csv before regular-season filtering.
    #
    # TeamStatisticsExtended has valid 2021 team rows, but many 2021 gameType
    # values are missing there. Games.csv has the same gameIds with the correct
    # gameType, so this repair prevents valid 2021 regular-season team context
    # from being dropped.
    if "gameId" not in team.columns or "gameType" not in team.columns:
        return team

    games_header = load_raw_table("Games", nrows=0)
    game_columns = _available_columns(["gameId", "gameType"], games_header.columns.tolist())
    if set(game_columns) != {"gameId", "gameType"}:
        return team

    result = team.copy()
    result["gameId"] = _normalize_id(result["gameId"])
    games = load_raw_table("Games", columns=game_columns).copy()
    games["gameId"] = _normalize_id(games["gameId"])
    games = games.drop_duplicates("gameId").rename(columns={"gameType": "gameType_from_games"})

    result = result.merge(games, on="gameId", how="left", validate="many_to_one")
    result["gameType"] = result["gameType"].fillna(result["gameType_from_games"])
    return result.drop(columns=["gameType_from_games"])


def add_player_name(df: pd.DataFrame) -> pd.DataFrame:
    # Add readable player_name.
    result = df.copy()
    result["player_name"] = (
        result["firstName"].fillna("").astype(str).str.strip()
        + " "
        + result["lastName"].fillna("").astype(str).str.strip()
    ).str.strip()
    return result


def clean_player_profile(df: pd.DataFrame) -> pd.DataFrame:
    # Clean static player profile features.
    result = df.copy()
    for column in ["guard", "forward", "center"]:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column], errors="coerce").fillna(0).astype(int)

    if {"draftYear", "game_date"}.issubset(result.columns):
        draft_year = pd.to_numeric(result["draftYear"], errors="coerce")
        result["years_since_draft"] = result["game_date"].dt.year - draft_year
        result.loc[result["years_since_draft"] < 0, "years_since_draft"] = np.nan

    return result


def add_primary_position_group(df: pd.DataFrame) -> pd.DataFrame:
    # Add a compact G/F/C position group from starter slot and profile flags.
    result = df.copy()
    result["primary_position_group"] = result.get("startingPosition", pd.Series(index=result.index, dtype="object"))
    result["primary_position_group"] = result["primary_position_group"].where(
        result["primary_position_group"].isin(["G", "F", "C"])
    )

    missing = result["primary_position_group"].isna()
    if "center" in result.columns:
        result.loc[missing & result["center"].eq(1), "primary_position_group"] = "C"
    missing = result["primary_position_group"].isna()
    if "guard" in result.columns:
        result.loc[missing & result["guard"].eq(1), "primary_position_group"] = "G"
    missing = result["primary_position_group"].isna()
    if "forward" in result.columns:
        result.loc[missing & result["forward"].eq(1), "primary_position_group"] = "F"

    result["primary_position_group"] = result["primary_position_group"].fillna("Unknown")
    return result


def add_starter_minute_sources(df: pd.DataFrame) -> pd.DataFrame:
    # Add same-game role-minute raw material used only through shifted rolling features.
    result = df.copy()
    minutes = pd.to_numeric(result["numMinutes"], errors="coerce")
    starter = pd.to_numeric(result["current_is_starter"], errors="coerce").fillna(0)
    result["starter_minutes"] = starter * minutes
    result["bench_minutes"] = (1 - starter) * minutes
    return result


def add_fantasy_points(df: pd.DataFrame) -> pd.DataFrame:
    # Calculate the fantasy-points target from player-game box-score columns.
    missing = [column for column in TARGET_COMPONENTS if column not in df.columns]
    if missing:
        raise ValueError(f"Cannot calculate fantasy_points. Missing columns: {missing}")

    result = df.copy()
    for column in TARGET_COMPONENTS:
        result[column] = pd.to_numeric(result[column], errors="coerce")

    result["fantasy_points"] = sum(result[column] * weight for column, weight in TARGET_WEIGHTS.items())
    return result


def load_player_base_last4_regular(season_starts: list[int] | None = None) -> pd.DataFrame:
    # Load the player-game base from PlayerStatisticsExtended.
    header = load_raw_table("PlayerStatisticsExtended", nrows=0)
    columns = _available_columns(PLAYER_BASE_COLUMNS, header.columns.tolist())
    player = load_raw_table("PlayerStatisticsExtended", columns=columns)
    player = filter_regular_last4(player, season_starts=season_starts)

    for column in ["gameId", "personId", "playerteamId", "opponentteamId"]:
        if column in player.columns:
            player[column] = _normalize_id(player[column])

    player = add_player_name(add_fantasy_points(player))
    player["current_is_starter"] = player["startingPosition"].notna().astype(int)
    player = add_starter_minute_sources(player)
    return player.sort_values(["game_date", "gameId", "personId"]).reset_index(drop=True)



def load_team_games_last4_regular(season_starts: list[int] | None = None) -> pd.DataFrame:
    # Load team-game rows from TeamStatisticsExtended.
    header = load_raw_table("TeamStatisticsExtended", nrows=0)
    columns = _available_columns(TEAM_CONTEXT_COLUMNS, header.columns.tolist())
    team = load_raw_table("TeamStatisticsExtended", columns=columns)

    for column in ["gameId", "teamId", "opponentTeamId"]:
        if column in team.columns:
            team[column] = _normalize_id(team[column])
    team = repair_team_game_type_from_games(team)
    team = filter_regular_last4(team, season_starts=season_starts)
    team["win"] = pd.to_numeric(team["win"], errors="coerce")
    return team.sort_values(["game_date", "gameId", "teamId"]).reset_index(drop=True)


def add_player_profile(base: pd.DataFrame) -> pd.DataFrame:
    # Merge static player profile data.
    header = load_raw_table("Players", nrows=0)
    columns = _available_columns(PLAYER_PROFILE_COLUMNS, header.columns.tolist())
    players = load_raw_table("Players", columns=columns)
    players["personId"] = _normalize_id(players["personId"])

    result = base.merge(players, on="personId", how="left")
    return clean_player_profile(result)


def _prefixed_team_source(team_games: pd.DataFrame, prefix: str) -> pd.DataFrame:
    # Prefix team-context columns so own-team and opponent-team joins stay distinct.
    rename_map = {}
    for column in team_games.columns:
        if column in {"gameId", "teamId"}:
            continue
        rename_map[column] = f"{prefix}{column}"
    return team_games.rename(columns=rename_map)


def build_player_game_source_last4_regular(season_starts: list[int] | None = None) -> pd.DataFrame:
    # Build one row per player-game with selected player, team, and opponent raw material.
    base = add_player_profile(load_player_base_last4_regular(season_starts=season_starts))
    team_games = load_team_games_last4_regular(season_starts=season_starts)

    own_team = _prefixed_team_source(team_games, "team_").rename(columns={"teamId": "playerteamId"})
    source = base.merge(own_team, on=["gameId", "playerteamId"], how="left", validate="many_to_one")

    opponent_team = _prefixed_team_source(team_games, "oppteam_").rename(columns={"teamId": "opponentteamId"})
    source = source.merge(opponent_team, on=["gameId", "opponentteamId"], how="left", validate="many_to_one")

    source = add_primary_position_group(source)
    return source.sort_values(["game_date", "gameId", "personId"]).reset_index(drop=True)


def build_team_model_source(team_games: pd.DataFrame) -> pd.DataFrame:
    # Create team-game modeling source columns for own-team rolling features.
    result = team_games[
        [
            "gameId",
            "teamId",
            "game_date",
            "win",
            "teamScore",
            "opponentScore",
            "pace",
            "offensiveRating",
            "defensiveRating",
            "estimatedDefensiveRating",
            "fieldGoalsPercentage",
            "threePointersPercentage",
            "effectiveFieldGoalPercentage",
            "trueShootingPercentage",
            "assists",
            "reboundsTotal",
            "steals",
            "blocks",
            "turnovers",
            "pointsInThePaint",
            "pointsFastBreak",
            "pointsSecondChance",
            "pointsFromTurnovers",
        ]
    ].copy()
    return result.rename(
        columns={
            "win": "team_win",
            "teamScore": "team_score",
            "opponentScore": "team_points_allowed",
            "pace": "team_pace",
            "offensiveRating": "team_offensiveRating",
            "defensiveRating": "team_defensiveRating",
            "estimatedDefensiveRating": "team_estimatedDefensiveRating",
            "fieldGoalsPercentage": "team_fieldGoalsPercentage",
            "threePointersPercentage": "team_threePointersPercentage",
            "effectiveFieldGoalPercentage": "team_effectiveFieldGoalPercentage",
            "trueShootingPercentage": "team_trueShootingPercentage",
            "assists": "team_assists",
            "reboundsTotal": "team_reboundsTotal",
            "steals": "team_steals",
            "blocks": "team_blocks",
            "turnovers": "team_turnovers",
            "pointsInThePaint": "team_pointsInThePaint",
            "pointsFastBreak": "team_pointsFastBreak",
            "pointsSecondChance": "team_pointsSecondChance",
            "pointsFromTurnovers": "team_pointsFromTurnovers",
        }
    )


def build_opponent_model_source(team_games: pd.DataFrame) -> pd.DataFrame:
    # Create team-game modeling source columns for opponent defensive rolling features.
    result = team_games.copy()

    source_map = {
        "win": "opp_win",
        "opponentScore": "opp_points_allowed",
        "pace": "opp_pace",
        "defensiveRating": "opp_defensiveRating",
        "estimatedDefensiveRating": "opp_estimatedDefensiveRating",
        "opponentEffectiveFieldGoalPercentage": "opp_effectiveFieldGoalPercentage_allowed",
        "trueShootingPercentage": "opp_trueShootingPercentage_allowed",
        "fieldGoalsPercentage": "opp_fieldGoalsPercentage_allowed",
        "reboundsTotal": "opp_rebounds_allowed",
        "assists": "opp_assists_allowed",
        "steals": "opp_steals",
        "blocks": "opp_blocks",
        "turnovers": "opp_turnovers_forced",
        "opponentPointsInPaint": "opp_pointsInThePaint_allowed",
        "opponentPointsFastBreak": "opp_pointsFastBreak_allowed",
        "opponentPointsSecondChance": "opp_pointsSecondChance_allowed",
        "opponentPointsOffTurnovers": "opp_pointsFromTurnovers_allowed",
    }
    columns = ["gameId", "teamId", "game_date"] + [column for column in source_map if column in result.columns]
    return result[columns].rename(columns=source_map)


def build_opponent_position_model_source(source: pd.DataFrame) -> pd.DataFrame:
    # Create opponent-position game rows used for shifted matchup rolling features.
    required = {"gameId", "opponentteamId", "primary_position_group", "game_date", "fantasy_points", "numMinutes"}
    missing = required.difference(source.columns)
    if missing:
        raise ValueError(f"Cannot build opponent-position source. Missing columns: {sorted(missing)}")

    result = source.copy()
    result["fantasy_points"] = pd.to_numeric(result["fantasy_points"], errors="coerce")
    result["numMinutes"] = pd.to_numeric(result["numMinutes"], errors="coerce")
    if "current_is_starter" in result.columns:
        result["current_is_starter"] = pd.to_numeric(result["current_is_starter"], errors="coerce").fillna(0)
    else:
        result["current_is_starter"] = 0
    result = result[result["primary_position_group"].ne("Unknown")].copy()

    grouped = (
        result.groupby(["gameId", "opponentteamId", "primary_position_group", "game_date"], as_index=False)
        .agg(
            opp_pos_fp_allowed=("fantasy_points", "sum"),
            opp_pos_fp_allowed_per_player=("fantasy_points", "mean"),
            opp_pos_minutes_allowed=("numMinutes", "sum"),
            opp_pos_player_count=("personId", "nunique"),
            opp_pos_starter_count=("current_is_starter", "sum"),
        )
        .rename(columns={"opponentteamId": "teamId"})
    )
    grouped["defense_position_key"] = grouped["teamId"].astype(str) + "__" + grouped["primary_position_group"].astype(str)
    return grouped.sort_values(["game_date", "gameId", "teamId", "primary_position_group"]).reset_index(drop=True)


def create_column_decisions_v1(output_path: Path | None = None) -> pd.DataFrame:
    # Create the first explicit keep/drop/rolling decision table.
    output_path = output_path or TABLES_DIR / "column_decisions_v1.csv"
    rows = []

    for column in PLAYER_BASE_COLUMNS:
        role = "drop_not_relevant_v1"
        direct = False
        rolling = False
        target = column in TARGET_COMPONENTS
        reason = "not selected for v1 modeling"
        if column in {"gameId", "personId", "playerteamId", "opponentteamId", "gameDateTimeEst"}:
            role, direct, reason = "id" if column != "gameDateTimeEst" else "date", True, "key/date needed for grain and joins"
        elif column in {"firstName", "lastName", "playerteamCity", "playerteamName", "opponentteamCity", "opponentteamName", "home", "startingPosition"}:
            role, direct, reason = "pre_game_context", True, "known before game or useful for grouping/analysis"
        elif target:
            role, reason = "target", "target component; use only to build fantasy_points and historical rolling"
        elif column in PLAYER_MODEL_ROLLING_SOURCES:
            role, rolling, reason = "rolling_source_player", True, "same-game player stat; use only after shift(1) rolling"
        elif column in {"comment", "plusMinusPoints"}:
            role, reason = "drop_same_game_leakage", "same-game outcome/comment field"

        rows.append(
            {
                "source_table": "PlayerStatisticsExtended",
                "column": column,
                "role": role,
                "direct_feature_allowed": direct,
                "rolling_source": rolling,
                "target_component": target,
                "reason": reason,
            }
        )

    for column, reason in {
        "fantasy_points": "generated target; also used only as shifted historical rolling source",
        "current_is_starter": "generated lineup-aware direct feature and shifted historical starter-rate source",
    }.items():
        rows.append(
            {
                "source_table": "generated",
                "column": column,
                "role": "target" if column == "fantasy_points" else "pre_game_context",
                "direct_feature_allowed": column == "current_is_starter",
                "rolling_source": True,
                "target_component": column == "fantasy_points",
                "reason": reason,
            }
        )

    for column in PLAYER_PROFILE_COLUMNS:
        rows.append(
            {
                "source_table": "Players",
                "column": column,
                "role": "static_profile" if column != "personId" else "id",
                "direct_feature_allowed": True,
                "rolling_source": False,
                "target_component": False,
                "reason": "static player profile feature",
            }
        )

    for column in TEAM_CONTEXT_COLUMNS:
        role = "rolling_source_team"
        reason = "same-game team stat; use only after shift(1) team/opponent rolling"
        direct = False
        rolling = True
        if column in {"gameId", "teamId", "opponentTeamId"}:
            role, direct, rolling, reason = "id", True, False, "team-game merge key"
        elif column in {"gameDateTimeEst", "gameType"}:
            role, direct, rolling, reason = "date", True, False, "filter/date column"
        elif column in {"teamCity", "teamName", "opponentTeamCity", "opponentTeamName", "home"}:
            role, direct, rolling, reason = "pre_game_context", True, False, "team labels/context"

        rows.append(
            {
                "source_table": "TeamStatisticsExtended",
                "column": column,
                "role": role,
                "direct_feature_allowed": direct,
                "rolling_source": rolling,
                "target_component": False,
                "reason": reason,
            }
        )

    decisions = pd.DataFrame(rows).drop_duplicates(["source_table", "column"]).reset_index(drop=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    decisions.to_csv(output_path, index=False)
    return decisions


def write_source_csv(df: pd.DataFrame, output_path: Path | None = None) -> Path:
    # Write the merged source table as CSV.
    output_path = output_path or PROCESSED_DIR / "player_game_source_last4_regular.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    return output_path


def add_rest_features(df: pd.DataFrame) -> pd.DataFrame:
    # Add player rest features known before the game.
    result = df.sort_values(["personId", "game_date", "gameId"]).copy()
    result["days_since_last_game"] = result.groupby("personId")["game_date"].diff().dt.days
    result["is_back_to_back"] = result["days_since_last_game"].eq(1).astype(int)
    return result.sort_index()


def build_modeling_table_last4_regular(source: pd.DataFrame | None = None, season_starts: list[int] | None = None) -> pd.DataFrame:
    # Build the leakage-safe modeling table with shifted rolling features.

    source = source.copy() if source is not None else build_player_game_source_last4_regular(season_starts=season_starts)
    source["game_date"] = pd.to_datetime(source["game_date"], errors="coerce")
    if "primary_position_group" not in source.columns:
        source = add_primary_position_group(source)
    if {"numMinutes", "current_is_starter"}.issubset(source.columns) and "starter_minutes" not in source.columns:
        source = add_starter_minute_sources(source)
    source = add_rest_features(source)

    direct_columns = [column for column in DIRECT_MODEL_COLUMNS if column in source.columns]
    for extra in ["days_since_last_game", "is_back_to_back"]:
        if extra in source.columns and extra not in direct_columns:
            direct_columns.append(extra)

    modeling = source[direct_columns].copy()

    player_sources = ["gameId", "personId", "game_date"] + [column for column in PLAYER_ROLLING_PLAN if column in source.columns]
    player_rolls = add_lagged_rolling_plan(
        source[player_sources],
        group_col="personId",
        date_col="game_date",
        rolling_plan={column: windows for column, windows in PLAYER_ROLLING_PLAN.items() if column in source.columns},
    )
    player_feature_cols = [
        column
        for column in player_rolls.columns
        if "_roll_" in column or "_momentum_" in column or "_z_" in column or "_std_" in column
    ]
    modeling = modeling.merge(player_rolls[["gameId", "personId"] + player_feature_cols], on=["gameId", "personId"], how="left", validate="one_to_one")
    for window in [3, 5, 10, 30]:
        starter_col = f"starter_minutes_roll_{window}"
        minutes_col = f"numMinutes_roll_{window}"
        share_col = f"starter_minutes_share_roll_{window}"
        if starter_col in modeling.columns and minutes_col in modeling.columns:
            modeling[share_col] = modeling[starter_col] / modeling[minutes_col].replace(0, np.nan)

    team_games = load_team_games_last4_regular(season_starts=season_starts)
    team_source = build_team_model_source(team_games)
    team_rolls = add_lagged_rolling_plan(
        team_source,
        group_col="teamId",
        date_col="game_date",
        rolling_plan={column: windows for column, windows in TEAM_ROLLING_PLAN.items() if column in team_source.columns},
    )
    team_feature_cols = [
        column
        for column in team_rolls.columns
        if column.startswith("team_") and ("_roll_" in column or "_momentum_" in column or "_z_" in column or "_std_" in column)
    ]
    team_rolls = team_rolls.rename(columns={"teamId": "playerteamId"})
    modeling = modeling.merge(
        team_rolls[["gameId", "playerteamId"] + team_feature_cols],
        on=["gameId", "playerteamId"],
        how="left",
        validate="many_to_one",
    )

    opp_source = build_opponent_model_source(team_games)
    fp_allowed = (
        source.groupby(["gameId", "opponentteamId"], as_index=False)["fantasy_points"]
        .sum()
        .rename(columns={"opponentteamId": "teamId", "fantasy_points": "opp_fp_allowed"})
    )
    opp_source = opp_source.merge(fp_allowed, on=["gameId", "teamId"], how="left", validate="one_to_one")
    opp_rolls = add_lagged_rolling_plan(
        opp_source,
        group_col="teamId",
        date_col="game_date",
        rolling_plan={column: windows for column, windows in OPPONENT_ROLLING_PLAN.items() if column in opp_source.columns},
    )
    opp_feature_cols = [
        column
        for column in opp_rolls.columns
        if column.startswith("opp_") and ("_roll_" in column or "_momentum_" in column or "_z_" in column or "_std_" in column)
    ]
    opp_rolls = opp_rolls.rename(columns={"teamId": "opponentteamId"})
    modeling = modeling.merge(
        opp_rolls[["gameId", "opponentteamId"] + opp_feature_cols],
        on=["gameId", "opponentteamId"],
        how="left",
        validate="many_to_one",
    )

    opp_pos_source = build_opponent_position_model_source(source)
    opp_pos_rolls = add_lagged_rolling_plan(
        opp_pos_source,
        group_col="defense_position_key",
        date_col="game_date",
        rolling_plan={column: windows for column, windows in OPPONENT_POSITION_ROLLING_PLAN.items() if column in opp_pos_source.columns},
    )
    opp_pos_feature_cols = [
        column
        for column in opp_pos_rolls.columns
        if column.startswith("opp_pos_") and ("_roll_" in column or "_momentum_" in column or "_z_" in column or "_std_" in column)
    ]
    opp_pos_rolls = opp_pos_rolls.rename(columns={"teamId": "opponentteamId"})
    modeling = modeling.merge(
        opp_pos_rolls[["gameId", "opponentteamId", "primary_position_group"] + opp_pos_feature_cols],
        on=["gameId", "opponentteamId", "primary_position_group"],
        how="left",
        validate="many_to_one",
    )
    for window in [3, 5, 10, 30]:
        fp_col = f"opp_pos_fp_allowed_per_player_roll_{window}"
        league_col = f"league_pos_fp_allowed_per_player_roll_{window}"
        diff_col = f"pos_opp_difficulty_roll_{window}"
        ratio_col = f"pos_opp_difficulty_ratio_{window}"
        if fp_col in modeling.columns:
            league_rolls = add_lagged_rolling_plan(
                opp_pos_source,
                group_col="primary_position_group",
                date_col="game_date",
                rolling_plan={"opp_pos_fp_allowed_per_player": [window]},
            )
            league_rolls = league_rolls.rename(
                columns={
                    "teamId": "opponentteamId",
                    f"opp_pos_fp_allowed_per_player_roll_{window}": league_col,
                }
            )
            modeling = modeling.merge(
                league_rolls[["gameId", "opponentteamId", "primary_position_group", league_col]],
                on=["gameId", "opponentteamId", "primary_position_group"],
                how="left",
                validate="many_to_one",
            )
            modeling[diff_col] = modeling[fp_col] - modeling[league_col]
            modeling[ratio_col] = modeling[fp_col] / modeling[league_col].replace(0, np.nan) - 1

    return modeling.sort_values(["game_date", "gameId", "personId"]).reset_index(drop=True)


def write_modeling_csv(df: pd.DataFrame, output_path: Path | None = None) -> Path:
    # Write the modeling table as CSV.
    output_path = output_path or PROCESSED_DIR / "modeling_table_last4_regular.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    return output_path


def has_target_components(columns: list[str] | pd.Index) -> bool:
    # Return True when a table has all columns needed to create fantasy_points.
    return set(TARGET_COMPONENTS).issubset(set(map(str, columns)))


def candidate_base_tables(column_summary: pd.DataFrame) -> pd.DataFrame:
    # Find raw tables that could support one row per player-game target creation.
    if column_summary.empty:
        return pd.DataFrame(
            columns=[
                "table_name",
                "has_game_id",
                "has_player_id",
                "has_target_components",
                "target_components_present",
                "target_components_missing",
            ]
        )

    rows = []
    for table_name, group in column_summary.groupby("table_name", sort=True):
        columns = set(group["column"].astype(str))
        target_present = [column for column in TARGET_COMPONENTS if column in columns]
        target_missing = [column for column in TARGET_COMPONENTS if column not in columns]
        rows.append(
            {
                "table_name": table_name,
                "has_game_id": bool({"gameId", "game_id", "GAME_ID"}.intersection(columns)),
                "has_player_id": bool({"personId", "playerId", "PLAYER_ID", "PersonID"}.intersection(columns)),
                "has_target_components": not target_missing,
                "target_components_present": ", ".join(target_present),
                "target_components_missing": ", ".join(target_missing),
            }
        )
    return pd.DataFrame(rows)


def classify_raw_columns(column_summary: pd.DataFrame) -> pd.DataFrame:
    # Create an initial leakage-focused raw-column classification table.
    output_columns = [
        "column",
        "source_table",
        "meaning",
        "keep_for_id",
        "target_component",
        "same_game_leakage",
        "rolling_candidate",
        "drop_reason",
    ]
    if column_summary.empty:
        return pd.DataFrame(columns=output_columns)

    rows = []
    for record in column_summary.to_dict("records"):
        column = str(record["column"])
        keep_for_id = column in ID_COLUMNS or bool(record.get("is_suspected_key", False))
        target_component = column in TARGET_COMPONENTS
        same_game_leakage = target_component or column in SAME_GAME_STAT_HINTS
        rolling_candidate = same_game_leakage and not target_component
        drop_reason = ""
        if target_component:
            drop_reason = "target component; use only to build fantasy_points"
        elif same_game_leakage:
            drop_reason = "same-game statistic; convert to shifted historical rolling feature before modeling"

        rows.append(
            {
                "column": column,
                "source_table": record["table_name"],
                "meaning": "",
                "keep_for_id": keep_for_id,
                "target_component": target_component,
                "same_game_leakage": same_game_leakage,
                "rolling_candidate": rolling_candidate,
                "drop_reason": drop_reason,
            }
        )

    return pd.DataFrame(rows, columns=output_columns)


In [6]:
column_decisions = create_column_decisions_v1(output_path=TABLES_DIR / "column_decisions_v1.csv")
source = build_player_game_source_last4_regular(season_starts=LAST4_SEASON_STARTS)
source_path = write_source_csv(source, output_path=PROCESSED_DIR / "player_game_source_last4_regular.csv")
modeling = build_modeling_table_last4_regular(source=source, season_starts=LAST4_SEASON_STARTS)
modeling_path = write_modeling_csv(modeling, output_path=PROCESSED_DIR / "modeling_table_last4_regular.csv")

team_2021_check = load_team_games_last4_regular(season_starts=[2021])
team_2021_check_summary = pd.DataFrame(
    {
        "check": ["2021 regular-season team rows after Games.csv gameType repair"],
        "rows": [len(team_2021_check)],
        "regular_season_rows": [int(team_2021_check["gameType"].eq("Regular Season").sum())],
    }
)
team_2021_check_summary.to_csv(TABLES_DIR / "team_2021_game_type_repair_check.csv", index=False)

print(f"column decisions: {column_decisions.shape}")
print(f"source: {source.shape} -> {source_path}")
print(f"modeling: {modeling.shape} -> {modeling_path}")
display(team_2021_check_summary)
display(source[["gameId", "personId", "player_name", "game_date", "season_start", "fantasy_points", "current_is_starter", "starter_minutes", "primary_position_group"]].head())
needed = ["fantasy_points_roll_3", "usagePercentage_roll_3", "usagePercentage_momentum_3_vs_10", "starter_minutes", "starter_minutes_roll_3", "pos_opp_difficulty_roll_30"]
print({feature: feature in modeling.columns for feature in needed})


column decisions: (134, 7)
source: (129750, 200) -> C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\runs\final\data\processed\player_game_source_last4_regular.csv
modeling: (129750, 576) -> C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\runs\final\data\processed\modeling_table_last4_regular.csv


,check,rows,regular_season_rows
0,2021 regular-season team rows after Games.csv ...,2460,2460


,gameId,personId,player_name,game_date,season_start,fantasy_points,current_is_starter,starter_minutes,primary_position_group
0,22100001,1626192,Pat Connaughton,2021-10-19 19:30:00,2021,32.6,0,0.0,G
1,22100001,1627761,DeAndre' Bembry,2021-10-19 19:30:00,2021,0.0,0,0.0,G
2,22100001,1628960,Grayson Allen,2021-10-19 19:30:00,2021,29.8,0,0.0,G
3,22100001,1628971,Bruce Brown,2021-10-19 19:30:00,2021,4.2,0,0.0,G
4,22100001,1628975,Jevon Carter,2021-10-19 19:30:00,2021,1.4,0,0.0,G


{'fantasy_points_roll_3': True, 'usagePercentage_roll_3': True, 'usagePercentage_momentum_3_vs_10': True, 'starter_minutes': True, 'starter_minutes_roll_3': True, 'pos_opp_difficulty_roll_30': True}


## 5. Data Cleansing Checks

The notebook checks data grain, dates, missingness, target distribution, and leakage categories through the source/modeling reports and EDA outputs.


## 6. Exploratory Data Analysis

EDA explains target distribution, player role differences, position differences, season drift, missingness, and simple target correlations.


In [7]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns



TARGET_COL = "fantasy_points"
SOURCE_EDA_COLUMNS = [
    "gameId",
    "personId",
    "numMinutes",
    "points",
    "reboundsTotal",
    "assists",
    "steals",
    "blocks",
    "turnovers",
    "usagePercentage",
    "trueShootingPercentage",
    "playerImpactEstimate",
]
DIRECT_LEAKAGE_COLUMNS = {
    "points",
    "reboundsTotal",
    "assists",
    "steals",
    "blocks",
    "turnovers",
    "numMinutes",
    "usagePercentage",
    "trueShootingPercentage",
    "pace",
    "offensiveRating",
    "defensiveRating",
}


def load_eda_data(
    modeling_path: Path | None = None,
    source_path: Path | None = None,
) -> pd.DataFrame:
    # Load modeling rows plus current-game columns used only for EDA segmentation.
    modeling_path = modeling_path or PROCESSED_DIR / "modeling_table_last4_regular.csv"
    source_path = source_path or PROCESSED_DIR / "player_game_source_last4_regular.csv"

    modeling = pd.read_csv(modeling_path, low_memory=False)
    source_header = pd.read_csv(source_path, nrows=0)
    source_cols = [column for column in SOURCE_EDA_COLUMNS if column in source_header.columns]
    source = pd.read_csv(source_path, usecols=source_cols, low_memory=False)

    rename_map = {column: f"actual_{column}" for column in source.columns if column not in {"gameId", "personId"}}
    source = source.rename(columns=rename_map)
    data = modeling.merge(source, on=["gameId", "personId"], how="left", validate="one_to_one")
    return add_eda_segments(data)


def add_eda_segments(df: pd.DataFrame) -> pd.DataFrame:
    # Add position, minutes, target, and role buckets for EDA.
    result = df.copy()
    result["game_date"] = pd.to_datetime(result["game_date"], errors="coerce")

    result["primary_position"] = "Unknown"
    if "guard" in result.columns:
        result.loc[result["guard"].eq(1), "primary_position"] = "Guard"
    if "forward" in result.columns:
        result.loc[result["forward"].eq(1), "primary_position"] = "Forward"
    if "center" in result.columns:
        result.loc[result["center"].eq(1), "primary_position"] = "Center"

    if "actual_numMinutes" in result.columns:
        result["minutes_bucket"] = pd.cut(
            pd.to_numeric(result["actual_numMinutes"], errors="coerce"),
            bins=[-0.1, 5, 15, 25, 35, np.inf],
            labels=["0-5", "5-15", "15-25", "25-35", "35+"],
        )

    result["starter_label"] = np.where(result["current_is_starter"].eq(1), "Starter", "Bench")
    result["fp_bucket"] = pd.cut(
        result[TARGET_COL],
        bins=[-np.inf, 10, 20, 30, 40, 50, np.inf],
        labels=["<=10", "10-20", "20-30", "30-40", "40-50", "50+"],
    )
    return result


def dataset_summary(df: pd.DataFrame) -> pd.DataFrame:
    # Return high-level dataset health metrics.
    metrics = {
        "rows": len(df),
        "columns": len(df.columns),
        "unique_players": df["personId"].nunique(),
        "unique_games": df["gameId"].nunique(),
        "unique_player_games": df[["gameId", "personId"]].drop_duplicates().shape[0],
        "duplicate_player_games": df.duplicated(["gameId", "personId"]).sum(),
        "date_min": df["game_date"].min(),
        "date_max": df["game_date"].max(),
        "seasons": ", ".join(map(str, sorted(df["season_start"].dropna().unique()))),
        "target_missing": df[TARGET_COL].isna().sum(),
        "target_mean": df[TARGET_COL].mean(),
        "target_median": df[TARGET_COL].median(),
        "target_std": df[TARGET_COL].std(),
        "target_min": df[TARGET_COL].min(),
        "target_max": df[TARGET_COL].max(),
    }
    return pd.DataFrame({"metric": metrics.keys(), "value": metrics.values()})


def missingness_summary(df: pd.DataFrame) -> pd.DataFrame:
    # Return missingness and constant-column diagnostics.
    rows = []
    for column in df.columns:
        series = df[column]
        rows.append(
            {
                "column": column,
                "dtype": str(series.dtype),
                "missing_count": int(series.isna().sum()),
                "missing_pct": float(series.isna().mean()),
                "non_null_count": int(series.notna().sum()),
                "unique_count_profiled": int(series.nunique(dropna=True)) if not series.apply(lambda x: isinstance(x, (list, tuple, np.ndarray))).any() else pd.NA,
                "all_null": bool(series.isna().all()),
                "near_constant": bool(series.nunique(dropna=True) <= 1),
            }
        )
    return pd.DataFrame(rows).sort_values(["missing_pct", "column"], ascending=[False, True]).reset_index(drop=True)


def numeric_feature_columns(df: pd.DataFrame) -> list[str]:
    # Numeric modeling features excluding IDs and target.
    exclude = {
        TARGET_COL,
        "gameId",
        "personId",
        "season_start",
        "playerteamId",
        "opponentteamId",
    }
    return [column for column in df.select_dtypes(include="number").columns if column not in exclude and not column.startswith("actual_")]


def current_game_diagnostic_correlations(df: pd.DataFrame) -> pd.DataFrame:
    # Correlations for current-game EDA-only columns that are not model features.
    rows = []
    for column in [column for column in df.select_dtypes(include="number").columns if column.startswith("actual_")]:
        corr = df[[column, TARGET_COL]].corr().iloc[0, 1]
        rows.append(
            {
                "feature": column,
                "correlation": corr,
                "abs_correlation": abs(corr) if pd.notna(corr) else np.nan,
                "missing_pct": df[column].isna().mean(),
                "feature_policy": "EDA only; same-game current value is not a pre-game model feature",
            }
        )
    return pd.DataFrame(rows).sort_values("abs_correlation", ascending=False).reset_index(drop=True)


def feature_correlations(df: pd.DataFrame) -> pd.DataFrame:
    # Correlation of numeric features with fantasy points.
    rows = []
    for column in numeric_feature_columns(df):
        corr = df[[column, TARGET_COL]].corr().iloc[0, 1]
        rows.append(
            {
                "feature": column,
                "correlation": corr,
                "abs_correlation": abs(corr) if pd.notna(corr) else np.nan,
                "missing_pct": df[column].isna().mean(),
            }
        )
    return pd.DataFrame(rows).sort_values("abs_correlation", ascending=False).reset_index(drop=True)


def highly_correlated_pairs(df: pd.DataFrame, feature_rank: pd.DataFrame, top_n_features: int = 150, threshold: float = 0.95) -> pd.DataFrame:
    # Find highly redundant numeric feature pairs among the strongest features.
    candidate_cols = feature_rank["feature"].head(top_n_features).tolist()
    candidate_cols = [column for column in candidate_cols if column in df.columns]
    if len(candidate_cols) < 2:
        return pd.DataFrame(columns=["feature_a", "feature_b", "correlation", "abs_correlation"])

    corr = df[candidate_cols].corr()
    rows = []
    for i, feature_a in enumerate(candidate_cols):
        for feature_b in candidate_cols[i + 1 :]:
            value = corr.loc[feature_a, feature_b]
            if pd.notna(value) and abs(value) >= threshold:
                rows.append(
                    {
                        "feature_a": feature_a,
                        "feature_b": feature_b,
                        "correlation": value,
                        "abs_correlation": abs(value),
                    }
                )
    return pd.DataFrame(rows).sort_values("abs_correlation", ascending=False).reset_index(drop=True)


def target_summary_by(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    # Summarize fantasy points by one segment.
    return (
        df.groupby(group_col, observed=True)[TARGET_COL]
        .agg(rows="size", mean="mean", median="median", std="std", min="min", p25=lambda x: x.quantile(0.25), p75=lambda x: x.quantile(0.75), max="max")
        .reset_index()
        .sort_values("mean", ascending=False)
    )


def player_volume_summary(df: pd.DataFrame) -> pd.DataFrame:
    # Player-level volume and target summary.
    return (
        df.groupby(["personId", "player_name"], observed=True)
        .agg(
            games=("gameId", "nunique"),
            starts=("current_is_starter", "sum"),
            starter_rate=("current_is_starter", "mean"),
            mean_fp=(TARGET_COL, "mean"),
            median_fp=(TARGET_COL, "median"),
            max_fp=(TARGET_COL, "max"),
            mean_minutes=("actual_numMinutes", "mean"),
        )
        .reset_index()
        .sort_values(["games", "mean_fp"], ascending=False)
    )


def outlier_tables(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    # Create outlier and special-case EDA tables.
    common_cols = [
        "game_date",
        "season_start",
        "gameId",
        "personId",
        "player_name",
        "playerteamName",
        "opponentteamName",
        "starter_label",
        "primary_position",
        "actual_numMinutes",
        TARGET_COL,
        "fantasy_points_roll_10",
        "fantasy_points_z_5_vs_30",
    ]
    common_cols = [column for column in common_cols if column in df.columns]
    z = (df[TARGET_COL] - df[TARGET_COL].mean()) / df[TARGET_COL].std()
    with_z = df.assign(target_z=z)

    tables = {
        "eda_outlier_games": with_z.reindex(with_z["target_z"].abs().sort_values(ascending=False).index).head(200)[common_cols + ["target_z"]],
        "eda_starter_underperformance_games": df[df["current_is_starter"].eq(1)].sort_values(TARGET_COL).head(200)[common_cols],
        "eda_low_minutes_high_fp_games": df[df["actual_numMinutes"].le(15)].sort_values(TARGET_COL, ascending=False).head(200)[common_cols],
        "eda_high_minutes_low_fp_games": df[df["actual_numMinutes"].ge(30)].sort_values(TARGET_COL).head(200)[common_cols],
    }
    return tables


def leakage_checks(df: pd.DataFrame) -> pd.DataFrame:
    # Return explicit leakage sanity checks.
    first_by_player = df.sort_values(["personId", "game_date", "gameId"]).groupby("personId", observed=True).head(1)
    rows = [
        {
            "check": "duplicate_game_person_rows",
            "value": int(df.duplicated(["gameId", "personId"]).sum()),
            "passed": bool(df.duplicated(["gameId", "personId"]).sum() == 0),
            "notes": "modeling grain should be one row per player-game",
        },
        {
            "check": "direct_same_game_leakage_columns_present",
            "value": ", ".join(sorted(DIRECT_LEAKAGE_COLUMNS.intersection(df.columns))),
            "passed": bool(not DIRECT_LEAKAGE_COLUMNS.intersection(df.columns)),
            "notes": "raw same-game stats should not be direct model columns",
        },
        {
            "check": "first_player_game_fp_roll_5_null_rate",
            "value": float(first_by_player["fantasy_points_roll_5"].isna().mean()) if "fantasy_points_roll_5" in df.columns else np.nan,
            "passed": bool("fantasy_points_roll_5" in df.columns and first_by_player["fantasy_points_roll_5"].isna().mean() == 1.0),
            "notes": "first row per player should not have previous-game rolling fantasy points",
        },
        {
            "check": "current_starter_feature_policy",
            "value": "current_is_starter" in df.columns,
            "passed": bool("current_is_starter" in df.columns),
            "notes": "current_is_starter is lineup-aware; early projection models should exclude it",
        },
    ]
    return pd.DataFrame(rows)


def _savefig(path: Path) -> None:
    # Save the current Matplotlib figure to disk and close it.
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def plot_target_distribution(df: pd.DataFrame, figures_dir: Path = FIGURES_DIR) -> None:
    # Save histograms that describe the fantasy-points target distribution.
    plt.figure(figsize=(10, 6))
    sns.histplot(df[TARGET_COL], bins=60, kde=True)
    plt.title("Fantasy Points Distribution")
    plt.xlabel("Fantasy points")
    _savefig(figures_dir / "eda_target_distribution.png")


def plot_group_boxplots(df: pd.DataFrame, figures_dir: Path = FIGURES_DIR) -> None:
    # Save boxplots comparing fantasy points across key player/game groups.
    for column, file_name, title in [
        ("season_start", "eda_target_by_season.png", "Fantasy Points By Season"),
        ("starter_label", "eda_target_by_starter.png", "Fantasy Points By Starter Status"),
        ("primary_position", "eda_target_by_position.png", "Fantasy Points By Primary Position"),
        ("minutes_bucket", "eda_target_by_minutes_bucket.png", "Fantasy Points By Minutes Bucket"),
    ]:
        if column not in df.columns:
            continue
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=column, y=TARGET_COL)
        plt.title(title)
        plt.xlabel(column)
        plt.ylabel("Fantasy points")
        _savefig(figures_dir / file_name)


def plot_feature_relationships(df: pd.DataFrame, figures_dir: Path = FIGURES_DIR, sample_size: int = 50_000) -> None:
    # Save scatter plots for selected feature-target relationships.
    features = [
        "actual_numMinutes",
        "fantasy_points_roll_10",
        "numMinutes_roll_10",
        "current_is_starter_roll_10",
        "usagePercentage_roll_10",
        "fantasy_points_momentum_5_vs_10",
        "team_pace_roll_10",
        "team_score_roll_10",
        "opp_fp_allowed_roll_10",
        "opp_defensiveRating_roll_10",
    ]
    plot_df = df.sample(min(sample_size, len(df)), random_state=42)
    for feature in features:
        if feature not in plot_df.columns:
            continue
        plt.figure(figsize=(8, 6))
        sns.regplot(data=plot_df, x=feature, y=TARGET_COL, scatter_kws={"alpha": 0.18, "s": 8}, line_kws={"color": "red"})
        plt.title(f"{feature} vs Fantasy Points")
        plt.xlabel(feature)
        plt.ylabel("Fantasy points")
        _savefig(figures_dir / f"eda_{feature}_vs_fp.png")


def plot_correlations(feature_rank: pd.DataFrame, figures_dir: Path = FIGURES_DIR) -> None:
    # Save a bar chart of the strongest feature-target correlations.
    top = feature_rank.head(30).sort_values("abs_correlation")
    plt.figure(figsize=(10, 10))
    plt.barh(top["feature"], top["correlation"])
    plt.title("Top Feature Correlations With Fantasy Points")
    plt.xlabel("Correlation")
    _savefig(figures_dir / "eda_top_feature_correlations.png")


def plot_missingness(missing: pd.DataFrame, figures_dir: Path = FIGURES_DIR) -> None:
    # Save a bar chart of the columns with the most missing values.
    top = missing[missing["missing_pct"].gt(0)].head(30).sort_values("missing_pct")
    if top.empty:
        return
    plt.figure(figsize=(10, 8))
    plt.barh(top["column"], top["missing_pct"])
    plt.title("Top Missing Feature Rates")
    plt.xlabel("Missing percentage")
    _savefig(figures_dir / "eda_missingness_top30.png")


def plot_correlation_heatmap(df: pd.DataFrame, feature_rank: pd.DataFrame, figures_dir: Path = FIGURES_DIR) -> None:
    # Save a heatmap for highly correlated feature groups.
    top_features = [feature for feature in feature_rank["feature"].head(24) if feature in df.columns]
    columns = [TARGET_COL] + top_features
    corr = df[columns].corr()
    plt.figure(figsize=(14, 12))
    sns.heatmap(corr, cmap="coolwarm", center=0, square=False)
    plt.title("Correlation Heatmap: Target And Top Features")
    _savefig(figures_dir / "eda_top_feature_correlation_heatmap.png")


def run_full_eda(
    modeling_path: Path | None = None,
    source_path: Path | None = None,
    tables_dir: Path = TABLES_DIR,
    figures_dir: Path = FIGURES_DIR,
) -> dict[str, pd.DataFrame]:
    # Run full EDA and write report tables/figures.
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    df = load_eda_data(modeling_path=modeling_path, source_path=source_path)
    outputs = {
        "eda_dataset_summary": dataset_summary(df),
        "eda_missingness_summary": missingness_summary(df),
    }
    outputs["eda_feature_correlations"] = feature_correlations(df)
    outputs["eda_current_game_diagnostic_correlations"] = current_game_diagnostic_correlations(df)
    outputs["eda_highly_correlated_feature_pairs"] = highly_correlated_pairs(df, outputs["eda_feature_correlations"])
    outputs["eda_target_by_position"] = target_summary_by(df, "primary_position")
    outputs["eda_target_by_starter"] = target_summary_by(df, "starter_label")
    outputs["eda_target_by_minutes_bucket"] = target_summary_by(df, "minutes_bucket")
    outputs["eda_target_by_season"] = target_summary_by(df, "season_start")
    outputs["eda_top_players_volume"] = player_volume_summary(df)
    outputs["eda_leakage_checks"] = leakage_checks(df)
    outputs.update(outlier_tables(df))

    for name, table in outputs.items():
        table.to_csv(tables_dir / f"{name}.csv", index=False)

    plot_target_distribution(df, figures_dir=figures_dir)
    plot_group_boxplots(df, figures_dir=figures_dir)
    plot_feature_relationships(df, figures_dir=figures_dir)
    plot_correlations(outputs["eda_feature_correlations"], figures_dir=figures_dir)
    plot_missingness(outputs["eda_missingness_summary"], figures_dir=figures_dir)
    plot_correlation_heatmap(df, outputs["eda_feature_correlations"], figures_dir=figures_dir)

    return outputs


In [8]:
eda_outputs = run_full_eda(modeling_path=PROCESSED_DIR / "modeling_table_last4_regular.csv", source_path=PROCESSED_DIR / "player_game_source_last4_regular.csv", tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR)
for name, table in eda_outputs.items():
    print(name, table.shape)
display(eda_outputs["eda_dataset_summary"])
display(eda_outputs["eda_feature_correlations"].head(15))
display(eda_outputs["eda_leakage_checks"])


eda_dataset_summary (15, 2)
eda_missingness_summary (590, 8)
eda_feature_correlations (560, 4)
eda_current_game_diagnostic_correlations (10, 5)
eda_highly_correlated_feature_pairs (60, 4)
eda_target_by_position (3, 9)
eda_target_by_starter (2, 9)
eda_target_by_minutes_bucket (5, 9)
eda_target_by_season (5, 9)
eda_top_players_volume (1032, 9)
eda_leakage_checks (4, 4)
eda_outlier_games (200, 14)
eda_starter_underperformance_games (200, 13)
eda_low_minutes_high_fp_games (200, 13)
eda_high_minutes_low_fp_games (200, 13)


,metric,value
0,rows,129750
1,columns,590
2,unique_players,1029
3,unique_games,6077
4,unique_player_games,129750
5,duplicate_player_games,0
6,date_min,2021-10-19 19:30:00
7,date_max,2026-04-12 20:30:00
8,seasons,"2021, 2022, 2023, 2024, 2025"
9,target_missing,0


,feature,correlation,abs_correlation,missing_pct
0,fantasy_points_roll_10,0.729771,0.729771,0.007931
1,fantasy_points_roll_30,0.725086,0.725086,0.007931
2,fantasy_points_roll_5,0.719723,0.719723,0.007931
3,fantasy_points_roll_3,0.702278,0.702278,0.007931
4,points_roll_10,0.690530,0.690530,0.007931
5,points_roll_30,0.687814,0.687814,0.007931
6,points_roll_5,0.678570,0.678570,0.007931
7,numMinutes_roll_10,0.650290,0.650290,0.007931
8,numMinutes_roll_5,0.647741,0.647741,0.007931
9,numMinutes_roll_30,0.642161,0.642161,0.007931


,check,value,passed,notes
0,duplicate_game_person_rows,0,True,modeling grain should be one row per player-game
1,direct_same_game_leakage_columns_present,,True,raw same-game stats should not be direct model...
2,first_player_game_fp_roll_5_null_rate,1.0,True,first row per player should not have previous-...
3,current_starter_feature_policy,True,True,current_is_starter is lineup-aware; early proj...


## 7. Feature Engineering

The core feature engineering method is shifted rolling history. This keeps useful basketball signal while preventing current-game outcomes from entering the model.


In [9]:
from pathlib import Path

import pandas as pd



TARGET_COL = "fantasy_points"

NON_FEATURE_COLUMNS = {
    "gameId",
    "personId",
    "game_date",
    "season_start",
    "firstName",
    "lastName",
    "player_name",
    "playerteamId",
    "playerteamCity",
    "playerteamName",
    "opponentteamId",
    "opponentteamCity",
    "opponentteamName",
    "startingPosition",
    "primary_position_group",
    TARGET_COL,
}

DIRECT_SAME_GAME_PREFIXES = ("actual_",)
DIRECT_SAME_GAME_FEATURES = {
    "starter_minutes",
    "bench_minutes",
}

STARTER_UNSTABLE_FEATURES = {
    "current_is_starter_std_30",
    "current_is_starter_z_5_vs_30",
    "current_is_starter_z_10_vs_30",
}

DROP_PREFIXES = (
    "estimatedDefensiveRating_",
    "team_estimatedDefensiveRating_",
    "opp_estimatedDefensiveRating_",
    "fieldGoalsPercentage_",
    "team_fieldGoalsPercentage_",
    "opp_fieldGoalsPercentage_allowed_",
)

DROP_Z_PREFIXES = (
    "blocks_z_",
    "steals_z_",
    "turnovers_z_",
    "threePointersPercentage_z_",
    "freeThrowsPercentage_z_",
)

MODEL_ALLOWED_Z_PREFIXES = (
    "fantasy_points_z_",
    "numMinutes_z_",
    "usagePercentage_z_",
)

BASELINE_FEATURES = [
    "current_is_starter",
    "fantasy_points_roll_10",
    "fantasy_points_roll_30",
    "numMinutes_roll_10",
    "numMinutes_roll_30",
]

PLAYER_ROLE_BASES = [
    "home",
    "guard",
    "forward",
    "center",
    "heightInches",
    "bodyWeightLbs",
    "years_since_draft",
    "days_since_last_game",
    "is_back_to_back",
    "current_is_starter",
    "starter_minutes",
    "bench_minutes",
    "starter_minutes_share",
    "fantasy_points",
    "numMinutes",
    "points",
    "reboundsTotal",
    "assists",
    "usagePercentage",
]

PLAYER_ADVANCED_BASES = [
    "trueShootingPercentage",
    "effectiveFieldGoalPercentage",
    "freeThrowsPercentage",
    "playerImpactEstimate",
    "offensiveRating",
    "defensiveRating",
    "pace",
    "steals",
    "blocks",
    "turnovers",
]

TEAM_OPP_PREFIXES = ("team_", "opp_", "pos_opp_")
PRE_LINEUP_EXCLUDED_FEATURES = {"current_is_starter"}

COMPACT_40_FEATURE_PLAN = [
    ("home", "game_context", "Home court can affect opportunity, rotation, and efficiency."),
    ("guard", "player_profile", "Position flag helps the model separate guard-style fantasy production."),
    ("forward", "player_profile", "Position flag helps the model separate forward-style fantasy production."),
    ("center", "player_profile", "Position flag helps the model separate center-style fantasy production."),
    ("years_since_draft", "player_profile", "Experience is a stable proxy for role and career stage."),
    ("days_since_last_game", "rest_context", "Rest can affect minutes and performance."),
    ("is_back_to_back", "rest_context", "Back-to-back games can affect rotation and fatigue."),
    ("fantasy_points_roll_5", "recent_production", "Short-term fantasy form."),
    ("fantasy_points_roll_10", "recent_production", "Medium-term fantasy form."),
    ("fantasy_points_roll_30", "recent_production", "Longer-term fantasy baseline."),
    ("fantasy_points_momentum_5_vs_10", "recent_production", "Recent fantasy trend versus medium-term baseline."),
    ("fantasy_points_momentum_10_vs_30", "recent_production", "Medium-term fantasy trend versus long-term baseline."),
    ("numMinutes_roll_5", "playing_time", "Short-term playing-time role."),
    ("numMinutes_roll_10", "playing_time", "Medium-term playing-time role."),
    ("numMinutes_roll_30", "playing_time", "Longer-term playing-time baseline."),
    ("current_is_starter_roll_10", "starter_history", "Historical starter rate without using today's lineup."),
    ("starter_minutes_roll_5", "starter_history", "Recent minutes earned as a starter."),
    ("starter_minutes_roll_10", "starter_history", "Medium-term starter-minute role."),
    ("starter_minutes_roll_30", "starter_history", "Longer-term starter-minute role."),
    ("starter_minutes_share_roll_10", "starter_history", "Share of recent minutes that came as a starter."),
    ("usagePercentage_roll_5", "offensive_involvement", "Short-term offensive involvement."),
    ("usagePercentage_roll_10", "offensive_involvement", "Medium-term offensive involvement."),
    ("usagePercentage_roll_30", "offensive_involvement", "Longer-term offensive involvement baseline."),
    ("usagePercentage_momentum_5_vs_10", "offensive_involvement", "Recent usage trend."),
    ("trueShootingPercentage_roll_10", "efficiency", "Recent scoring efficiency."),
    ("playerImpactEstimate_roll_10", "advanced_player_quality", "Recent all-around player impact."),
    ("points_roll_10", "box_score_profile", "Recent scoring contribution."),
    ("reboundsTotal_roll_10", "box_score_profile", "Recent rebounding contribution."),
    ("assists_roll_10", "box_score_profile", "Recent playmaking contribution."),
    ("turnovers_roll_10", "box_score_profile", "Recent negative fantasy component and ball-handling load."),
    ("team_pace_roll_10", "team_context", "Team possession environment."),
    ("team_offensiveRating_roll_10", "team_context", "Recent team offensive quality."),
    ("team_assists_roll_10", "team_context", "Recent team ball movement."),
    ("team_turnovers_roll_10", "team_context", "Recent team turnover environment."),
    ("opp_pace_roll_10", "opponent_context", "Opponent possession environment."),
    ("opp_defensiveRating_roll_10", "opponent_context", "Opponent defensive strength."),
    ("opp_points_allowed_roll_10", "opponent_context", "Opponent recent scoring allowed."),
    ("opp_fp_allowed_roll_10", "opponent_context", "Opponent recent total fantasy points allowed."),
    ("opp_pos_fp_allowed_per_player_roll_30", "positional_matchup", "Opponent fantasy points allowed to the player's position."),
    ("pos_opp_difficulty_roll_30", "positional_matchup", "Position-specific opponent difficulty versus league position baseline."),
]

LINEUP_AWARE_COMPACT_EXTRA = (
    "current_is_starter",
    "lineup_context",
    "Known starter status for lineup-aware upper-bound experiments only.",
)


def _has_any_prefix(column: str, prefixes: tuple[str, ...]) -> bool:
    # Return whether a column starts with any prefix in a prefix list.
    return any(column.startswith(prefix) for prefix in prefixes)


def _is_roll_family(column: str, base: str) -> bool:
    # Return whether a column belongs to a rolling feature family for a base stat.
    return column.startswith(f"{base}_") and (
        "_roll_" in column or "_momentum_" in column or "_std_" in column or "_z_" in column
    )


def drop_reason(column: str) -> str | None:
    # Return the first model-drop reason for a column, or None if it is eligible.
    if column in NON_FEATURE_COLUMNS:
        return "identifier, label, date, or target column; not a numeric model feature"
    if _has_any_prefix(column, DIRECT_SAME_GAME_PREFIXES):
        return "EDA-only same-game diagnostic column; not known before prediction"
    if column in DIRECT_SAME_GAME_FEATURES:
        return "same-game role-minute diagnostic column; use shifted rolling versions for pre-game models"
    if column in STARTER_UNSTABLE_FEATURES:
        return "binary starter z/std feature is unstable and has high missingness"
    if _has_any_prefix(column, DROP_PREFIXES):
        return "redundant first-pass feature family based on EDA redundancy review"
    if _has_any_prefix(column, DROP_Z_PREFIXES):
        return "noisy low-volume z-score feature; keep rolling/momentum first"
    return None


def eligible_model_features(columns: list[str]) -> list[str]:
    # Return numeric model-eligible columns after first-pass drop policy.
    return [column for column in columns if drop_reason(column) is None]


def feature_drop_decisions(columns: list[str], eda_correlations: pd.DataFrame | None = None) -> pd.DataFrame:
    # Create detailed keep/drop decisions for every modeling-table column.
    corr_map = {}
    missing_map = {}
    if eda_correlations is not None and not eda_correlations.empty:
        corr_map = eda_correlations.set_index("feature")["correlation"].to_dict()
        missing_map = eda_correlations.set_index("feature")["missing_pct"].to_dict()

    rows = []
    for column in columns:
        reason = drop_reason(column)
        rows.append(
            {
                "feature": column,
                "decision": "drop" if reason else "eligible",
                "reason": reason or "eligible for at least one first-pass feature set",
                "correlation_with_target": corr_map.get(column),
                "missing_pct": missing_map.get(column),
            }
        )
    return pd.DataFrame(rows)


def _family_features(columns: list[str], bases: list[str]) -> list[str]:
    # Collect eligible rolling-family features for selected base stats.
    selected = []
    for column in columns:
        if column in bases or any(_is_roll_family(column, base) for base in bases):
            selected.append(column)
    return selected


def _ordered_existing(columns: list[str], selected: list[str]) -> list[str]:
    # Return selected features in modeling-table order while dropping missing columns.
    selected_set = set(selected)
    return [column for column in columns if column in selected_set]


def compact_feature_report(columns: list[str]) -> pd.DataFrame:
    # Return the compact feature plan with availability and drop-policy details.
    eligible = set(eligible_model_features(columns))
    available = set(columns)
    rows = []
    for position, (feature, group, reason) in enumerate(COMPACT_40_FEATURE_PLAN, start=1):
        rows.append(
            {
                "feature_set": "pre_lineup_compact_40",
                "position": position,
                "feature": feature,
                "feature_group": group,
                "selection_reason": reason,
                "available": feature in available,
                "eligible": feature in eligible,
                "drop_reason": drop_reason(feature),
            }
        )

    feature, group, reason = LINEUP_AWARE_COMPACT_EXTRA
    rows.append(
        {
            "feature_set": "lineup_aware_compact_40",
            "position": 1,
            "feature": feature,
            "feature_group": group,
            "selection_reason": reason,
            "available": feature in available,
            "eligible": feature in eligible,
            "drop_reason": drop_reason(feature),
        }
    )
    return pd.DataFrame(rows)


def build_compact_feature_sets(columns: list[str]) -> dict[str, list[str]]:
    # Build compact, lecturer-explainable feature sets around 40 columns.
    eligible = set(eligible_model_features(columns))
    compact = [feature for feature, _, _ in COMPACT_40_FEATURE_PLAN if feature in eligible]

    lineup_extra = LINEUP_AWARE_COMPACT_EXTRA[0]
    lineup_compact = ([lineup_extra] if lineup_extra in eligible else []) + compact

    return {
        "lineup_aware_compact_40": lineup_compact,
        "pre_lineup_compact_40": compact,
    }


def build_lineup_aware_feature_sets(columns: list[str]) -> dict[str, list[str]]:
    # Build named lineup-aware feature sets from available modeling columns.
    eligible = eligible_model_features(columns)

    baseline = [column for column in BASELINE_FEATURES if column in eligible]
    player_role = sorted(
        set(baseline)
        | set(_family_features(eligible, PLAYER_ROLE_BASES))
        | {"home", "guard", "forward", "center", "heightInches", "bodyWeightLbs", "years_since_draft", "days_since_last_game", "is_back_to_back"}
    )
    player_role = _ordered_existing(eligible, list(player_role))

    player_advanced = sorted(set(player_role) | set(_family_features(eligible, PLAYER_ADVANCED_BASES)))
    player_advanced = _ordered_existing(eligible, list(player_advanced))

    team_opp = [column for column in eligible if column.startswith(TEAM_OPP_PREFIXES)]
    player_team_opp = _ordered_existing(eligible, list(set(player_advanced) | set(team_opp)))

    return {
        "lineup_aware_baseline": baseline,
        "lineup_aware_player_role": player_role,
        "lineup_aware_player_advanced": player_advanced,
        "lineup_aware_player_team_opp": player_team_opp,
    }


def build_pre_lineup_feature_sets(columns: list[str]) -> dict[str, list[str]]:
    # Build feature sets for pre-lineup prediction.
    #
    # These exclude current-game starter status but keep shifted starter rolling
    # history such as `current_is_starter_roll_10`.
    lineup_sets = build_lineup_aware_feature_sets(columns)
    feature_sets = {}
    for name, features in lineup_sets.items():
        pre_name = name.replace("lineup_aware", "pre_lineup")
        feature_sets[pre_name] = [feature for feature in features if feature not in PRE_LINEUP_EXCLUDED_FEATURES]
    return feature_sets


def build_all_feature_sets(columns: list[str]) -> dict[str, list[str]]:
    # Build both lineup-aware and pre-lineup feature sets.
    feature_sets = {}
    feature_sets.update(build_lineup_aware_feature_sets(columns))
    feature_sets.update(build_pre_lineup_feature_sets(columns))
    feature_sets.update(build_compact_feature_sets(columns))
    return feature_sets


def feature_set_summary(feature_sets: dict[str, list[str]], eda_correlations: pd.DataFrame | None = None) -> pd.DataFrame:
    # Summarize feature-set sizes and correlation coverage.
    corr_map = {}
    if eda_correlations is not None and not eda_correlations.empty:
        corr_map = eda_correlations.set_index("feature")["abs_correlation"].to_dict()

    rows = []
    for name, features in feature_sets.items():
        correlations = [corr_map[feature] for feature in features if feature in corr_map and pd.notna(corr_map[feature])]
        rows.append(
            {
                "feature_set": name,
                "n_features": len(features),
                "mean_abs_correlation": sum(correlations) / len(correlations) if correlations else None,
                "max_abs_correlation": max(correlations) if correlations else None,
                "n_features_with_correlation": len(correlations),
            }
        )
    return pd.DataFrame(rows)


def feature_set_membership_table(feature_sets: dict[str, list[str]]) -> pd.DataFrame:
    # Return one row per feature per feature set.
    rows = []
    for name, features in feature_sets.items():
        for position, feature in enumerate(features, start=1):
            rows.append({"feature_set": name, "feature": feature, "position": position})
    return pd.DataFrame(rows)


def load_feature_set_inputs(
    modeling_path: Path | None = None,
    correlations_path: Path | None = None,
) -> tuple[list[str], pd.DataFrame]:
    # Load modeling schema and EDA correlations for feature-set generation.
    modeling_path = modeling_path or PROCESSED_DIR / "modeling_table_last4_regular.csv"
    correlations_path = correlations_path or TABLES_DIR / "eda_feature_correlations.csv"

    columns = pd.read_csv(modeling_path, nrows=0).columns.tolist()
    eda_correlations = pd.read_csv(correlations_path) if correlations_path.exists() else pd.DataFrame()
    return columns, eda_correlations


def write_feature_set_reports(
    modeling_path: Path | None = None,
    correlations_path: Path | None = None,
    output_dir: Path = TABLES_DIR,
) -> dict[str, pd.DataFrame]:
    # Build and write feature-set and drop-decision reports.
    output_dir.mkdir(parents=True, exist_ok=True)
    columns, eda_correlations = load_feature_set_inputs(modeling_path=modeling_path, correlations_path=correlations_path)
    feature_sets = build_all_feature_sets(columns)

    outputs = {
        "feature_drop_decisions_v1": feature_drop_decisions(columns, eda_correlations=eda_correlations),
        "feature_set_summary": feature_set_summary(feature_sets, eda_correlations=eda_correlations),
        "feature_set_membership": feature_set_membership_table(feature_sets),
        "compact_40_feature_plan": compact_feature_report(columns),
    }
    for name, table in outputs.items():
        table.to_csv(output_dir / f"{name}.csv", index=False)
    return outputs


## 8. Chronological Train/Validation/Test Split

The course workflow usually describes train/test splitting generically. Here the split is chronological because games are time ordered.

- Train: full 2021, 2022, and 2023 seasons
- Validation: full 2024 season
- Test: full 2025 season


In [10]:
from pathlib import Path

import pandas as pd


SPLIT_COL = "split_2021_train_2024_validation_2025_test"


def half_season_cutoff_date(df: pd.DataFrame, season_start: int = 2024, date_col: str = "game_date") -> pd.Timestamp:
    # Return the midpoint date by unique game dates for one season.
    #
    # This helper is kept for auditability because the old notebook used a half-2024
    # validation split. The polished notebook uses full seasons instead.
    season_dates = (
        pd.to_datetime(df.loc[df["season_start"].eq(season_start), date_col], errors="coerce")
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )
    if season_dates.empty:
        raise ValueError(f"No dates found for season_start={season_start}.")

    midpoint_index = (len(season_dates) // 2) - 1
    midpoint_index = max(midpoint_index, 0)
    return season_dates.iloc[midpoint_index]


def assign_split_v1(df: pd.DataFrame, date_col: str = "game_date") -> pd.Series:
    # Assign full-season chronological train/validation/test labels.
    #
    # Train: 2021, 2022, 2023 seasons.
    # Validation: full 2024 season.
    # Test: full 2025 season, untouched for model choice.
    result = pd.Series("exclude", index=df.index, dtype="string")
    result.loc[df["season_start"].isin([2021, 2022, 2023])] = "train"
    result.loc[df["season_start"].eq(2024)] = "validation"
    result.loc[df["season_start"].eq(2025)] = "test"
    return result


def split_summary(df: pd.DataFrame, split_col: str = SPLIT_COL) -> pd.DataFrame:
    # Summarize rows, dates, seasons, games, and players by split.
    rows = []
    data = df.copy()
    data["game_date"] = pd.to_datetime(data["game_date"], errors="coerce")
    for split_name, group in data.groupby(split_col, sort=False):
        rows.append(
            {
                "split": split_name,
                "rows": len(group),
                "unique_games": group["gameId"].nunique(),
                "unique_players": group["personId"].nunique(),
                "date_min": group["game_date"].min(),
                "date_max": group["game_date"].max(),
                "seasons": ", ".join(map(str, sorted(group["season_start"].dropna().unique()))),
                "target_mean": group["fantasy_points"].mean(),
                "target_median": group["fantasy_points"].median(),
                "target_std": group["fantasy_points"].std(),
            }
        )
    order = {"train": 0, "validation": 1, "test": 2, "exclude": 3}
    return pd.DataFrame(rows).sort_values("split", key=lambda s: s.map(order).fillna(99)).reset_index(drop=True)


def write_split_reports(
    modeling_path: Path | None = None,
    output_dir: Path = TABLES_DIR,
    processed_dir: Path = PROCESSED_DIR,
) -> dict[str, pd.DataFrame]:
    # Write split assignment and summary reports.
    modeling_path = modeling_path or processed_dir / "modeling_table_last4_regular.csv"
    output_dir.mkdir(parents=True, exist_ok=True)

    id_cols = ["gameId", "personId", "game_date", "season_start", "fantasy_points"]
    data = pd.read_csv(modeling_path, usecols=id_cols, low_memory=False)
    data[SPLIT_COL] = assign_split_v1(data)

    rows_path = output_dir / "modeling_rows_with_split_v1.csv"
    summary_path = output_dir / "split_summary_v1.csv"
    data.to_csv(rows_path, index=False)
    summary = split_summary(data)
    summary.to_csv(summary_path, index=False)

    return {
        "modeling_rows_with_split_v1": data,
        "split_summary_v1": summary,
    }


In [11]:
split_rows = modeling[["gameId", "personId", "game_date", "season_start", "fantasy_points"]].copy()
split_rows[SPLIT_COL] = assign_split_v1(split_rows)
split_report = split_summary(split_rows)
split_rows.to_csv(TABLES_DIR / "modeling_rows_with_split_v1.csv", index=False)
split_report.to_csv(TABLES_DIR / "split_summary_v1.csv", index=False)
display(split_report)


,split,rows,unique_games,unique_players,date_min,date_max,seasons,target_mean,target_median,target_std
0,train,76937,3624,818,2021-10-19 19:30:00,2024-04-14 15:30:00,"2021, 2022, 2023",21.458609,19.4,14.866904
1,validation,26162,1223,569,2024-10-22 19:30:00,2025-04-13 15:30:00,2024,21.722498,20.0,14.911720
2,test,26651,1230,582,2025-10-21 19:30:00,2026-04-12 20:30:00,2025,21.620104,20.0,14.499562


## 9. Feature Sets And Modeling Datasets

This section creates the broad, compact, lineup-aware, starter-history, and leakage-diagnostic feature sets used later by the models.


In [12]:
feature_outputs = write_feature_set_reports(
    modeling_path=PROCESSED_DIR / "modeling_table_last4_regular.csv",
    correlations_path=TABLES_DIR / "eda_feature_correlations.csv",
    output_dir=TABLES_DIR,
)

# Compact scoring variant: the compact 40 plus direct historical scoring form.
SCORING_REFINEMENT_FEATURES = [
    "points_roll_5",
    "points_roll_30",
    "points_momentum_5_vs_10",
    "points_momentum_10_vs_30",
    "usagePercentage_z_5_vs_30",
]

# Starter-focused valid variants.
STARTER_HISTORY_FEATURES = [
    "home",
    "days_since_last_game",
    "is_back_to_back",
    "fantasy_points_roll_10",
    "numMinutes_roll_10",
    "current_is_starter_roll_10",
    "starter_minutes_roll_10",
    "starter_minutes_share_roll_10",
    "usagePercentage_roll_10",
    "pos_opp_difficulty_roll_30",
]

membership = feature_outputs["feature_set_membership"].copy()
feature_sets = {
    name: group.sort_values("position")["feature"].tolist()
    for name, group in membership.groupby("feature_set", sort=False)
}

compact = feature_sets["pre_lineup_compact_40"]
compact_scoring = list(dict.fromkeys(compact + [feature for feature in SCORING_REFINEMENT_FEATURES if feature in modeling.columns]))
eligible_for_starter = set(eligible_model_features(list(modeling.columns)))
starter_history = [
    feature
    for feature in STARTER_HISTORY_FEATURES
    if feature in eligible_for_starter and feature not in PRE_LINEUP_EXCLUDED_FEATURES
]

lineup_compact = feature_sets["lineup_aware_compact_40"]
lineup_compact_scoring = list(dict.fromkeys(lineup_compact + [feature for feature in SCORING_REFINEMENT_FEATURES if feature in modeling.columns]))
lineup_starter_flag_roll10 = list(dict.fromkeys([feature for feature in ["current_is_starter", *starter_history] if feature in eligible_model_features(list(modeling.columns))]))

extra_feature_sets = {
    "pre_lineup_compact_40_plus_scoring": compact_scoring,
    "pre_lineup_starter_history_roll10": starter_history,
    "lineup_aware_starter_flag_roll10": lineup_starter_flag_roll10,
    "lineup_aware_compact_40_plus_scoring": lineup_compact_scoring,
}
extra_rows = []
for feature_set, features in extra_feature_sets.items():
    for position, feature in enumerate(features, start=1):
        extra_rows.append({"feature_set": feature_set, "feature": feature, "position": position})

membership = pd.concat(
    [membership[~membership["feature_set"].isin(extra_feature_sets)], pd.DataFrame(extra_rows)],
    ignore_index=True,
)
feature_sets = {
    name: group.sort_values("position")["feature"].tolist()
    for name, group in membership.groupby("feature_set", sort=False)
}
feature_outputs["feature_set_membership"] = membership
feature_outputs["feature_set_summary"] = feature_set_summary(feature_sets, eda_outputs["eda_feature_correlations"])
feature_outputs["feature_set_membership"].to_csv(TABLES_DIR / "feature_set_membership.csv", index=False)
feature_outputs["feature_set_summary"].to_csv(TABLES_DIR / "feature_set_summary.csv", index=False)

starter_validity = pd.DataFrame(
    [
        {"feature_set": "pre_lineup_starter_history_roll10", "validity": "pre-lineup", "uses_current_is_starter": False, "uses_current_game_minutes": False, "reason": "Uses shifted starter history, including starter_minutes_roll_10."},
        {"feature_set": "lineup_aware_starter_flag_roll10", "validity": "lineup-aware", "uses_current_is_starter": True, "uses_current_game_minutes": False, "reason": "Uses today's current_is_starter plus shifted starter_minutes_roll_10 history."},
        {"feature_set": "lineup_aware_compact_40_plus_scoring", "validity": "lineup-aware", "uses_current_is_starter": True, "uses_current_game_minutes": False, "reason": "Uses current_is_starter only; valid after lineup news is known."},
    ]
)
starter_validity.to_csv(TABLES_DIR / "starter_feature_set_validity.csv", index=False)

for name, table in feature_outputs.items():
    print(name, table.shape)
display(feature_outputs["feature_set_summary"])
display(starter_validity)


feature_drop_decisions_v1 (576, 5)
feature_set_summary (14, 5)
feature_set_membership (1681, 3)
compact_40_feature_plan (41, 8)


,feature_set,n_features,mean_abs_correlation,max_abs_correlation,n_features_with_correlation
0,lineup_aware_baseline,5,0.647856,0.729771,5
1,lineup_aware_player_role,91,0.272515,0.729771,91
2,lineup_aware_player_advanced,163,0.240312,0.729771,163
3,lineup_aware_player_team_opp,487,0.088598,0.729771,482
4,pre_lineup_baseline,4,0.686827,0.729771,4
5,pre_lineup_player_role,90,0.270076,0.729771,90
6,pre_lineup_player_advanced,162,0.238758,0.729771,162
7,pre_lineup_player_team_opp,486,0.087759,0.729771,481
8,lineup_aware_compact_40,41,0.304237,0.729771,41
9,pre_lineup_compact_40,40,0.299544,0.729771,40


,feature_set,validity,uses_current_is_starter,uses_current_game_minutes,reason
0,pre_lineup_starter_history_roll10,pre-lineup,False,False,"Uses shifted starter history, including starte..."
1,lineup_aware_starter_flag_roll10,lineup-aware,True,False,Uses today's current_is_starter plus shifted s...
2,lineup_aware_compact_40_plus_scoring,lineup-aware,True,False,Uses current_is_starter only; valid after line...


## 10. Main Model Comparison

The main comparison evaluates simple baselines, Ridge, and HistGradientBoosting on selected legal feature sets. The validation season is 2024.


In [13]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score


def time_train_validation_test_split(
    df: pd.DataFrame,
    date_col: str = "game_date",
    train_size: float = 0.70,
    validation_size: float = 0.15,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Split rows chronologically into train, validation, and test sets.
    if train_size + validation_size >= 1:
        raise ValueError("train_size + validation_size must be less than 1.")

    ordered = df.sort_values(date_col).reset_index(drop=True)
    train_end = int(len(ordered) * train_size)
    validation_end = int(len(ordered) * (train_size + validation_size))
    return (
        ordered.iloc[:train_end].copy(),
        ordered.iloc[train_end:validation_end].copy(),
        ordered.iloc[validation_end:].copy(),
    )


def expanding_time_cv_splits(
    df: pd.DataFrame,
    date_col: str = "game_date",
    n_splits: int = 4,
) -> list[tuple[np.ndarray, np.ndarray]]:
    # Create expanding-window cross-validation splits over chronological rows.
    ordered = df.sort_values(date_col).reset_index(drop=True)
    indices = np.arange(len(ordered))
    fold_size = len(ordered) // (n_splits + 1)
    splits = []

    for fold in range(1, n_splits + 1):
        train_end = fold * fold_size
        validation_end = (fold + 1) * fold_size
        splits.append((indices[:train_end], indices[train_end:validation_end]))

    return splits


def mean_baseline_prediction(y_train: pd.Series, n_predictions: int) -> np.ndarray:
    # Predict the training target mean for each validation/test row.
    return np.full(n_predictions, float(pd.Series(y_train).mean()))


def regression_metrics(y_true, y_pred) -> dict[str, float]:
    # Return standard regression metrics for model comparison.
    y_true_array = np.asarray(y_true)
    y_pred_array = np.asarray(y_pred)
    errors = y_true_array - y_pred_array

    return {
        "mae": float(mean_absolute_error(y_true_array, y_pred_array)),
        "rmse": float(np.sqrt(np.mean(errors**2))),
        "r2": float(r2_score(y_true_array, y_pred_array)),
    }


In [14]:
from dataclasses import dataclass
import os
from pathlib import Path
from time import perf_counter

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler



TARGET_COL = "fantasy_points"
MODEL_RANDOM_STATE = 42
MODEL_NAMES = [
    "mean_baseline",
    "rolling_fp_10_baseline",
    "rolling_fp_30_baseline",
    "ridge",
    "random_forest",
    "hist_gradient_boosting",
]


@dataclass(frozen=True)
class ValidationData:
    data: pd.DataFrame
    feature_sets: dict[str, list[str]]


def load_feature_sets(path: Path | None = None) -> dict[str, list[str]]:
    # Load feature-set membership report as an ordered dictionary.
    path = path or TABLES_DIR / "feature_set_membership.csv"
    membership = pd.read_csv(path).sort_values(["feature_set", "position"])
    return {
        feature_set: group["feature"].tolist()
        for feature_set, group in membership.groupby("feature_set", sort=False)
    }


def load_validation_data(
    modeling_path: Path | None = None,
    split_path: Path | None = None,
    feature_set_path: Path | None = None,
    source_path: Path | None = None,
) -> ValidationData:
    # Load train/validation rows, feature columns, split labels, and EDA-only segment columns.
    modeling_path = modeling_path or PROCESSED_DIR / "modeling_table_last4_regular.csv"
    split_path = split_path or TABLES_DIR / "modeling_rows_with_split_v1.csv"
    source_path = source_path or PROCESSED_DIR / "player_game_source_last4_regular.csv"
    feature_sets = load_feature_sets(feature_set_path)

    feature_union = sorted({feature for features in feature_sets.values() for feature in features})
    meta_cols = [
        "gameId",
        "personId",
        "game_date",
        "season_start",
        "player_name",
        "playerteamName",
        "opponentteamName",
        "current_is_starter",
        "guard",
        "forward",
        "center",
        TARGET_COL,
    ]
    available_cols = pd.read_csv(modeling_path, nrows=0).columns
    usecols = [column for column in dict.fromkeys(meta_cols + feature_union) if column in available_cols]
    data = pd.read_csv(modeling_path, usecols=usecols, low_memory=False)

    splits = pd.read_csv(split_path, usecols=["gameId", "personId", SPLIT_COL], low_memory=False)
    data = data.merge(splits, on=["gameId", "personId"], how="left", validate="one_to_one")

    source_cols = ["gameId", "personId", "numMinutes"]
    source_available = pd.read_csv(source_path, nrows=0).columns
    source_cols = [column for column in source_cols if column in source_available]
    if len(source_cols) == 3:
        source = pd.read_csv(source_path, usecols=source_cols, low_memory=False).rename(columns={"numMinutes": "actual_numMinutes"})
        data = data.merge(source, on=["gameId", "personId"], how="left", validate="one_to_one")

    data["game_date"] = pd.to_datetime(data["game_date"], errors="coerce")
    data = add_validation_segments(data)
    return ValidationData(data=data, feature_sets=feature_sets)


def add_validation_segments(df: pd.DataFrame) -> pd.DataFrame:
    # Add validation error-analysis segments.
    result = df.copy()
    result["starter_label"] = np.where(result["current_is_starter"].eq(1), "Starter", "Bench")
    result["primary_position"] = "Unknown"
    result.loc[result.get("guard", pd.Series(False, index=result.index)).eq(1), "primary_position"] = "Guard"
    result.loc[result.get("forward", pd.Series(False, index=result.index)).eq(1), "primary_position"] = "Forward"
    result.loc[result.get("center", pd.Series(False, index=result.index)).eq(1), "primary_position"] = "Center"
    if "actual_numMinutes" in result.columns:
        result["minutes_bucket"] = pd.cut(
            pd.to_numeric(result["actual_numMinutes"], errors="coerce"),
            bins=[-0.1, 5, 15, 25, 35, np.inf],
            labels=["0-5", "5-15", "15-25", "25-35", "35+"],
        )
    result["fp_bucket"] = pd.cut(
        result[TARGET_COL],
        bins=[-np.inf, 10, 20, 30, 40, 50, np.inf],
        labels=["<=10", "10-20", "20-30", "30-40", "40-50", "50+"],
    )
    return result


def make_model(model_name: str):
    # Return a model pipeline for numeric tabular features.
    if model_name == "ridge":
        return Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=10.0)),
            ]
        )
    if model_name == "random_forest":
        return Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                (
                    "model",
                    RandomForestRegressor(
                        n_estimators=60,
                        max_depth=18,
                        min_samples_leaf=10,
                        max_features=0.5,
                        n_jobs=1,
                        random_state=MODEL_RANDOM_STATE,
                    ),
                ),
            ]
        )
    if model_name == "hist_gradient_boosting":
        return Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                (
                    "model",
                    HistGradientBoostingRegressor(
                        learning_rate=0.05,
                        max_iter=160,
                        max_leaf_nodes=31,
                        l2_regularization=0.1,
                        random_state=MODEL_RANDOM_STATE,
                    ),
                ),
            ]
        )
    raise ValueError(f"Unknown fitted model: {model_name}")


def _baseline_predictions(model_name: str, y_train: pd.Series, validation: pd.DataFrame) -> np.ndarray:
    # Generate validation predictions for the simple baseline models.
    if model_name == "mean_baseline":
        return np.full(len(validation), float(y_train.mean()))
    if model_name == "rolling_fp_10_baseline":
        return validation["fantasy_points_roll_10"].fillna(float(y_train.mean())).to_numpy()
    if model_name == "rolling_fp_30_baseline":
        return validation["fantasy_points_roll_30"].fillna(float(y_train.mean())).to_numpy()
    raise ValueError(f"Unknown baseline model: {model_name}")


def _validate_feature_frame(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    # Check that all requested features exist and coerce them to numeric values.
    missing = [feature for feature in features if feature not in df.columns]
    if missing:
        raise ValueError(f"Missing features in modeling table: {missing[:20]}")
    return df[features].apply(pd.to_numeric, errors="coerce")


def run_one_experiment(
    data: pd.DataFrame,
    feature_set: str,
    features: list[str],
    model_name: str,
) -> tuple[dict[str, object], pd.DataFrame]:
    # Fit/evaluate one model on train and validation only.
    train = data[data[SPLIT_COL].eq("train")].copy()
    validation = data[data[SPLIT_COL].eq("validation")].copy()
    if train.empty or validation.empty:
        raise ValueError("Train and validation splits must both be non-empty.")
    if data[data[SPLIT_COL].eq("test")].index.intersection(train.index).any():
        raise ValueError("Test rows leaked into train.")

    y_train = train[TARGET_COL]
    y_validation = validation[TARGET_COL]
    start = perf_counter()

    if model_name.endswith("_baseline"):
        predictions = _baseline_predictions(model_name, y_train, validation)
    else:
        model = make_model(model_name)
        x_train = _validate_feature_frame(train, features)
        x_validation = _validate_feature_frame(validation, features)
        model.fit(x_train, y_train)
        predictions = model.predict(x_validation)

    elapsed = perf_counter() - start
    metrics = regression_metrics(y_validation, predictions)
    errors = y_validation.to_numpy() - predictions

    row = {
        "feature_set": feature_set,
        "model": model_name,
        "n_features": len(features),
        "train_rows": len(train),
        "validation_rows": len(validation),
        "fit_predict_seconds": round(elapsed, 3),
        "mae": metrics["mae"],
        "rmse": metrics["rmse"],
        "r2": metrics["r2"],
        "mean_error": float(np.mean(errors)),
        "median_absolute_error": float(np.median(np.abs(errors))),
    }

    prediction_cols = [
        "gameId",
        "personId",
        "game_date",
        "season_start",
        "player_name",
        "playerteamName",
        "opponentteamName",
        "starter_label",
        "primary_position",
        "minutes_bucket",
        "fp_bucket",
        "actual_numMinutes",
        TARGET_COL,
    ]
    prediction_cols = [column for column in prediction_cols if column in validation.columns]
    pred_df = validation[prediction_cols].copy()
    pred_df["feature_set"] = feature_set
    pred_df["model"] = model_name
    pred_df["prediction"] = predictions
    pred_df["error"] = pred_df[TARGET_COL] - pred_df["prediction"]
    pred_df["absolute_error"] = pred_df["error"].abs()
    return row, pred_df


def error_by_segment(predictions: pd.DataFrame) -> pd.DataFrame:
    # Summarize validation error by key segments.
    rows = []
    for segment in ["starter_label", "primary_position", "minutes_bucket", "fp_bucket"]:
        if segment not in predictions.columns:
            continue
        grouped = predictions.groupby(["feature_set", "model", segment], observed=True)
        for keys, group in grouped:
            feature_set, model_name, value = keys
            rows.append(
                {
                    "feature_set": feature_set,
                    "model": model_name,
                    "segment": segment,
                    "segment_value": value,
                    "rows": len(group),
                    "mae": float(group["absolute_error"].mean()),
                    "rmse": float(np.sqrt(np.mean(group["error"] ** 2))),
                    "mean_error": float(group["error"].mean()),
                    "target_mean": float(group[TARGET_COL].mean()),
                    "prediction_mean": float(group["prediction"].mean()),
                }
            )
    return pd.DataFrame(rows)


def feature_set_inputs(feature_sets: dict[str, list[str]]) -> pd.DataFrame:
    # Return one row per feature used in each model feature set.
    rows = []
    for feature_set, features in feature_sets.items():
        for position, feature in enumerate(features, start=1):
            rows.append({"feature_set": feature_set, "feature": feature, "position": position})
    return pd.DataFrame(rows)


def _savefig(path: Path) -> None:
    # Save the current Matplotlib figure to disk and close it.
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def write_validation_plots(predictions: pd.DataFrame, results: pd.DataFrame, figures_dir: Path = FIGURES_DIR) -> None:
    # Write actual-vs-predicted and residual plots for top validation models.
    figures_dir.mkdir(parents=True, exist_ok=True)
    top = results.sort_values("mae").head(6)[["feature_set", "model"]]
    for record in top.to_dict("records"):
        mask = predictions["feature_set"].eq(record["feature_set"]) & predictions["model"].eq(record["model"])
        plot_df = predictions.loc[mask]
        suffix = f"{record['feature_set']}__{record['model']}"

        sample = plot_df.sample(min(len(plot_df), 20_000), random_state=MODEL_RANDOM_STATE)
        plt.figure(figsize=(7, 7))
        plt.scatter(sample[TARGET_COL], sample["prediction"], alpha=0.25, s=10)
        min_value = min(sample[TARGET_COL].min(), sample["prediction"].min())
        max_value = max(sample[TARGET_COL].max(), sample["prediction"].max())
        plt.plot([min_value, max_value], [min_value, max_value], color="red", linewidth=1)
        plt.xlabel("Actual fantasy points")
        plt.ylabel("Predicted fantasy points")
        plt.title(f"Validation Actual vs Predicted: {suffix}")
        _savefig(figures_dir / f"validation_actual_vs_predicted_{suffix}.png")

        plt.figure(figsize=(9, 5))
        plt.hist(plot_df["error"], bins=60)
        plt.xlabel("Actual - predicted")
        plt.ylabel("Rows")
        plt.title(f"Validation Residuals: {suffix}")
        _savefig(figures_dir / f"validation_residuals_{suffix}.png")


def run_validation_experiments(
    model_names: list[str] | None = None,
    tables_dir: Path = TABLES_DIR,
    figures_dir: Path = FIGURES_DIR,
) -> dict[str, pd.DataFrame]:
    # Run validation-only experiments and write report tables/figures.
    model_names = model_names or MODEL_NAMES
    loaded = load_validation_data()
    data = loaded.data
    feature_sets = loaded.feature_sets

    if data[data[SPLIT_COL].eq("test")].empty:
        raise ValueError("Expected untouched test rows in split report.")

    result_rows = []
    prediction_tables = []
    total_runs = len(feature_sets) * len(model_names)
    completed_runs = 0
    for feature_set, features in feature_sets.items():
        for model_name in model_names:
            completed_runs += 1
            print(f"[{completed_runs}/{total_runs}] {feature_set} / {model_name}", flush=True)
            row, preds = run_one_experiment(data, feature_set, features, model_name)
            result_rows.append(row)
            prediction_tables.append(preds)

            partial_results = pd.DataFrame(result_rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
            partial_predictions = pd.concat(prediction_tables, ignore_index=True)
            tables_dir.mkdir(parents=True, exist_ok=True)
            partial_results.to_csv(tables_dir / "validation_model_results_v1.partial.csv", index=False)
            partial_predictions.to_csv(tables_dir / "validation_predictions_v1.partial.csv", index=False)

    results = pd.DataFrame(result_rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
    predictions = pd.concat(prediction_tables, ignore_index=True)
    segment_errors = error_by_segment(predictions)
    inputs = feature_set_inputs(feature_sets)

    tables_dir.mkdir(parents=True, exist_ok=True)
    outputs = {
        "validation_model_results_v1": results,
        "validation_predictions_v1": predictions,
        "validation_error_by_segment_v1": segment_errors,
        "model_feature_set_inputs_v1": inputs,
    }
    for name, table in outputs.items():
        table.to_csv(tables_dir / f"{name}.csv", index=False)

    write_validation_plots(predictions, results, figures_dir=figures_dir)
    return outputs


In [15]:
selected_feature_sets = ["pre_lineup_baseline", "pre_lineup_player_team_opp", "pre_lineup_compact_40", "pre_lineup_compact_40_plus_scoring"]
selected_models = ["mean_baseline", "rolling_fp_10_baseline", "rolling_fp_30_baseline", "ridge", "hist_gradient_boosting"]
loaded = load_validation_data(modeling_path=PROCESSED_DIR / "modeling_table_last4_regular.csv", split_path=TABLES_DIR / "modeling_rows_with_split_v1.csv", feature_set_path=TABLES_DIR / "feature_set_membership.csv", source_path=PROCESSED_DIR / "player_game_source_last4_regular.csv")
rows, prediction_tables = [], []
for feature_set in selected_feature_sets:
    for model_name in selected_models:
        print(feature_set, model_name)
        row, preds = run_one_experiment(loaded.data, feature_set, loaded.feature_sets[feature_set], model_name)
        rows.append(row)
        prediction_tables.append(preds)
selected_results = pd.DataFrame(rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
selected_predictions = pd.concat(prediction_tables, ignore_index=True)
selected_segments = error_by_segment(selected_predictions)
selected_results.to_csv(TABLES_DIR / "selected_model_results.csv", index=False)
selected_predictions.to_csv(TABLES_DIR / "selected_model_predictions.csv", index=False)
selected_segments.to_csv(TABLES_DIR / "selected_model_error_by_segment.csv", index=False)
write_validation_plots(selected_predictions, selected_results, figures_dir=FIGURES_DIR)
display(selected_results.head(15))


pre_lineup_baseline mean_baseline


pre_lineup_baseline rolling_fp_10_baseline


pre_lineup_baseline rolling_fp_30_baseline


pre_lineup_baseline ridge


pre_lineup_baseline hist_gradient_boosting


...

pre_lineup_compact_40_plus_scoring rolling_fp_10_baseline


pre_lineup_compact_40_plus_scoring rolling_fp_30_baseline


pre_lineup_compact_40_plus_scoring ridge


pre_lineup_compact_40_plus_scoring hist_gradient_boosting


,feature_set,model,n_features,train_rows,validation_rows,fit_predict_seconds,mae,rmse,r2,mean_error,median_absolute_error
0,pre_lineup_player_team_opp,hist_gradient_boosting,486,76937,26162,77.810,7.731509,9.900762,0.559142,0.163186,6.336169
1,pre_lineup_player_team_opp,ridge,486,76937,26162,37.336,7.747664,9.912286,0.558116,-0.018882,6.387147
2,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,45,76937,26162,4.912,7.781769,9.947805,0.554943,0.045904,6.415698
3,pre_lineup_compact_40,hist_gradient_boosting,40,76937,26162,4.406,7.783238,9.948584,0.554873,0.036283,6.432807
4,pre_lineup_compact_40_plus_scoring,ridge,45,76937,26162,0.863,7.800043,9.954068,0.554382,-0.147658,6.459782
5,pre_lineup_compact_40,ridge,40,76937,26162,0.875,7.801774,9.955278,0.554274,-0.149823,6.454571
6,pre_lineup_baseline,ridge,4,76937,26162,0.147,7.908809,10.093486,0.541812,0.043496,6.573926
7,pre_lineup_baseline,hist_gradient_boosting,4,76937,26162,1.013,7.930103,10.091192,0.542020,-0.002999,6.618360
8,pre_lineup_baseline,rolling_fp_10_baseline,4,76937,26162,0.001,7.985892,10.269158,0.525724,0.083235,6.520000
9,pre_lineup_player_team_opp,rolling_fp_10_baseline,486,76937,26162,0.001,7.985892,10.269158,0.525724,0.083235,6.520000


## 10b. Alternative Modeling Approach And Validity Check

This section integrates the different ideas from the alternative notebook into the project story. It does three things:

1. Records the stronger reported results from that notebook.
2. Separates valid reusable ideas from same-game leakage.
3. Explains why the below-7 MAE results are diagnostic only, while the corrected ideas are rerun in the next section under the clean split and leakage rules.

The key distinction is:

- `current_is_starter`: lineup-aware information that can be valid if known before the game.
- `starter_minutes_roll_10`: shifted historical starter-minutes average, valid because it uses previous games only.
- direct `starter_minutes = current_is_starter * numMinutes`: invalid same-game data because `numMinutes` is the actual minutes played in the game being predicted.


In [16]:

alternative_modeling_review = pd.DataFrame(
    [
        {
            "item": "reported_best_xgboost_mae",
            "value": "6.7061",
            "status": "diagnostic_only",
            "reason": "Reported on an 80/20 chronological row split using direct starter_minutes and other non-final assumptions.",
        },
        {
            "item": "reported_best_extra_trees_mae",
            "value": "6.7576",
            "status": "diagnostic_only",
            "reason": "Strong result, but not comparable to the clean 2021-2023 train / 2024 validation / 2025 test design.",
        },
        {
            "item": "direct_starter_minutes",
            "value": "used",
            "status": "leakage",
            "reason": "starter_minutes = current_is_starter * actual numMinutes, and actual minutes are known only after/during the game.",
        },
        {
            "item": "target_formula",
            "value": "alternative notebook used -1.5 * turnovers in places",
            "status": "corrected",
            "reason": "This project target uses -1.0 * turnovers, matching the agreed fantasy-points formula.",
        },
        {
            "item": "valid_ideas_kept",
            "value": "tree ensembles, boosted models, time decay, rolling windows, momentum features",
            "status": "kept",
            "reason": "These ideas can be evaluated under the clean split and leakage rules.",
        },
    ]
)

leakage_diagnostic_actual_minutes = pd.DataFrame(
    [
        {"model_family": "Alternative XGBoost final", "reported_mae": 6.7061, "valid_for_final_ranking": False, "leakage_feature": "starter_minutes", "explanation": "Uses actual current-game minutes through starter_minutes."},
        {"model_family": "Alternative ExtraTrees final", "reported_mae": 6.7576, "valid_for_final_ranking": False, "leakage_feature": "starter_minutes", "explanation": "Uses actual current-game minutes through starter_minutes."},
        {"model_family": "Alternative 11-feature ExtraTrees", "reported_mae": 6.9955, "valid_for_final_ranking": False, "leakage_feature": "starter_minutes", "explanation": "This explains why the below-7 MAE was possible but not a valid pre-game prediction result."},
    ]
)

alternative_modeling_review.to_csv(TABLES_DIR / "alternative_modeling_review.csv", index=False)
leakage_diagnostic_actual_minutes.to_csv(TABLES_DIR / "leakage_diagnostic_actual_minutes.csv", index=False)

display(alternative_modeling_review)
display(leakage_diagnostic_actual_minutes)


,item,value,status,reason
0,reported_best_xgboost_mae,6.7061,diagnostic_only,Reported on an 80/20 chronological row split u...
1,reported_best_extra_trees_mae,6.7576,diagnostic_only,"Strong result, but not comparable to the clean..."
2,direct_starter_minutes,used,leakage,starter_minutes = current_is_starter * actual ...
3,target_formula,alternative notebook used -1.5 * turnovers in ...,corrected,"This project target uses -1.0 * turnovers, mat..."
4,valid_ideas_kept,"tree ensembles, boosted models, time decay, ro...",kept,These ideas can be evaluated under the clean s...


,model_family,reported_mae,valid_for_final_ranking,leakage_feature,explanation
0,Alternative XGBoost final,6.7061,False,starter_minutes,Uses actual current-game minutes through start...
1,Alternative ExtraTrees final,6.7576,False,starter_minutes,Uses actual current-game minutes through start...
2,Alternative 11-feature ExtraTrees,6.9955,False,starter_minutes,This explains why the below-7 MAE was possible...


## 10c. Controlled Model Comparison With Alternative Modeling Ideas

This section reruns the useful model families from the alternative notebook under this project's clean split and leakage rules. It keeps model sizes controlled: RandomForest and ExtraTrees are capped at 150 trees, while XGBoost, LightGBM, and CatBoost use moderate boosted-tree settings.


In [17]:

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

CONTROLLED_MODEL_FEATURE_SETS = [
    "pre_lineup_starter_history_roll10",
    "lineup_aware_starter_flag_roll10",
    "pre_lineup_compact_40_plus_scoring",
]

CONTROLLED_MODELS = [
    "ridge",
    "lasso",
    "hist_gradient_boosting",
    "xgboost",
    "lightgbm",
    "catboost",
    "extra_trees",
    "random_forest",
]

BASELINE_MODELS = ["mean_baseline", "rolling_fp_10_baseline"]
TIME_DECAY_MODELS = ["ridge", "lasso", "hist_gradient_boosting", "xgboost", "lightgbm", "catboost", "extra_trees", "random_forest"]
TIME_DECAY_FEATURE_SET = "lineup_aware_starter_flag_roll10"


def make_controlled_model(model_name: str):
    # Create the final controlled comparison models with fixed seeds and bounded runtime.
    if model_name == "ridge":
        return Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("scaler", StandardScaler()), ("model", Ridge(alpha=10.0))])
    if model_name == "lasso":
        return Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("scaler", StandardScaler()), ("model", Lasso(alpha=0.01, max_iter=10_000, random_state=MODEL_RANDOM_STATE))])
    if model_name == "hist_gradient_boosting":
        return Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("model", HistGradientBoostingRegressor(learning_rate=0.05, max_iter=160, max_leaf_nodes=31, l2_regularization=0.1, random_state=MODEL_RANDOM_STATE))])
    if model_name == "xgboost":
        return Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("model", xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror", tree_method="hist", random_state=MODEL_RANDOM_STATE, n_jobs=1, verbosity=0))])
    if model_name == "lightgbm":
        return Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("model", lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=MODEL_RANDOM_STATE, n_jobs=1, verbosity=-1))])
    if model_name == "catboost":
        return Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("model", CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, loss_function="RMSE", random_seed=MODEL_RANDOM_STATE, verbose=False, allow_writing_files=False))])
    if model_name == "extra_trees":
        return Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("model", ExtraTreesRegressor(n_estimators=150, max_depth=12, min_samples_leaf=5, n_jobs=1, random_state=MODEL_RANDOM_STATE))])
    if model_name == "random_forest":
        return Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("model", RandomForestRegressor(n_estimators=150, max_depth=12, min_samples_leaf=5, max_features="sqrt", n_jobs=1, random_state=MODEL_RANDOM_STATE))])
    raise ValueError(f"Unknown controlled model: {model_name}")


def time_decay_weights(train_frame: pd.DataFrame, half_life_days: float = 180.0) -> np.ndarray:
    # Create train-only exponential time-decay weights relative to the latest train date.
    max_train_date = pd.to_datetime(train_frame["game_date"], errors="coerce").max()
    days_since = (max_train_date - pd.to_datetime(train_frame["game_date"], errors="coerce")).dt.days.clip(lower=0)
    return np.exp(-days_since / half_life_days).to_numpy()


def fit_predict_controlled_model(model_name, x_train, y_train, x_validation, sample_weight=None):
    # Fit one controlled model and return fitted model plus predictions.
    model = make_controlled_model(model_name)
    fit_kwargs = {}
    if sample_weight is not None:
        fit_kwargs["model__sample_weight"] = sample_weight
    model.fit(x_train, y_train, **fit_kwargs)
    return model, model.predict(x_train), model.predict(x_validation)


def run_controlled_model_experiment(data, feature_sets, feature_set, model_name, use_time_decay=False):
    # Evaluate one controlled model with train and validation overfitting diagnostics.
    train = data[data[SPLIT_COL].eq("train")].copy()
    validation = data[data[SPLIT_COL].eq("validation")].copy()
    if train.empty or validation.empty:
        raise ValueError("Train and validation splits must both be non-empty.")
    if data.loc[data[SPLIT_COL].eq("test")].index.intersection(train.index).any():
        raise ValueError("Test rows leaked into train.")

    y_train = train[TARGET_COL]
    y_validation = validation[TARGET_COL]
    features = feature_sets[feature_set]
    x_train = _validate_feature_frame(train, features)
    x_validation = _validate_feature_frame(validation, features)
    weights = time_decay_weights(train) if use_time_decay else None

    start = perf_counter()
    if model_name in BASELINE_MODELS:
        train_pred = _baseline_predictions(model_name, y_train, train)
        validation_pred = _baseline_predictions(model_name, y_train, validation)
    else:
        _, train_pred, validation_pred = fit_predict_controlled_model(model_name, x_train, y_train, x_validation, sample_weight=weights)
    elapsed = perf_counter() - start

    train_metrics = regression_metrics(y_train, train_pred)
    validation_metrics = regression_metrics(y_validation, validation_pred)
    return {
        "feature_set": feature_set,
        "model": model_name,
        "uses_time_decay": bool(use_time_decay),
        "validity": "lineup-aware" if feature_set.startswith("lineup_aware") else "pre-lineup",
        "n_features": len(features),
        "train_rows": len(train),
        "validation_rows": len(validation),
        "fit_predict_seconds": round(elapsed, 3),
        "train_mae": train_metrics["mae"],
        "validation_mae": validation_metrics["mae"],
        "mae_gap": validation_metrics["mae"] - train_metrics["mae"],
        "train_rmse": train_metrics["rmse"],
        "validation_rmse": validation_metrics["rmse"],
        "rmse_gap": validation_metrics["rmse"] - train_metrics["rmse"],
        "train_r2": train_metrics["r2"],
        "validation_r2": validation_metrics["r2"],
        "r2_gap": train_metrics["r2"] - validation_metrics["r2"],
        "valid_for_final_ranking": True,
    }

if "loaded" not in globals():
    loaded = load_validation_data(
        modeling_path=PROCESSED_DIR / "modeling_table_last4_regular.csv",
        split_path=TABLES_DIR / "modeling_rows_with_split_v1.csv",
        feature_set_path=TABLES_DIR / "feature_set_membership.csv",
        source_path=PROCESSED_DIR / "player_game_source_last4_regular.csv",
    )

valid_model_rows = []
for feature_set in CONTROLLED_MODEL_FEATURE_SETS:
    for model_name in CONTROLLED_MODELS:
        print(f"controlled feature_set={feature_set} model={model_name}")
        valid_model_rows.append(run_controlled_model_experiment(loaded.data, loaded.feature_sets, feature_set, model_name, use_time_decay=False))

for model_name in BASELINE_MODELS:
    print(f"controlled baseline feature_set=pre_lineup_compact_40_plus_scoring model={model_name}")
    valid_model_rows.append(run_controlled_model_experiment(loaded.data, loaded.feature_sets, "pre_lineup_compact_40_plus_scoring", model_name, use_time_decay=False))

valid_model_comparison = pd.DataFrame(valid_model_rows).sort_values(["validation_mae", "validation_rmse"]).reset_index(drop=True)
valid_model_comparison.to_csv(TABLES_DIR / "valid_model_comparison.csv", index=False)

time_decay_rows = []
for model_name in TIME_DECAY_MODELS:
    print(f"time_decay feature_set={TIME_DECAY_FEATURE_SET} model={model_name}")
    time_decay_rows.append(run_controlled_model_experiment(loaded.data, loaded.feature_sets, TIME_DECAY_FEATURE_SET, model_name, use_time_decay=True))

time_decay_model_comparison = pd.DataFrame(time_decay_rows).sort_values(["validation_mae", "validation_rmse"]).reset_index(drop=True)
time_decay_model_comparison.to_csv(TABLES_DIR / "time_decay_model_comparison.csv", index=False)

combined_controlled_model_comparison = pd.concat(
    [
        valid_model_comparison.assign(comparison_source="standard"),
        time_decay_model_comparison.assign(comparison_source="time_decay"),
    ],
    ignore_index=True,
).sort_values(["validation_mae", "validation_rmse"]).reset_index(drop=True)
combined_controlled_model_comparison.to_csv(TABLES_DIR / "controlled_model_comparison.csv", index=False)

plt.figure(figsize=(10, 6))
plot_data = combined_controlled_model_comparison.head(15).copy()
plot_data["label"] = plot_data["model"] + " / " + plot_data["feature_set"].str.replace("_", " ").str[:28]
plt.barh(plot_data["label"], plot_data["validation_mae"])
plt.gca().invert_yaxis()
plt.xlabel("Validation MAE")
plt.title("Top Controlled Valid Model Comparisons")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "valid_model_comparison_top_mae.png", dpi=160)
plt.close()

display(valid_model_comparison.head(12))
display(time_decay_model_comparison.head(8))


controlled feature_set=pre_lineup_starter_history_roll10 model=ridge


controlled feature_set=pre_lineup_starter_history_roll10 model=lasso


controlled feature_set=pre_lineup_starter_history_roll10 model=hist_gradient_boosting


controlled feature_set=pre_lineup_starter_history_roll10 model=xgboost


controlled feature_set=pre_lineup_starter_history_roll10 model=lightgbm


...

time_decay feature_set=lineup_aware_starter_flag_roll10 model=catboost


time_decay feature_set=lineup_aware_starter_flag_roll10 model=extra_trees


time_decay feature_set=lineup_aware_starter_flag_roll10 model=random_forest


,feature_set,model,uses_time_decay,validity,n_features,train_rows,validation_rows,fit_predict_seconds,train_mae,validation_mae,mae_gap,train_rmse,validation_rmse,rmse_gap,train_r2,validation_r2,r2_gap,valid_for_final_ranking
0,lineup_aware_starter_flag_roll10,catboost,False,lineup-aware,11,76937,26162,3.666,7.614005,7.643728,0.029723,9.720225,9.779999,0.059775,0.572519,0.569831,0.002687,True
1,lineup_aware_starter_flag_roll10,hist_gradient_boosting,False,lineup-aware,11,76937,26162,2.547,7.621970,7.655891,0.033922,9.724880,9.782516,0.057636,0.572109,0.569610,0.002499,True
2,lineup_aware_starter_flag_roll10,lightgbm,False,lineup-aware,11,76937,26162,3.130,7.445526,7.667858,0.222332,9.501938,9.811787,0.309850,0.591503,0.567030,0.024473,True
3,lineup_aware_starter_flag_roll10,xgboost,False,lineup-aware,11,76937,26162,3.850,7.331904,7.678525,0.346621,9.343641,9.831919,0.488278,0.605000,0.565252,0.039748,True
4,lineup_aware_starter_flag_roll10,random_forest,False,lineup-aware,11,76937,26162,12.895,7.277802,7.691886,0.414083,9.285506,9.807239,0.521732,0.609900,0.567432,0.042468,True
5,lineup_aware_starter_flag_roll10,extra_trees,False,lineup-aware,11,76937,26162,13.311,7.419121,7.693669,0.274548,9.460947,9.804209,0.343262,0.595020,0.567699,0.027321,True
6,lineup_aware_starter_flag_roll10,ridge,False,lineup-aware,11,76937,26162,0.131,7.794193,7.721617,-0.072576,9.961254,9.865872,-0.095382,0.551056,0.562244,-0.011188,True
7,lineup_aware_starter_flag_roll10,lasso,False,lineup-aware,11,76937,26162,0.268,7.795094,7.723638,-0.071457,9.961378,9.866198,-0.095181,0.551044,0.562215,-0.011171,True
8,pre_lineup_compact_40_plus_scoring,catboost,False,pre-lineup,45,76937,26162,5.377,7.535847,7.768785,0.232938,9.605539,9.931402,0.325863,0.582547,0.556409,0.026137,True
9,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,False,pre-lineup,45,76937,26162,6.073,7.490265,7.781769,0.291505,9.542915,9.947805,0.404890,0.587972,0.554943,0.033029,True


,feature_set,model,uses_time_decay,validity,n_features,train_rows,validation_rows,fit_predict_seconds,train_mae,validation_mae,mae_gap,train_rmse,validation_rmse,rmse_gap,train_r2,validation_r2,r2_gap,valid_for_final_ranking
0,lineup_aware_starter_flag_roll10,catboost,True,lineup-aware,11,76937,26162,3.174,7.621384,7.660687,0.039303,9.767212,9.794362,0.027150,0.568376,0.568567,-0.000191,True
1,lineup_aware_starter_flag_roll10,hist_gradient_boosting,True,lineup-aware,11,76937,26162,2.131,7.663640,7.661765,-0.001874,9.825421,9.785613,-0.039809,0.563216,0.569337,-0.006122,True
2,lineup_aware_starter_flag_roll10,extra_trees,True,lineup-aware,11,76937,26162,12.342,7.432498,7.672109,0.239611,9.538866,9.800190,0.261324,0.588322,0.568053,0.020268,True
3,lineup_aware_starter_flag_roll10,random_forest,True,lineup-aware,11,76937,26162,12.985,7.332834,7.679750,0.346916,9.423464,9.808967,0.385503,0.598222,0.567279,0.030943,True
4,lineup_aware_starter_flag_roll10,lightgbm,True,lineup-aware,11,76937,26162,2.938,7.479083,7.703348,0.224265,9.598127,9.864383,0.266255,0.583191,0.562376,0.020814,True
5,lineup_aware_starter_flag_roll10,lasso,True,lineup-aware,11,76937,26162,0.286,7.807407,7.703949,-0.103458,10.003008,9.848097,-0.154912,0.547284,0.563820,-0.016536,True
6,lineup_aware_starter_flag_roll10,ridge,True,lineup-aware,11,76937,26162,0.148,7.809060,7.704211,-0.104849,10.004723,9.847907,-0.156815,0.547129,0.563837,-0.016708,True
7,lineup_aware_starter_flag_roll10,xgboost,True,lineup-aware,11,76937,26162,4.196,7.396362,7.729025,0.332663,9.488690,9.884715,0.396025,0.592641,0.560570,0.032071,True


## 11. Starter-Focused Model Comparison

Starter uncertainty is a main basketball modeling problem. This section compares valid shifted starter history against a lineup-aware starter flag model. Direct actual current-game minutes are excluded from model comparisons.


In [18]:
starter_feature_sets = [
    "pre_lineup_starter_history_roll10",
    "lineup_aware_starter_flag_roll10",
]
starter_models = ["ridge", "hist_gradient_boosting"]

loaded = load_validation_data(
    modeling_path=PROCESSED_DIR / "modeling_table_last4_regular.csv",
    split_path=TABLES_DIR / "modeling_rows_with_split_v1.csv",
    feature_set_path=TABLES_DIR / "feature_set_membership.csv",
    source_path=PROCESSED_DIR / "player_game_source_last4_regular.csv",
)

validity_lookup = pd.read_csv(TABLES_DIR / "starter_feature_set_validity.csv").set_index("feature_set")["validity"].to_dict()

rows, prediction_tables = [], []
for feature_set in starter_feature_sets:
    if feature_set not in loaded.feature_sets:
        continue
    for model_name in starter_models:
        print(feature_set, model_name)
        row, preds = run_one_experiment(loaded.data, feature_set, loaded.feature_sets[feature_set], model_name)
        row["validity"] = validity_lookup.get(feature_set, "unknown")
        preds["validity"] = row["validity"]
        rows.append(row)
        prediction_tables.append(preds)

starter_results = pd.DataFrame(rows).sort_values(["validity", "mae", "rmse"]).reset_index(drop=True)
starter_predictions = pd.concat(prediction_tables, ignore_index=True)
starter_segments = error_by_segment(starter_predictions)
starter_results.to_csv(TABLES_DIR / "starter_model_results.csv", index=False)
starter_predictions.to_csv(TABLES_DIR / "starter_model_predictions.csv", index=False)
starter_segments.to_csv(TABLES_DIR / "starter_model_error_by_segment.csv", index=False)

display(starter_results)


pre_lineup_starter_history_roll10 ridge


pre_lineup_starter_history_roll10 hist_gradient_boosting


lineup_aware_starter_flag_roll10 ridge


lineup_aware_starter_flag_roll10 hist_gradient_boosting


,feature_set,model,n_features,train_rows,validation_rows,fit_predict_seconds,mae,rmse,r2,mean_error,median_absolute_error,validity
0,lineup_aware_starter_flag_roll10,hist_gradient_boosting,11,76937,26162,1.534,7.655891,9.782516,0.569610,0.145533,6.297061,lineup-aware
1,lineup_aware_starter_flag_roll10,ridge,11,76937,26162,0.142,7.721617,9.865872,0.562244,-0.085598,6.378066,lineup-aware
2,pre_lineup_starter_history_roll10,hist_gradient_boosting,10,76937,26162,1.625,7.909683,10.087826,0.542326,0.101576,6.558810,pre-lineup
3,pre_lineup_starter_history_roll10,ridge,10,76937,26162,0.214,7.952468,10.150775,0.536596,-0.079548,6.600303,pre-lineup


## 12. Expanding Time Cross-Validation

Random cross-validation is not appropriate for this project. Expanding time CV trains on earlier time windows and validates on later windows.


In [19]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def make_expanding_time_cv_folds(
    data: pd.DataFrame,
    date_col: str = "game_date",
    n_splits: int = 4,
    min_train_fraction: float = 0.40,
) -> pd.DataFrame:
    # Create expanding chronological CV folds using contiguous validation date blocks.
    if not 0 < min_train_fraction < 1:
        raise ValueError("min_train_fraction must be between 0 and 1.")

    frame = data.copy()
    frame[date_col] = pd.to_datetime(frame[date_col], errors="coerce")
    frame = frame.dropna(subset=[date_col]).sort_values(date_col)
    unique_dates = pd.Series(frame[date_col].drop_duplicates().sort_values().to_numpy())
    if len(unique_dates) < n_splits + 2:
        raise ValueError("Not enough unique dates to create expanding time CV folds.")

    first_validation_idx = max(1, int(len(unique_dates) * min_train_fraction))
    validation_dates = unique_dates.iloc[first_validation_idx:]
    validation_blocks = [block for block in np.array_split(validation_dates.to_numpy(), n_splits) if len(block)]

    rows = []
    for fold, block in enumerate(validation_blocks, start=1):
        validation_start = pd.Timestamp(block[0])
        validation_end = pd.Timestamp(block[-1])
        train_mask = frame[date_col] < validation_start
        validation_mask = frame[date_col].between(validation_start, validation_end, inclusive="both")
        train_rows = int(train_mask.sum())
        validation_rows = int(validation_mask.sum())
        if train_rows == 0 or validation_rows == 0:
            continue
        rows.append(
            {
                "fold": fold,
                "train_start": frame.loc[train_mask, date_col].min(),
                "train_end": frame.loc[train_mask, date_col].max(),
                "validation_start": validation_start,
                "validation_end": validation_end,
                "train_rows": train_rows,
                "validation_rows": validation_rows,
                "train_unique_games": int(frame.loc[train_mask, "gameId"].nunique()) if "gameId" in frame else np.nan,
                "validation_unique_games": int(frame.loc[validation_mask, "gameId"].nunique()) if "gameId" in frame else np.nan,
            }
        )
    return pd.DataFrame(rows)


def run_one_time_cv_experiment(
    data: pd.DataFrame,
    fold: pd.Series,
    feature_set: str,
    features: list[str],
    model_name: str,
    date_col: str = "game_date",
) -> dict[str, object]:
    # Fit one model on one expanding CV fold and return validation metrics.
    frame = data.copy()
    frame[date_col] = pd.to_datetime(frame[date_col], errors="coerce")
    train = frame[(frame[date_col] >= fold["train_start"]) & (frame[date_col] <= fold["train_end"])].copy()
    validation = frame[(frame[date_col] >= fold["validation_start"]) & (frame[date_col] <= fold["validation_end"])].copy()
    if train.empty or validation.empty:
        raise ValueError(f"Fold {fold['fold']} produced an empty train or validation block.")

    y_train = train[TARGET_COL]
    y_validation = validation[TARGET_COL]
    start = perf_counter()

    if model_name.endswith("_baseline"):
        predictions = _baseline_predictions(model_name, y_train, validation)
    else:
        model = make_model(model_name)
        x_train = _validate_feature_frame(train, features)
        x_validation = _validate_feature_frame(validation, features)
        model.fit(x_train, y_train)
        predictions = model.predict(x_validation)

    elapsed = perf_counter() - start
    metrics = regression_metrics(y_validation, predictions)
    errors = y_validation.to_numpy() - predictions
    return {
        "fold": int(fold["fold"]),
        "feature_set": feature_set,
        "model": model_name,
        "n_features": len(features),
        "train_start": fold["train_start"],
        "train_end": fold["train_end"],
        "validation_start": fold["validation_start"],
        "validation_end": fold["validation_end"],
        "train_rows": len(train),
        "validation_rows": len(validation),
        "fit_predict_seconds": round(elapsed, 3),
        "mae": metrics["mae"],
        "rmse": metrics["rmse"],
        "r2": metrics["r2"],
        "mean_error": float(np.mean(errors)),
        "median_absolute_error": float(np.median(np.abs(errors))),
    }


def summarize_time_cv_results(results: pd.DataFrame) -> pd.DataFrame:
    # Summarize fold-level CV metrics by feature set and model.
    grouped = results.groupby(["feature_set", "model", "n_features"], as_index=False)
    summary = grouped.agg(
        folds=("fold", "nunique"),
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        min_mae=("mae", "min"),
        max_mae=("mae", "max"),
        mean_rmse=("rmse", "mean"),
        mean_r2=("r2", "mean"),
        total_fit_predict_seconds=("fit_predict_seconds", "sum"),
    )
    summary["std_mae"] = summary["std_mae"].fillna(0.0)
    summary["mae_range"] = summary["max_mae"] - summary["min_mae"]
    return summary.sort_values(["mean_mae", "std_mae", "mean_rmse"]).reset_index(drop=True)


def write_time_cv_plot(results: pd.DataFrame, figures_dir: Path = FIGURES_DIR) -> Path:
    # Save a fold-by-fold MAE plot for the time CV experiment.
    figures_dir.mkdir(parents=True, exist_ok=True)
    plot_path = figures_dir / "time_cv_mae_by_fold.png"
    plt.figure(figsize=(11, 6))
    for (feature_set, model_name), group in results.groupby(["feature_set", "model"]):
        ordered = group.sort_values("fold")
        label = f"{feature_set} | {model_name}"
        plt.plot(ordered["fold"], ordered["mae"], marker="o", linewidth=1.5, label=label)
    plt.xlabel("Expanding time CV fold")
    plt.ylabel("Validation MAE")
    plt.title("Expanding Time Cross-Validation MAE By Fold")
    plt.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150)
    plt.close()
    return plot_path


def write_time_cv_step_summary(path: Path, fold_plan: pd.DataFrame, summary: pd.DataFrame) -> None:
    # Write a short Markdown explanation of the time CV setup and top results.
    best = summary.iloc[0]
    content = f"""# Step 15b: Expanding Time Cross-Validation

## Goal

This section checks whether selected pre-game models are stable across multiple earlier chronological validation blocks.

It uses only rows from the existing `train` split. The official validation split and the held-out 2025 test split are not used here.

## Fold Design

- Number of folds: {fold_plan['fold'].nunique()}
- First train date: {fold_plan['train_start'].min()}
- Last CV validation date: {fold_plan['validation_end'].max()}
- Fold type: expanding train window with contiguous future validation blocks

## Best Mean CV Result

```text
feature_set = {best['feature_set']}
model       = {best['model']}
features    = {int(best['n_features'])}
mean MAE    = {best['mean_mae']:.4f}
std MAE     = {best['std_mae']:.4f}
mean RMSE   = {best['mean_rmse']:.4f}
mean R2     = {best['mean_r2']:.4f}
```

## Interpretation Rule

This result is supporting evidence only. The final model recommendation should still be judged against the official chronological validation split unless the project explicitly changes its model-selection protocol.
"""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


In [20]:
CV_N_SPLITS = 4
CV_MIN_TRAIN_FRACTION = 0.40
CV_FEATURE_SETS = [
    "pre_lineup_baseline",
    "pre_lineup_compact_40",
    "pre_lineup_compact_40_plus_scoring",
    "pre_lineup_player_team_opp",
]
CV_MODELS = [
    "rolling_fp_10_baseline",
    "ridge",
    "hist_gradient_boosting",
]

if "loaded" not in globals():
    loaded = load_validation_data(
        modeling_path=PROCESSED_DIR / "modeling_table_last4_regular.csv",
        split_path=TABLES_DIR / "modeling_rows_with_split_v1.csv",
        feature_set_path=TABLES_DIR / "feature_set_membership.csv",
        source_path=PROCESSED_DIR / "player_game_source_last4_regular.csv",
    )

cv_data = loaded.data[loaded.data[SPLIT_COL].eq("train")].copy()
cv_fold_plan = make_expanding_time_cv_folds(
    cv_data,
    date_col="game_date",
    n_splits=CV_N_SPLITS,
    min_train_fraction=CV_MIN_TRAIN_FRACTION,
)

cv_rows = []
for _, fold in cv_fold_plan.iterrows():
    for feature_set in CV_FEATURE_SETS:
        features = loaded.feature_sets[feature_set]
        for model_name in CV_MODELS:
            print(f"fold={int(fold['fold'])} feature_set={feature_set} model={model_name}")
            cv_rows.append(run_one_time_cv_experiment(cv_data, fold, feature_set, features, model_name))

time_cv_results = pd.DataFrame(cv_rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
time_cv_summary = summarize_time_cv_results(time_cv_results)
time_cv_plot_path = write_time_cv_plot(time_cv_results, figures_dir=FIGURES_DIR)

cv_fold_plan.to_csv(TABLES_DIR / "time_cv_fold_plan.csv", index=False)
time_cv_results.to_csv(TABLES_DIR / "time_cv_results.csv", index=False)
time_cv_summary.to_csv(TABLES_DIR / "time_cv_summary.csv", index=False)
write_time_cv_step_summary(
    STEP_SUMMARIES_DIR / "step_15b_time_cross_validation.md",
    fold_plan=cv_fold_plan,
    summary=time_cv_summary,
)

print(f"time CV plot: {time_cv_plot_path}")
display(cv_fold_plan)
display(time_cv_summary.head(12))


fold=1 feature_set=pre_lineup_baseline model=rolling_fp_10_baseline


fold=1 feature_set=pre_lineup_baseline model=ridge


fold=1 feature_set=pre_lineup_baseline model=hist_gradient_boosting


fold=1 feature_set=pre_lineup_compact_40 model=rolling_fp_10_baseline


fold=1 feature_set=pre_lineup_compact_40 model=ridge


...

fold=4 feature_set=pre_lineup_player_team_opp model=ridge


fold=4 feature_set=pre_lineup_player_team_opp model=hist_gradient_boosting


time CV plot: C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\runs\final\reports\figures\time_cv_mae_by_fold.png


,fold,train_start,train_end,validation_start,validation_end,train_rows,validation_rows,train_unique_games,validation_unique_games
0,1,2021-10-19 19:30:00,2022-11-13 21:00:00,2022-11-13 21:30:00,2023-01-26 19:30:00,30304,11241,1427,535
1,2,2021-10-19 19:30:00,2023-01-26 19:30:00,2023-01-26 20:00:00,2023-10-30 19:30:00,41545,11375,1962,543
2,3,2021-10-19 19:30:00,2023-10-30 19:30:00,2023-10-30 20:00:00,2024-01-25 19:00:00,52920,11760,2505,545
3,4,2021-10-19 19:30:00,2024-01-25 19:00:00,2024-01-25 19:30:00,2024-04-14 15:30:00,64680,12257,3050,574


,feature_set,model,n_features,folds,mean_mae,std_mae,min_mae,max_mae,mean_rmse,mean_r2,total_fit_predict_seconds,mae_range
0,pre_lineup_player_team_opp,hist_gradient_boosting,486,4,7.706301,0.089609,7.601743,7.820166,9.865728,0.571595,135.959,0.218422
1,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,45,4,7.754679,0.081225,7.659346,7.857985,9.912696,0.567477,8.726,0.198639
2,pre_lineup_compact_40,hist_gradient_boosting,40,4,7.754969,0.084993,7.653962,7.861969,9.918676,0.566944,8.762,0.208007
3,pre_lineup_compact_40,ridge,40,4,7.777511,0.092294,7.672422,7.896836,9.926930,0.566253,1.664,0.224414
4,pre_lineup_compact_40_plus_scoring,ridge,45,4,7.777714,0.092767,7.673603,7.898837,9.926582,0.566281,2.037,0.225234
5,pre_lineup_baseline,ridge,4,4,7.874359,0.119996,7.735110,8.028298,10.034975,0.556730,0.155,0.293188
6,pre_lineup_baseline,hist_gradient_boosting,4,4,7.902684,0.110467,7.766217,8.035916,10.048229,0.555603,1.873,0.269698
7,pre_lineup_baseline,rolling_fp_10_baseline,4,4,7.930304,0.167125,7.739094,8.146255,10.204214,0.541567,0.002,0.407161
8,pre_lineup_compact_40,rolling_fp_10_baseline,40,4,7.930304,0.167125,7.739094,8.146255,10.204214,0.541567,0.001,0.407161
9,pre_lineup_compact_40_plus_scoring,rolling_fp_10_baseline,45,4,7.930304,0.167125,7.739094,8.146255,10.204214,0.541567,0.001,0.407161


## 13. PCA And Dimensionality Reduction

PCA tests whether many legal unused broad features can help after compression. Ranking, missingness filtering, imputation, scaling, and PCA fitting use train rows only.


In [21]:
from sklearn.base import clone
from sklearn.decomposition import PCA


PCA_SOURCE_FEATURE_SET = "pre_lineup_player_team_opp"
PCA_BASE_FEATURE_SET = "pre_lineup_compact_40_plus_scoring"
PCA_TOP_N_CANDIDATES = 100
PCA_MISSINGNESS_MAX = 0.40
PCA_COMPONENT_GRID = [10, 20, 30]


def _numeric_feature_columns(data: pd.DataFrame, features: list[str]) -> list[str]:
    # Return feature columns that can be converted to numeric values.
    usable = []
    for feature in features:
        if feature not in data.columns:
            continue
        values = pd.to_numeric(data[feature], errors="coerce")
        if values.notna().sum() > 0:
            usable.append(feature)
    return usable


def rank_pca_candidates(data: pd.DataFrame, candidate_features: list[str], target_col: str = TARGET_COL) -> pd.DataFrame:
    # Rank candidate features by train-only absolute correlation with the target.
    train = data[data[SPLIT_COL].eq("train")].copy()
    rows = []
    y = pd.to_numeric(train[target_col], errors="coerce")
    for feature in candidate_features:
        x = pd.to_numeric(train[feature], errors="coerce")
        missing_rate = float(x.isna().mean())
        valid = x.notna() & y.notna()
        corr = np.nan
        if valid.sum() >= 25 and x[valid].nunique(dropna=True) > 1:
            corr = float(x[valid].corr(y[valid]))
        rows.append(
            {
                "feature": feature,
                "train_missing_rate": missing_rate,
                "train_abs_target_corr": abs(corr) if pd.notna(corr) else np.nan,
                "train_target_corr": corr,
                "train_non_missing_rows": int(x.notna().sum()),
            }
        )
    ranking = pd.DataFrame(rows)
    ranking = ranking[ranking["train_missing_rate"].le(PCA_MISSINGNESS_MAX)]
    return ranking.sort_values(["train_abs_target_corr", "train_non_missing_rows"], ascending=[False, False]).reset_index(drop=True)


def top_interfeature_correlations(train: pd.DataFrame, features: list[str], top_n: int = 50) -> pd.DataFrame:
    # Report the highest absolute train-only correlations among PCA source features.
    if len(features) < 2:
        return pd.DataFrame(columns=["feature_a", "feature_b", "correlation", "abs_correlation"])
    matrix = train[features].apply(pd.to_numeric, errors="coerce").corr()
    rows = []
    for i, feature_a in enumerate(features):
        for feature_b in features[i + 1 :]:
            corr = matrix.loc[feature_a, feature_b]
            if pd.notna(corr):
                rows.append({"feature_a": feature_a, "feature_b": feature_b, "correlation": float(corr), "abs_correlation": float(abs(corr))})
    return pd.DataFrame(rows).sort_values("abs_correlation", ascending=False).head(top_n).reset_index(drop=True)


def train_validation_metrics(model, train_x, train_y, validation_x, validation_y) -> dict[str, float]:
    # Fit a model and return train/validation diagnostics for overfitting checks.
    fitted = clone(model)
    fitted.fit(train_x, train_y)
    train_pred = fitted.predict(train_x)
    validation_pred = fitted.predict(validation_x)
    train_metrics = regression_metrics(train_y, train_pred)
    validation_metrics = regression_metrics(validation_y, validation_pred)
    return {
        "train_mae": train_metrics["mae"],
        "validation_mae": validation_metrics["mae"],
        "mae_gap": validation_metrics["mae"] - train_metrics["mae"],
        "train_rmse": train_metrics["rmse"],
        "validation_rmse": validation_metrics["rmse"],
        "rmse_gap": validation_metrics["rmse"] - train_metrics["rmse"],
        "train_r2": train_metrics["r2"],
        "validation_r2": validation_metrics["r2"],
        "r2_gap": train_metrics["r2"] - validation_metrics["r2"],
    }


loaded = load_validation_data(
    modeling_path=PROCESSED_DIR / "modeling_table_last4_regular.csv",
    split_path=TABLES_DIR / "modeling_rows_with_split_v1.csv",
    feature_set_path=TABLES_DIR / "feature_set_membership.csv",
    source_path=PROCESSED_DIR / "player_game_source_last4_regular.csv",
)
data = loaded.data
train = data[data[SPLIT_COL].eq("train")].copy()
validation = data[data[SPLIT_COL].eq("validation")].copy()

broad_features = _numeric_feature_columns(data, loaded.feature_sets[PCA_SOURCE_FEATURE_SET])
base_features = _numeric_feature_columns(data, loaded.feature_sets[PCA_BASE_FEATURE_SET])
candidate_features = [feature for feature in broad_features if feature not in set(base_features)]

candidate_ranking = rank_pca_candidates(data, candidate_features)
pca_source_features = candidate_ranking.head(PCA_TOP_N_CANDIDATES)["feature"].tolist()
candidate_ranking.to_csv(TABLES_DIR / "pca_candidate_feature_ranking.csv", index=False)

interfeature_corrs = top_interfeature_correlations(train, pca_source_features)
interfeature_corrs.to_csv(TABLES_DIR / "pca_candidate_interfeature_correlations.csv", index=False)

pca_train_raw = train[pca_source_features].apply(pd.to_numeric, errors="coerce")
pca_validation_raw = validation[pca_source_features].apply(pd.to_numeric, errors="coerce")
train_medians = pca_train_raw.median()
pca_train_imputed = pca_train_raw.fillna(train_medians)
pca_validation_imputed = pca_validation_raw.fillna(train_medians)

train_means = pca_train_imputed.mean()
train_stds = pca_train_imputed.std(ddof=0).replace(0, 1)
pca_train_scaled = (pca_train_imputed - train_means) / train_stds
pca_validation_scaled = (pca_validation_imputed - train_means) / train_stds

full_pca = PCA(random_state=MODEL_RANDOM_STATE)
full_pca.fit(pca_train_scaled)
explained = pd.DataFrame(
    {
        "component": np.arange(1, len(full_pca.explained_variance_ratio_) + 1),
        "explained_variance_ratio": full_pca.explained_variance_ratio_,
        "cumulative_explained_variance_ratio": np.cumsum(full_pca.explained_variance_ratio_),
    }
)
n_95 = int(explained.loc[explained["cumulative_explained_variance_ratio"].ge(0.95), "component"].iloc[0])
component_grid = sorted({component for component in PCA_COMPONENT_GRID + [n_95] if component <= len(pca_source_features)})

component_plan = explained[explained["component"].isin(component_grid)].copy()
component_plan["plan_reason"] = np.where(component_plan["component"].eq(n_95), "95% variance target", "fixed grid")
component_plan.to_csv(TABLES_DIR / "pca_component_plan.csv", index=False)

loading_rows = []
for component_index in range(min(max(component_grid), full_pca.components_.shape[0])):
    loadings = pd.Series(full_pca.components_[component_index], index=pca_source_features)
    for rank, (feature, loading) in enumerate(loadings.abs().sort_values(ascending=False).head(10).items(), start=1):
        loading_rows.append(
            {
                "component": component_index + 1,
                "rank": rank,
                "feature": feature,
                "loading": float(loadings.loc[feature]),
                "abs_loading": float(loading),
            }
        )
pca_loadings_top = pd.DataFrame(loading_rows)
pca_loadings_top.to_csv(TABLES_DIR / "pca_component_loadings_top.csv", index=False)

plt.figure(figsize=(8, 5))
plt.plot(explained["component"], explained["cumulative_explained_variance_ratio"], marker="o", markersize=2)
plt.axhline(0.95, color="red", linestyle="--", linewidth=1)
plt.xlabel("PCA components")
plt.ylabel("Cumulative explained variance ratio")
plt.title("PCA Explained Variance From Train-Only Broad Unused Features")
_savefig(FIGURES_DIR / "pca_explained_variance.png")

train_base = train[base_features].apply(pd.to_numeric, errors="coerce")
validation_base = validation[base_features].apply(pd.to_numeric, errors="coerce")
train_y = train[TARGET_COL]
validation_y = validation[TARGET_COL]

model_specs = {
    "ridge": make_model("ridge"),
    "hist_gradient_boosting": make_model("hist_gradient_boosting"),
}

result_rows = []
reference_specs = {
    "compact_scoring_hgb_reference": ("hist_gradient_boosting", base_features),
    "broad_hgb_reference": ("hist_gradient_boosting", broad_features),
}
for label, (model_name, features) in reference_specs.items():
    train_x = train[features].apply(pd.to_numeric, errors="coerce")
    validation_x = validation[features].apply(pd.to_numeric, errors="coerce")
    metrics = train_validation_metrics(make_model(model_name), train_x, train_y, validation_x, validation_y)
    result_rows.append(
        {
            "experiment": label,
            "model": model_name,
            "feature_set": "reference",
            "n_features": len(features),
            "n_pca_source_features": 0,
            "n_pca_components": 0,
            "pca_explained_variance_ratio_sum": np.nan,
            **metrics,
        }
    )

for n_components in component_grid:
    pca = PCA(n_components=n_components, random_state=MODEL_RANDOM_STATE)
    train_components = pca.fit_transform(pca_train_scaled)
    validation_components = pca.transform(pca_validation_scaled)
    component_cols = [f"pca_unused_broad_{i:02d}" for i in range(1, n_components + 1)]
    train_pca_df = pd.DataFrame(train_components, index=train.index, columns=component_cols)
    validation_pca_df = pd.DataFrame(validation_components, index=validation.index, columns=component_cols)
    train_x = pd.concat([train_base.reset_index(drop=True), train_pca_df.reset_index(drop=True)], axis=1)
    validation_x = pd.concat([validation_base.reset_index(drop=True), validation_pca_df.reset_index(drop=True)], axis=1)

    for model_name, model in model_specs.items():
        metrics = train_validation_metrics(model, train_x, train_y, validation_x, validation_y)
        result_rows.append(
            {
                "experiment": f"compact_scoring_plus_pca_{n_components}",
                "model": model_name,
                "feature_set": PCA_BASE_FEATURE_SET,
                "n_features": train_x.shape[1],
                "n_pca_source_features": len(pca_source_features),
                "n_pca_components": n_components,
                "pca_explained_variance_ratio_sum": float(pca.explained_variance_ratio_.sum()),
                **metrics,
            }
        )

pca_results = pd.DataFrame(result_rows).sort_values(["validation_mae", "validation_rmse"]).reset_index(drop=True)
pca_results["overfitting_interpretation"] = np.select(
    [
        pca_results["mae_gap"].gt(1.0),
        pca_results["mae_gap"].le(1.0) & pca_results["validation_mae"].gt(pca_results["validation_mae"].min() + 0.05),
    ],
    [
        "Large train advantage over validation = possible overfitting.",
        "Small gap with worse validation MAE = underpowered or unhelpful features.",
    ],
    default="Small gap with improved validation MAE = useful compression.",
)
pca_results.to_csv(TABLES_DIR / "pca_model_results.csv", index=False)
pca_results.to_csv(TABLES_DIR / "pca_overfit_comparison.csv", index=False)

plot_df = pca_results[pca_results["n_pca_components"].gt(0)].copy()
plt.figure(figsize=(8, 5))
for model_name, group in plot_df.groupby("model"):
    plt.plot(group["n_pca_components"], group["validation_mae"], marker="o", label=model_name)
plt.xlabel("PCA components")
plt.ylabel("Validation MAE")
plt.title("Validation MAE By PCA Component Count")
plt.legend()
_savefig(FIGURES_DIR / "pca_validation_mae_by_components.png")

best_pca = pca_results[pca_results["n_pca_components"].gt(0)].iloc[0]
summary = f"""# Step 13 PCA Feature Compression

PCA used `{len(pca_source_features)}` train-selected source features from `{PCA_SOURCE_FEATURE_SET}` after removing `{PCA_BASE_FEATURE_SET}` features and filtering train missingness above `{PCA_MISSINGNESS_MAX:.0%}`.

Best PCA validation MAE: `{best_pca['validation_mae']:.4f}` from `{best_pca['experiment']}` with `{best_pca['model']}`.

All ranking, missingness filtering, imputation, scaling, and PCA fitting used train rows only. Validation rows were transformed only. Test rows were not used.
"""
(STEP_SUMMARIES_DIR / "step_13_pca_feature_compression.md").write_text(summary, encoding="utf-8")

display(candidate_ranking.head(15))
display(component_plan)
display(pca_results)
print(summary)


,feature,train_missing_rate,train_abs_target_corr,train_target_corr,train_non_missing_rows
0,fantasy_points_roll_3,0.010632,0.707585,0.707585,76119
1,numMinutes_roll_3,0.010632,0.643235,0.643235,76119
2,turnovers_roll_30,0.010632,0.629840,0.629840,76119
3,freeThrowsPercentage_roll_30,0.010632,0.586123,0.586123,76119
4,turnovers_roll_5,0.010632,0.580729,0.580729,76119
5,trueShootingPercentage_std_30,0.020965,0.567966,-0.567966,75324
6,assists_roll_30,0.010632,0.563465,0.563465,76119
7,freeThrowsPercentage_roll_10,0.010632,0.557763,0.557763,76119
8,assists_roll_5,0.010632,0.554047,0.554047,76119
9,effectiveFieldGoalPercentage_std_30,0.020965,0.553744,-0.553744,75324


,component,explained_variance_ratio,cumulative_explained_variance_ratio,plan_reason
9,10,0.026450,0.667944,fixed grid
19,20,0.011488,0.832208,fixed grid
29,30,0.005298,0.907999,fixed grid
40,41,0.002930,0.951937,95% variance target


,experiment,model,feature_set,n_features,n_pca_source_features,n_pca_components,pca_explained_variance_ratio_sum,train_mae,validation_mae,mae_gap,train_rmse,validation_rmse,rmse_gap,train_r2,validation_r2,r2_gap,overfitting_interpretation
0,broad_hgb_reference,hist_gradient_boosting,reference,484,0,0,NaN,7.354057,7.731509,0.377452,9.368112,9.900762,0.532649,0.602928,0.559142,0.043786,Small gap with improved validation MAE = usefu...
1,compact_scoring_plus_pca_30,hist_gradient_boosting,pre_lineup_compact_40_plus_scoring,75,100,30,0.907999,7.469244,7.747985,0.278741,9.520424,9.907535,0.387111,0.589912,0.558539,0.031373,Small gap with improved validation MAE = usefu...
2,compact_scoring_plus_pca_41,hist_gradient_boosting,pre_lineup_compact_40_plus_scoring,86,100,41,0.951937,7.452800,7.749076,0.296276,9.499486,9.903476,0.403991,0.591714,0.558901,0.032813,Small gap with improved validation MAE = usefu...
3,compact_scoring_plus_pca_30,ridge,pre_lineup_compact_40_plus_scoring,75,100,30,0.907999,7.712556,7.751899,0.039343,9.858115,9.900598,0.042483,0.560304,0.559157,0.001147,Small gap with improved validation MAE = usefu...
4,compact_scoring_plus_pca_20,ridge,pre_lineup_compact_40_plus_scoring,65,100,20,0.832208,7.715315,7.752298,0.036983,9.861522,9.903931,0.042409,0.560000,0.558860,0.001140,Small gap with improved validation MAE = usefu...
5,compact_scoring_plus_pca_10,ridge,pre_lineup_compact_40_plus_scoring,55,100,10,0.667944,7.723319,7.754614,0.031295,9.871290,9.908288,0.036998,0.559128,0.558472,0.000656,Small gap with improved validation MAE = usefu...
6,compact_scoring_plus_pca_41,ridge,pre_lineup_compact_40_plus_scoring,86,100,41,0.951937,7.709114,7.756352,0.047238,9.854068,9.902714,0.048646,0.560665,0.558969,0.001697,Small gap with improved validation MAE = usefu...
7,compact_scoring_plus_pca_20,hist_gradient_boosting,pre_lineup_compact_40_plus_scoring,65,100,20,0.832208,7.513089,7.762020,0.248932,9.574866,9.922236,0.347370,0.585208,0.557228,0.027980,Small gap with improved validation MAE = usefu...
8,compact_scoring_plus_pca_10,hist_gradient_boosting,pre_lineup_compact_40_plus_scoring,55,100,10,0.667944,7.522907,7.763282,0.240376,9.587419,9.917818,0.330399,0.584120,0.557622,0.026498,Small gap with improved validation MAE = usefu...
9,compact_scoring_hgb_reference,hist_gradient_boosting,reference,45,0,0,NaN,7.490265,7.781769,0.291505,9.542915,9.947805,0.404890,0.587972,0.554943,0.033029,Small gap with worse validation MAE = underpow...


# Step 13 PCA Feature Compression

PCA used `100` train-selected source features from `pre_lineup_player_team_opp` after removing `pre_lineup_compact_40_plus_scoring` features and filtering train missingness above `40%`.

Best PCA validation MAE: `7.7480` from `compact_scoring_plus_pca_30` with `hist_gradient_boosting`.

All ranking, missingness filtering, imputation, scaling, and PCA fitting used train rows only. Validation rows were transformed only. Test rows were not used.



## 13b. Starter PCA Comparison

This section combines valid starter model families with PCA features. It compares the pre-lineup starter-history model and the lineup-aware starter-flag model with no actual current-game minutes.


In [22]:
STARTER_PCA_SOURCE_FEATURE_SET = "pre_lineup_player_team_opp"
STARTER_PCA_BASE_FEATURE_SETS = [
    "pre_lineup_starter_history_roll10",
    "lineup_aware_starter_flag_roll10",
]


def run_starter_pca_comparison(
    data: pd.DataFrame,
    feature_sets: dict[str, list[str]],
    tables_dir: Path = TABLES_DIR,
    figures_dir: Path = FIGURES_DIR,
) -> pd.DataFrame:
    # Compare valid starter feature sets with train-only PCA components.
    train = data[data[SPLIT_COL].eq("train")].copy()
    validation = data[data[SPLIT_COL].eq("validation")].copy()
    if data[data[SPLIT_COL].eq("test")].empty:
        raise ValueError("Expected untouched 2025 test rows in the split report.")

    source_features = _numeric_feature_columns(data, feature_sets[STARTER_PCA_SOURCE_FEATURE_SET])
    train_y = train[TARGET_COL]
    validation_y = validation[TARGET_COL]
    rows = []
    model_specs = {
        "ridge": make_model("ridge"),
        "hist_gradient_boosting": make_model("hist_gradient_boosting"),
    }

    for base_feature_set in STARTER_PCA_BASE_FEATURE_SETS:
        base_features = _numeric_feature_columns(data, feature_sets[base_feature_set])
        candidate_features = [feature for feature in source_features if feature not in set(base_features)]
        ranking = rank_pca_candidates(data, candidate_features)
        pca_features = ranking.head(PCA_TOP_N_CANDIDATES)["feature"].tolist()

        pca_train_raw = train[pca_features].apply(pd.to_numeric, errors="coerce")
        pca_validation_raw = validation[pca_features].apply(pd.to_numeric, errors="coerce")
        train_medians = pca_train_raw.median()
        pca_train_imputed = pca_train_raw.fillna(train_medians)
        pca_validation_imputed = pca_validation_raw.fillna(train_medians)
        train_means = pca_train_imputed.mean()
        train_stds = pca_train_imputed.std(ddof=0).replace(0, 1)
        pca_train_scaled = (pca_train_imputed - train_means) / train_stds
        pca_validation_scaled = (pca_validation_imputed - train_means) / train_stds

        full_pca = PCA(random_state=MODEL_RANDOM_STATE)
        full_pca.fit(pca_train_scaled)
        cumulative = np.cumsum(full_pca.explained_variance_ratio_)
        n_95 = int(np.where(cumulative >= 0.95)[0][0] + 1)
        component_grid = sorted({component for component in PCA_COMPONENT_GRID + [n_95] if component <= len(pca_features)})

        train_base = train[base_features].apply(pd.to_numeric, errors="coerce")
        validation_base = validation[base_features].apply(pd.to_numeric, errors="coerce")
        for model_name, model in model_specs.items():
            metrics = train_validation_metrics(model, train_base, train_y, validation_base, validation_y)
            rows.append(
                {
                    "experiment": f"{base_feature_set}_reference",
                    "base_feature_set": base_feature_set,
                    "model": model_name,
                    "n_features": len(base_features),
                    "n_pca_source_features": 0,
                    "n_pca_components": 0,
                    "pca_explained_variance_ratio_sum": np.nan,
                    **metrics,
                }
            )

        for n_components in component_grid:
            pca = PCA(n_components=n_components, random_state=MODEL_RANDOM_STATE)
            train_components = pca.fit_transform(pca_train_scaled)
            validation_components = pca.transform(pca_validation_scaled)
            component_cols = [f"starter_pca_{i:02d}" for i in range(1, n_components + 1)]
            train_pca_df = pd.DataFrame(train_components, index=train.index, columns=component_cols)
            validation_pca_df = pd.DataFrame(validation_components, index=validation.index, columns=component_cols)
            train_x = pd.concat([train_base.reset_index(drop=True), train_pca_df.reset_index(drop=True)], axis=1)
            validation_x = pd.concat([validation_base.reset_index(drop=True), validation_pca_df.reset_index(drop=True)], axis=1)

            for model_name, model in model_specs.items():
                metrics = train_validation_metrics(model, train_x, train_y, validation_x, validation_y)
                rows.append(
                    {
                        "experiment": f"{base_feature_set}_plus_pca_{n_components}",
                        "base_feature_set": base_feature_set,
                        "model": model_name,
                        "n_features": train_x.shape[1],
                        "n_pca_source_features": len(pca_features),
                        "n_pca_components": n_components,
                        "pca_explained_variance_ratio_sum": float(pca.explained_variance_ratio_.sum()),
                        **metrics,
                    }
                )

    results = pd.DataFrame(rows).sort_values(["validation_mae", "validation_rmse"]).reset_index(drop=True)
    results["overfitting_interpretation"] = np.select(
        [
            results["mae_gap"].gt(1.0),
            results["mae_gap"].le(1.0) & results["validation_mae"].gt(results["validation_mae"].min() + 0.05),
        ],
        [
            "Large train advantage over validation = possible overfitting.",
            "Small gap with worse validation MAE = underpowered or unhelpful features.",
        ],
        default="Small gap with improved validation MAE = useful compression.",
    )
    results.to_csv(tables_dir / "starter_pca_model_results.csv", index=False)
    results.to_csv(tables_dir / "starter_pca_overfit_comparison.csv", index=False)

    plot_df = results[results["n_pca_components"].gt(0)].copy()
    plt.figure(figsize=(10, 5))
    for (base_feature_set, model_name), group in plot_df.groupby(["base_feature_set", "model"]):
        label = f"{base_feature_set} / {model_name}"
        plt.plot(group["n_pca_components"], group["validation_mae"], marker="o", label=label)
    plt.xlabel("PCA components")
    plt.ylabel("Validation MAE")
    plt.title("Starter Model Validation MAE By PCA Component Count")
    plt.legend(fontsize=8)
    _savefig(figures_dir / "starter_pca_validation_mae_by_components.png")

    best = results.iloc[0]
    summary = f"""# Step 13b Starter PCA Comparison

This section compares two valid starter model families with and without train-only PCA features:

- `pre_lineup_starter_history_roll10`
- `lineup_aware_starter_flag_roll10`

Direct current-game `starter_minutes` is not used as a model feature.

Best starter PCA/reference validation MAE: `{best['validation_mae']:.4f}` from `{best['experiment']}` with `{best['model']}`.

PCA ranking, imputation, scaling, and fitting used train rows only. Validation rows were transformed only. Test rows were not used.
"""
    (STEP_SUMMARIES_DIR / "step_13b_starter_pca_comparison.md").write_text(summary, encoding="utf-8")
    return results


starter_pca_results = run_starter_pca_comparison(loaded.data, loaded.feature_sets)
display(starter_pca_results)


,experiment,base_feature_set,model,n_features,n_pca_source_features,n_pca_components,pca_explained_variance_ratio_sum,train_mae,validation_mae,mae_gap,train_rmse,validation_rmse,rmse_gap,train_r2,validation_r2,r2_gap,overfitting_interpretation
0,lineup_aware_starter_flag_roll10_plus_pca_30,lineup_aware_starter_flag_roll10,hist_gradient_boosting,41,100,30,0.927621,7.409026,7.581493,0.172467,9.452620,9.694047,0.241428,0.595732,0.577359,0.018373,Small gap with improved validation MAE = usefu...
1,lineup_aware_starter_flag_roll10_plus_pca_37,lineup_aware_starter_flag_roll10,hist_gradient_boosting,48,100,37,0.952870,7.349468,7.581563,0.232095,9.374061,9.689160,0.315099,0.602424,0.577785,0.024639,Small gap with improved validation MAE = usefu...
2,lineup_aware_starter_flag_roll10_plus_pca_20,lineup_aware_starter_flag_roll10,hist_gradient_boosting,31,100,20,0.860610,7.427530,7.584785,0.157255,9.475666,9.698204,0.222538,0.593759,0.576997,0.016762,Small gap with improved validation MAE = usefu...
3,lineup_aware_starter_flag_roll10_plus_pca_30,lineup_aware_starter_flag_roll10,ridge,41,100,30,0.927621,7.637865,7.589703,-0.048162,9.766861,9.718277,-0.048584,0.568407,0.575244,-0.006837,Small gap with improved validation MAE = usefu...
4,lineup_aware_starter_flag_roll10_plus_pca_37,lineup_aware_starter_flag_roll10,ridge,48,100,37,0.952870,7.631900,7.591357,-0.040543,9.758039,9.712857,-0.045182,0.569186,0.575717,-0.006531,Small gap with improved validation MAE = usefu...
5,lineup_aware_starter_flag_roll10_plus_pca_10,lineup_aware_starter_flag_roll10,hist_gradient_boosting,21,100,10,0.721649,7.435551,7.595783,0.160232,9.489775,9.709210,0.219435,0.592548,0.576036,0.016512,Small gap with improved validation MAE = usefu...
6,lineup_aware_starter_flag_roll10_plus_pca_20,lineup_aware_starter_flag_roll10,ridge,31,100,20,0.860610,7.645401,7.597605,-0.047796,9.776657,9.725851,-0.050806,0.567541,0.574582,-0.007041,Small gap with improved validation MAE = usefu...
7,lineup_aware_starter_flag_roll10_plus_pca_10,lineup_aware_starter_flag_roll10,ridge,21,100,10,0.721649,7.669121,7.615705,-0.053416,9.806693,9.751323,-0.055370,0.564879,0.572350,-0.007471,Small gap with improved validation MAE = usefu...
8,lineup_aware_starter_flag_roll10_reference,lineup_aware_starter_flag_roll10,hist_gradient_boosting,11,0,0,NaN,7.621970,7.655891,0.033922,9.724880,9.782516,0.057636,0.572109,0.569610,0.002499,Small gap with worse validation MAE = underpow...
9,lineup_aware_starter_flag_roll10_reference,lineup_aware_starter_flag_roll10,ridge,11,0,0,NaN,7.794193,7.721617,-0.072576,9.961254,9.865872,-0.095382,0.551056,0.562244,-0.011188,Small gap with worse validation MAE = underpow...


## 13c. Isolation Forest Anomaly Features

This section uses train-only `IsolationForest` models to classify unusual player-game rows from legal pre-game and lineup-aware features. The rows are not removed. The anomaly score and outlier flag are added as candidate reliability/model features, then validation error is checked by anomaly group.


In [23]:
ISOLATION_CONTAMINATION = 0.05
ISOLATION_N_ESTIMATORS = 200
ISOLATION_FEATURE_COLUMNS = ["isolation_anomaly_score", "isolation_is_outlier", "isolation_score_decile"]
ISOLATION_VARIANTS = {
    "pre_lineup_anomaly": "pre_lineup_player_team_opp",
    "lineup_aware_anomaly": "lineup_aware_starter_flag_roll10",
}
ISOLATION_EXCLUDED_FEATURES = {TARGET_COL, "fantasy_points", "starter_minutes", "bench_minutes", "actual_numMinutes", "numMinutes"}


def legal_isolation_features(data: pd.DataFrame, features: list[str]) -> list[str]:
    # Return numeric legal features for anomaly detection, excluding direct current-game columns.
    numeric_features = _numeric_feature_columns(data, features)
    return [feature for feature in numeric_features if feature not in ISOLATION_EXCLUDED_FEATURES]


def assign_deciles_from_train(train_scores: np.ndarray, scores: np.ndarray, n_bins: int = 10) -> np.ndarray:
    # Assign score deciles using train-score thresholds only; higher decile means more anomalous.
    thresholds = np.quantile(train_scores, np.linspace(0.1, 0.9, n_bins - 1))
    return np.searchsorted(thresholds, scores, side="right") + 1


def fit_isolation_variant(data: pd.DataFrame, variant_name: str, source_feature_set: str) -> tuple[pd.DataFrame, dict[str, object]]:
    # Fit Isolation Forest on train rows only and transform train/validation rows.
    if data[data[SPLIT_COL].eq("test")].empty:
        raise ValueError("Expected untouched 2025 test rows in the split report.")
    source_features = legal_isolation_features(data, loaded.feature_sets[source_feature_set])
    if not source_features:
        raise ValueError(f"No legal numeric Isolation Forest features for {variant_name}.")

    train_mask = data[SPLIT_COL].eq("train")
    validation_mask = data[SPLIT_COL].eq("validation")
    fit_mask = train_mask | validation_mask
    train = data.loc[train_mask].copy()
    validation = data.loc[validation_mask].copy()

    train_raw = train[source_features].apply(pd.to_numeric, errors="coerce")
    validation_raw = validation[source_features].apply(pd.to_numeric, errors="coerce")
    train_medians = train_raw.median()
    train_imputed = train_raw.fillna(train_medians)
    validation_imputed = validation_raw.fillna(train_medians)
    train_means = train_imputed.mean()
    train_stds = train_imputed.std(ddof=0).replace(0, 1)
    train_scaled = (train_imputed - train_means) / train_stds
    validation_scaled = (validation_imputed - train_means) / train_stds

    isolation = IsolationForest(
        n_estimators=ISOLATION_N_ESTIMATORS,
        contamination=ISOLATION_CONTAMINATION,
        random_state=MODEL_RANDOM_STATE,
        n_jobs=1,
    )
    isolation.fit(train_scaled)

    train_score = -isolation.decision_function(train_scaled)
    validation_score = -isolation.decision_function(validation_scaled)
    train_outlier = (isolation.predict(train_scaled) == -1).astype(int)
    validation_outlier = (isolation.predict(validation_scaled) == -1).astype(int)
    train_decile = assign_deciles_from_train(train_score, train_score)
    validation_decile = assign_deciles_from_train(train_score, validation_score)

    features = pd.DataFrame(index=data.index, columns=ISOLATION_FEATURE_COLUMNS, dtype=float)
    features.loc[train.index, "isolation_anomaly_score"] = train_score
    features.loc[validation.index, "isolation_anomaly_score"] = validation_score
    features.loc[train.index, "isolation_is_outlier"] = train_outlier
    features.loc[validation.index, "isolation_is_outlier"] = validation_outlier
    features.loc[train.index, "isolation_score_decile"] = train_decile
    features.loc[validation.index, "isolation_score_decile"] = validation_decile

    plan = {
        "anomaly_variant": variant_name,
        "source_feature_set": source_feature_set,
        "n_input_features": len(source_features),
        "contamination": ISOLATION_CONTAMINATION,
        "n_estimators": ISOLATION_N_ESTIMATORS,
        "train_rows_fit": int(train_mask.sum()),
        "validation_rows_transformed": int(validation_mask.sum()),
        "test_rows_used": 0,
        "uses_direct_starter_minutes": "starter_minutes" in source_features,
        "uses_target": TARGET_COL in source_features,
        "input_features": ", ".join(source_features),
    }
    return features, plan


def fit_predict_with_train_validation_metrics(model, train_x, train_y, validation_x, validation_y):
    # Fit a model and return train/validation metrics plus validation predictions.
    fitted = clone(model)
    fitted.fit(train_x, train_y)
    train_pred = fitted.predict(train_x)
    validation_pred = fitted.predict(validation_x)
    train_metrics = regression_metrics(train_y, train_pred)
    validation_metrics = regression_metrics(validation_y, validation_pred)
    metrics = {
        "train_mae": train_metrics["mae"],
        "validation_mae": validation_metrics["mae"],
        "mae_gap": validation_metrics["mae"] - train_metrics["mae"],
        "train_rmse": train_metrics["rmse"],
        "validation_rmse": validation_metrics["rmse"],
        "rmse_gap": validation_metrics["rmse"] - train_metrics["rmse"],
        "train_r2": train_metrics["r2"],
        "validation_r2": validation_metrics["r2"],
        "r2_gap": train_metrics["r2"] - validation_metrics["r2"],
    }
    return metrics, validation_pred


def run_isolation_forest_experiment(data: pd.DataFrame, feature_sets: dict[str, list[str]]) -> dict[str, pd.DataFrame]:
    # Run Isolation Forest feature experiments without using test rows or current-game outcome columns.
    train = data[data[SPLIT_COL].eq("train")].copy()
    validation = data[data[SPLIT_COL].eq("validation")].copy()
    train_y = train[TARGET_COL]
    validation_y = validation[TARGET_COL]

    isolation_features_by_variant = {}
    feature_plan_rows = []
    for variant_name, source_feature_set in ISOLATION_VARIANTS.items():
        features, plan = fit_isolation_variant(data, variant_name, source_feature_set)
        isolation_features_by_variant[variant_name] = features
        feature_plan_rows.append(plan)
    feature_plan = pd.DataFrame(feature_plan_rows)
    feature_plan.to_csv(TABLES_DIR / "isolation_forest_feature_plan.csv", index=False)

    model_specs = {
        "ridge": make_model("ridge"),
        "hist_gradient_boosting": make_model("hist_gradient_boosting"),
    }
    experiment_specs = [
        {"experiment": "pre_lineup_player_team_opp_reference", "base_feature_set": "pre_lineup_player_team_opp", "anomaly_variant": None},
        {"experiment": "pre_lineup_player_team_opp_plus_isolation", "base_feature_set": "pre_lineup_player_team_opp", "anomaly_variant": "pre_lineup_anomaly"},
        {"experiment": "pre_lineup_compact_40_plus_scoring_reference", "base_feature_set": "pre_lineup_compact_40_plus_scoring", "anomaly_variant": None},
        {"experiment": "pre_lineup_compact_40_plus_scoring_plus_isolation", "base_feature_set": "pre_lineup_compact_40_plus_scoring", "anomaly_variant": "pre_lineup_anomaly"},
        {"experiment": "lineup_aware_starter_flag_roll10_reference", "base_feature_set": "lineup_aware_starter_flag_roll10", "anomaly_variant": None},
        {"experiment": "lineup_aware_starter_flag_roll10_plus_isolation", "base_feature_set": "lineup_aware_starter_flag_roll10", "anomaly_variant": "lineup_aware_anomaly"},
    ]

    best_starter_pca_row = starter_pca_results.sort_values(["validation_mae", "validation_rmse"]).iloc[0]
    include_starter_pca = pd.notna(best_starter_pca_row.get("n_pca_components", np.nan)) and int(best_starter_pca_row["n_pca_components"]) > 0

    rows = []
    prediction_tables = []

    for spec in experiment_specs:
        base_features = legal_isolation_features(data, feature_sets[spec["base_feature_set"]])
        train_x = train[base_features].apply(pd.to_numeric, errors="coerce")
        validation_x = validation[base_features].apply(pd.to_numeric, errors="coerce")
        if spec["anomaly_variant"] is not None:
            anomaly_features = isolation_features_by_variant[spec["anomaly_variant"]][ISOLATION_FEATURE_COLUMNS]
            train_x = pd.concat([train_x.reset_index(drop=True), anomaly_features.loc[train.index].reset_index(drop=True)], axis=1)
            validation_x = pd.concat([validation_x.reset_index(drop=True), anomaly_features.loc[validation.index].reset_index(drop=True)], axis=1)

        for model_name, model in model_specs.items():
            metrics, validation_pred = fit_predict_with_train_validation_metrics(model, train_x, train_y, validation_x, validation_y)
            rows.append(
                {
                    "experiment": spec["experiment"],
                    "base_feature_set": spec["base_feature_set"],
                    "anomaly_variant": spec["anomaly_variant"] or "none",
                    "model": model_name,
                    "n_base_features": len(base_features),
                    "n_features": train_x.shape[1],
                    "train_rows": len(train),
                    "validation_rows": len(validation),
                    **metrics,
                }
            )
            pred = validation[["gameId", "personId", "game_date", "season_start", "player_name", "playerteamName", "opponentteamName", "starter_label", "primary_position", TARGET_COL]].copy()
            pred["experiment"] = spec["experiment"]
            pred["base_feature_set"] = spec["base_feature_set"]
            pred["anomaly_variant"] = spec["anomaly_variant"] or "none"
            pred["model"] = model_name
            pred["prediction"] = validation_pred
            pred["error"] = pred[TARGET_COL] - pred["prediction"]
            pred["absolute_error"] = pred["error"].abs()
            if spec["anomaly_variant"] is not None:
                anomaly_features = isolation_features_by_variant[spec["anomaly_variant"]].loc[validation.index, ISOLATION_FEATURE_COLUMNS]
                pred = pd.concat([pred.reset_index(drop=True), anomaly_features.reset_index(drop=True)], axis=1)
            prediction_tables.append(pred)

    if include_starter_pca:
        pca_base_feature_set = str(best_starter_pca_row["base_feature_set"])
        n_components = int(best_starter_pca_row["n_pca_components"])
        anomaly_variant = "lineup_aware_anomaly" if pca_base_feature_set.startswith("lineup_aware") else "pre_lineup_anomaly"
        source_features = legal_isolation_features(data, feature_sets[STARTER_PCA_SOURCE_FEATURE_SET])
        base_features = legal_isolation_features(data, feature_sets[pca_base_feature_set])
        candidate_features = [feature for feature in source_features if feature not in set(base_features)]
        pca_features = rank_pca_candidates(data, candidate_features).head(PCA_TOP_N_CANDIDATES)["feature"].tolist()
        pca_train_raw = train[pca_features].apply(pd.to_numeric, errors="coerce")
        pca_validation_raw = validation[pca_features].apply(pd.to_numeric, errors="coerce")
        train_medians = pca_train_raw.median()
        pca_train_imputed = pca_train_raw.fillna(train_medians)
        pca_validation_imputed = pca_validation_raw.fillna(train_medians)
        train_means = pca_train_imputed.mean()
        train_stds = pca_train_imputed.std(ddof=0).replace(0, 1)
        pca_train_scaled = (pca_train_imputed - train_means) / train_stds
        pca_validation_scaled = (pca_validation_imputed - train_means) / train_stds
        pca = PCA(n_components=n_components, random_state=MODEL_RANDOM_STATE)
        train_components = pca.fit_transform(pca_train_scaled)
        validation_components = pca.transform(pca_validation_scaled)
        component_cols = [f"best_starter_pca_{i:02d}" for i in range(1, n_components + 1)]
        train_pca_df = pd.DataFrame(train_components, columns=component_cols)
        validation_pca_df = pd.DataFrame(validation_components, columns=component_cols)
        train_base = train[base_features].apply(pd.to_numeric, errors="coerce").reset_index(drop=True)
        validation_base = validation[base_features].apply(pd.to_numeric, errors="coerce").reset_index(drop=True)
        anomaly_features = isolation_features_by_variant[anomaly_variant][ISOLATION_FEATURE_COLUMNS]
        train_anomaly = anomaly_features.loc[train.index].reset_index(drop=True)
        validation_anomaly = anomaly_features.loc[validation.index].reset_index(drop=True)
        pca_train_x = pd.concat([train_base, train_pca_df], axis=1)
        pca_validation_x = pd.concat([validation_base, validation_pca_df], axis=1)
        iso_pca_train_x = pd.concat([pca_train_x, train_anomaly], axis=1)
        iso_pca_validation_x = pd.concat([pca_validation_x, validation_anomaly], axis=1)

        for experiment_name, train_x, validation_x, row_anomaly_variant in [
            (f"{pca_base_feature_set}_best_pca_reference", pca_train_x, pca_validation_x, "none"),
            (f"{pca_base_feature_set}_best_pca_plus_isolation", iso_pca_train_x, iso_pca_validation_x, anomaly_variant),
        ]:
            for model_name, model in model_specs.items():
                metrics, validation_pred = fit_predict_with_train_validation_metrics(model, train_x, train_y, validation_x, validation_y)
                rows.append(
                    {
                        "experiment": experiment_name,
                        "base_feature_set": pca_base_feature_set,
                        "anomaly_variant": row_anomaly_variant,
                        "model": model_name,
                        "n_base_features": len(base_features),
                        "n_features": train_x.shape[1],
                        "train_rows": len(train),
                        "validation_rows": len(validation),
                        **metrics,
                    }
                )
                pred = validation[["gameId", "personId", "game_date", "season_start", "player_name", "playerteamName", "opponentteamName", "starter_label", "primary_position", TARGET_COL]].copy()
                pred["experiment"] = experiment_name
                pred["base_feature_set"] = pca_base_feature_set
                pred["anomaly_variant"] = row_anomaly_variant
                pred["model"] = model_name
                pred["prediction"] = validation_pred
                pred["error"] = pred[TARGET_COL] - pred["prediction"]
                pred["absolute_error"] = pred["error"].abs()
                if row_anomaly_variant != "none":
                    pred = pd.concat([pred.reset_index(drop=True), validation_anomaly.reset_index(drop=True)], axis=1)
                prediction_tables.append(pred)

    results = pd.DataFrame(rows).sort_values(["validation_mae", "validation_rmse"]).reset_index(drop=True)
    results["overfitting_interpretation"] = np.select(
        [
            results["mae_gap"].gt(1.0),
            results["mae_gap"].le(1.0) & results["validation_mae"].gt(results["validation_mae"].min() + 0.05),
        ],
        [
            "Large train advantage over validation = possible overfitting.",
            "Small gap with worse validation MAE = underpowered or unhelpful features.",
        ],
        default="Small gap with improved validation MAE = useful compression.",
    )
    predictions = pd.concat(prediction_tables, ignore_index=True)

    group_rows = []
    anomaly_predictions = predictions[predictions["anomaly_variant"].ne("none")].copy()
    for group_col in ["isolation_is_outlier", "isolation_score_decile"]:
        if group_col not in anomaly_predictions.columns:
            continue
        grouped = anomaly_predictions.groupby(["experiment", "model", "anomaly_variant", group_col], observed=True)
        for keys, group in grouped:
            experiment, model_name, anomaly_variant, group_value = keys
            group_rows.append(
                {
                    "experiment": experiment,
                    "model": model_name,
                    "anomaly_variant": anomaly_variant,
                    "group": group_col,
                    "group_value": group_value,
                    "rows": len(group),
                    "mae": float(group["absolute_error"].mean()),
                    "rmse": float(np.sqrt(np.mean(group["error"] ** 2))),
                    "mean_error": float(group["error"].mean()),
                    "target_mean": float(group[TARGET_COL].mean()),
                    "prediction_mean": float(group["prediction"].mean()),
                }
            )
    error_by_group = pd.DataFrame(group_rows)

    example_cols = ["game_date", "season_start", "gameId", "personId", "player_name", "playerteamName", "opponentteamName", "starter_label", "primary_position", TARGET_COL]
    example_rows = []
    for variant_name, anomaly_features in isolation_features_by_variant.items():
        examples = validation[example_cols].copy()
        examples = pd.concat([examples.reset_index(drop=True), anomaly_features.loc[validation.index, ISOLATION_FEATURE_COLUMNS].reset_index(drop=True)], axis=1)
        examples["anomaly_variant"] = variant_name
        example_rows.append(examples.sort_values("isolation_anomaly_score", ascending=False).head(50))
    anomaly_examples = pd.concat(example_rows, ignore_index=True)

    results.to_csv(TABLES_DIR / "isolation_forest_model_results.csv", index=False)
    error_by_group.to_csv(TABLES_DIR / "isolation_forest_error_by_anomaly_group.csv", index=False)
    anomaly_examples.to_csv(TABLES_DIR / "isolation_forest_anomaly_examples.csv", index=False)

    plt.figure(figsize=(9, 5))
    for variant_name, anomaly_features in isolation_features_by_variant.items():
        scores = anomaly_features.loc[train.index.union(validation.index), "isolation_anomaly_score"].dropna()
        plt.hist(scores, bins=50, alpha=0.45, label=variant_name)
    plt.xlabel("Isolation anomaly score (higher = more anomalous)")
    plt.ylabel("Rows")
    plt.title("Isolation Forest Score Distribution")
    plt.legend()
    _savefig(FIGURES_DIR / "isolation_forest_score_distribution.png")

    best_anomaly = results[results["anomaly_variant"].ne("none")].iloc[0]
    plot_source = error_by_group[
        error_by_group["experiment"].eq(best_anomaly["experiment"])
        & error_by_group["model"].eq(best_anomaly["model"])
        & error_by_group["group"].eq("isolation_score_decile")
    ].sort_values("group_value")
    plt.figure(figsize=(8, 5))
    plt.plot(plot_source["group_value"], plot_source["mae"], marker="o")
    plt.xlabel("Isolation score decile (10 = most anomalous)")
    plt.ylabel("Validation MAE")
    plt.title(f"Validation MAE By Anomaly Decile: {best_anomaly['experiment']} / {best_anomaly['model']}")
    _savefig(FIGURES_DIR / "isolation_forest_validation_mae_by_decile.png")

    best_reference = results[results["anomaly_variant"].eq("none")].sort_values("validation_mae").iloc[0]
    best_anomaly = results[results["anomaly_variant"].ne("none")].sort_values("validation_mae").iloc[0]
    summary = f"""# Step 13c Isolation Forest Anomaly Features

Isolation Forest was fit on train rows only and transformed validation rows only. The 2025 test split was not used.

Direct current-game `starter_minutes`, actual current-game `numMinutes`, and the target were excluded from anomaly inputs.

Best reference validation MAE in this section: `{best_reference['validation_mae']:.4f}` from `{best_reference['experiment']}` with `{best_reference['model']}`.

Best anomaly-feature validation MAE: `{best_anomaly['validation_mae']:.4f}` from `{best_anomaly['experiment']}` with `{best_anomaly['model']}`.

If anomaly deciles show higher MAE in high-score groups, the feature can be useful as a prediction reliability indicator even when it does not improve the main validation MAE.
"""
    (STEP_SUMMARIES_DIR / "step_13c_isolation_forest_anomaly_features.md").write_text(summary, encoding="utf-8")

    return {
        "isolation_forest_feature_plan": feature_plan,
        "isolation_forest_model_results": results,
        "isolation_forest_error_by_anomaly_group": error_by_group,
        "isolation_forest_anomaly_examples": anomaly_examples,
    }


isolation_outputs = run_isolation_forest_experiment(loaded.data, loaded.feature_sets)
isolation_forest_results = isolation_outputs["isolation_forest_model_results"]
display(isolation_forest_results)
display(isolation_outputs["isolation_forest_error_by_anomaly_group"].head(20))


,experiment,base_feature_set,anomaly_variant,model,n_base_features,n_features,train_rows,validation_rows,train_mae,validation_mae,mae_gap,train_rmse,validation_rmse,rmse_gap,train_r2,validation_r2,r2_gap,overfitting_interpretation
0,lineup_aware_starter_flag_roll10_best_pca_plus...,lineup_aware_starter_flag_roll10,lineup_aware_anomaly,hist_gradient_boosting,11,44,76937,26162,7.359589,7.579672,0.220083,9.389132,9.694711,0.305579,0.601145,0.577301,0.023843,Small gap with improved validation MAE = usefu...
1,lineup_aware_starter_flag_roll10_best_pca_refe...,lineup_aware_starter_flag_roll10,none,hist_gradient_boosting,11,41,76937,26162,7.409026,7.581493,0.172467,9.452620,9.694047,0.241428,0.595732,0.577359,0.018373,Small gap with improved validation MAE = usefu...
2,lineup_aware_starter_flag_roll10_best_pca_refe...,lineup_aware_starter_flag_roll10,none,ridge,11,41,76937,26162,7.637865,7.589703,-0.048162,9.766861,9.718277,-0.048584,0.568407,0.575244,-0.006837,Small gap with improved validation MAE = usefu...
3,lineup_aware_starter_flag_roll10_best_pca_plus...,lineup_aware_starter_flag_roll10,lineup_aware_anomaly,ridge,11,44,76937,26162,7.637594,7.590075,-0.047519,9.766736,9.718879,-0.047857,0.568418,0.575191,-0.006773,Small gap with improved validation MAE = usefu...
4,lineup_aware_starter_flag_roll10_reference,lineup_aware_starter_flag_roll10,none,hist_gradient_boosting,11,11,76937,26162,7.621970,7.655891,0.033922,9.724880,9.782516,0.057636,0.572109,0.569610,0.002499,Small gap with worse validation MAE = underpow...
5,lineup_aware_starter_flag_roll10_plus_isolation,lineup_aware_starter_flag_roll10,lineup_aware_anomaly,hist_gradient_boosting,11,14,76937,26162,7.634612,7.662170,0.027559,9.738302,9.781376,0.043074,0.570927,0.569710,0.001217,Small gap with worse validation MAE = underpow...
6,lineup_aware_starter_flag_roll10_reference,lineup_aware_starter_flag_roll10,none,ridge,11,11,76937,26162,7.794193,7.721617,-0.072576,9.961254,9.865872,-0.095382,0.551056,0.562244,-0.011188,Small gap with worse validation MAE = underpow...
7,lineup_aware_starter_flag_roll10_plus_isolation,lineup_aware_starter_flag_roll10,lineup_aware_anomaly,ridge,11,14,76937,26162,7.792954,7.723145,-0.069809,9.959976,9.868341,-0.091634,0.551171,0.562025,-0.010854,Small gap with worse validation MAE = underpow...
8,pre_lineup_player_team_opp_plus_isolation,pre_lineup_player_team_opp,pre_lineup_anomaly,hist_gradient_boosting,484,487,76937,26162,7.349433,7.730823,0.381389,9.363137,9.897421,0.534284,0.603350,0.559440,0.043910,Small gap with worse validation MAE = underpow...
9,pre_lineup_player_team_opp_reference,pre_lineup_player_team_opp,none,hist_gradient_boosting,484,484,76937,26162,7.354057,7.731509,0.377452,9.368112,9.900762,0.532649,0.602928,0.559142,0.043786,Small gap with worse validation MAE = underpow...


,experiment,model,anomaly_variant,group,group_value,rows,mae,rmse,mean_error,target_mean,prediction_mean
0,lineup_aware_starter_flag_roll10_best_pca_plus...,hist_gradient_boosting,lineup_aware_anomaly,isolation_is_outlier,0.0,24301,7.545496,9.644414,0.069469,21.522715,21.453246
1,lineup_aware_starter_flag_roll10_best_pca_plus...,hist_gradient_boosting,lineup_aware_anomaly,isolation_is_outlier,1.0,1861,8.025945,10.329028,0.112129,24.331274,24.219144
2,lineup_aware_starter_flag_roll10_best_pca_plus...,ridge,lineup_aware_anomaly,isolation_is_outlier,0.0,24301,7.554846,9.667016,0.034003,21.522715,21.488712
3,lineup_aware_starter_flag_roll10_best_pca_plus...,ridge,lineup_aware_anomaly,isolation_is_outlier,1.0,1861,8.050094,10.372331,0.652197,24.331274,23.679076
4,lineup_aware_starter_flag_roll10_plus_isolation,hist_gradient_boosting,lineup_aware_anomaly,isolation_is_outlier,0.0,24301,7.631014,9.734473,0.119261,21.522715,21.403454
5,lineup_aware_starter_flag_roll10_plus_isolation,hist_gradient_boosting,lineup_aware_anomaly,isolation_is_outlier,1.0,1861,8.069009,10.374385,0.280094,24.331274,24.051179
6,lineup_aware_starter_flag_roll10_plus_isolation,ridge,lineup_aware_anomaly,isolation_is_outlier,0.0,24301,7.688484,9.816526,-0.139912,21.522715,21.662627
7,lineup_aware_starter_flag_roll10_plus_isolation,ridge,lineup_aware_anomaly,isolation_is_outlier,1.0,1861,8.175756,10.521547,0.733914,24.331274,23.597359
8,pre_lineup_compact_40_plus_scoring_plus_isolation,hist_gradient_boosting,pre_lineup_anomaly,isolation_is_outlier,0.0,23143,7.695540,9.799681,-0.016095,21.011887,21.027982
9,pre_lineup_compact_40_plus_scoring_plus_isolation,hist_gradient_boosting,pre_lineup_anomaly,isolation_is_outlier,1.0,3019,8.428679,10.961495,0.564491,27.169891,26.605400


## 14. HGB Tuning

After feature-family comparisons, this section tunes HistGradientBoosting on the compact scoring feature set using validation performance.


In [24]:
from pathlib import Path

import pandas as pd



REFINEMENT_GROUPS = {
    "role": [
        "current_is_starter_roll_5",
        "current_is_starter_roll_30",
        "starter_minutes_share_roll_30",
        "numMinutes_momentum_5_vs_10",
        "numMinutes_momentum_10_vs_30",
    ],
    "scoring": [
        "points_roll_5",
        "points_roll_30",
        "points_momentum_5_vs_10",
        "points_momentum_10_vs_30",
        "usagePercentage_z_5_vs_30",
    ],
    "ceiling": [
        "reboundsTotal_roll_30",
        "assists_roll_30",
        "playerImpactEstimate_roll_30",
        "playerImpactEstimate_momentum_5_vs_10",
        "trueShootingPercentage_roll_30",
    ],
    "matchup": [
        "opp_pos_fp_allowed_per_player_roll_10",
        "pos_opp_difficulty_roll_10",
        "opp_fp_allowed_roll_30",
        "opp_defensiveRating_roll_30",
        "opp_pace_roll_30",
    ],
}

REFINEMENT_MODEL_NAMES = ["ridge", "hist_gradient_boosting"]
FOCUS_SEGMENTS = {
    ("minutes_bucket", "35+"),
    ("fp_bucket", "40-50"),
    ("fp_bucket", "50+"),
    ("starter_label", "Starter"),
}


def _dedupe(features: list[str]) -> list[str]:
    # Return values in their first-seen order with duplicates removed.
    return list(dict.fromkeys(features))


def compact_base_features() -> list[str]:
    # Return the planned pre-lineup compact feature list.
    return [feature for feature, _, _ in COMPACT_40_FEATURE_PLAN]


def build_refinement_feature_sets(available_features: dict[str, list[str]]) -> dict[str, list[str]]:
    # Build compact refinement variants without changing the global feature-set reports.
    compact = available_features["pre_lineup_compact_40"]
    broad = available_features["pre_lineup_player_team_opp"]
    variants = {
        "pre_lineup_compact_40": compact,
        "pre_lineup_player_team_opp": broad,
    }

    for group_name, group_features in REFINEMENT_GROUPS.items():
        variants[f"pre_lineup_compact_40_plus_{group_name}"] = _dedupe(compact + group_features)

    all_candidates = []
    for group_features in REFINEMENT_GROUPS.values():
        all_candidates.extend(group_features)
    variants["pre_lineup_compact_40_plus_all_candidates"] = _dedupe(compact + all_candidates)
    return variants


def refinement_feature_plan(feature_sets: dict[str, list[str]]) -> pd.DataFrame:
    # Return a long table describing which refinement features are in each variant.
    rows = []
    base = set(feature_sets["pre_lineup_compact_40"])
    for feature_set, features in feature_sets.items():
        for position, feature in enumerate(features, start=1):
            group = "compact_40_base" if feature in base else _refinement_group_for_feature(feature)
            rows.append(
                {
                    "feature_set": feature_set,
                    "position": position,
                    "feature": feature,
                    "feature_group": group,
                    "is_added_candidate": feature not in base,
                }
            )
    return pd.DataFrame(rows)


def _refinement_group_for_feature(feature: str) -> str:
    # Label each refinement feature by the candidate group it came from.
    for group_name, features in REFINEMENT_GROUPS.items():
        if feature in features:
            return group_name
    return "broad_reference_only"


def focus_segment_errors(segment_errors: pd.DataFrame) -> pd.DataFrame:
    # Keep the high-value error slices used to judge compact refinement.
    mask = pd.Series(False, index=segment_errors.index)
    for segment, value in FOCUS_SEGMENTS:
        mask = mask | (segment_errors["segment"].eq(segment) & segment_errors["segment_value"].astype(str).eq(value))
    return segment_errors.loc[mask].sort_values(["segment", "segment_value", "mae"]).reset_index(drop=True)


def compact_refinement_summary(results: pd.DataFrame, focus_errors: pd.DataFrame) -> pd.DataFrame:
    # Create one row per fitted model with overall and focus-segment MAE.
    focus_pivot = focus_errors.pivot_table(
        index=["feature_set", "model"],
        columns=["segment", "segment_value"],
        values="mae",
        aggfunc="first",
    )
    focus_pivot.columns = [f"{segment}_{value}_mae" for segment, value in focus_pivot.columns]
    focus_pivot = focus_pivot.reset_index()

    summary = results.merge(focus_pivot, on=["feature_set", "model"], how="left")
    base_mae = summary.loc[
        summary["feature_set"].eq("pre_lineup_compact_40") & summary["model"].eq("hist_gradient_boosting"),
        "mae",
    ].min()
    broad_mae = summary.loc[
        summary["feature_set"].eq("pre_lineup_player_team_opp") & summary["model"].eq("hist_gradient_boosting"),
        "mae",
    ].min()
    summary["mae_delta_vs_compact_hgb"] = summary["mae"] - base_mae
    summary["mae_delta_vs_broad_hgb"] = summary["mae"] - broad_mae
    return summary.sort_values(["mae", "rmse"]).reset_index(drop=True)


def write_step_summary(path: Path, summary: pd.DataFrame) -> None:
    # Write a lecturer-friendly Markdown summary for compact refinement.
    best = summary.sort_values("mae").iloc[0]
    compact_hgb = summary[
        summary["feature_set"].eq("pre_lineup_compact_40") & summary["model"].eq("hist_gradient_boosting")
    ].iloc[0]
    broad_hgb = summary[
        summary["feature_set"].eq("pre_lineup_player_team_opp") & summary["model"].eq("hist_gradient_boosting")
    ].iloc[0]
    content = f"""# Step 08: Compact Feature Refinement

## Goal

This step refined the compact 40-feature model with small, basketball-explainable candidate groups.

The goal was to improve the compact model without returning to a broad 400+ feature approach.

## Variants Tested

- `pre_lineup_compact_40`: original compact baseline.
- `pre_lineup_compact_40_plus_role`: adds role and minute-momentum features.
- `pre_lineup_compact_40_plus_scoring`: adds scoring and usage-ceiling features.
- `pre_lineup_compact_40_plus_ceiling`: adds all-around ceiling features.
- `pre_lineup_compact_40_plus_matchup`: adds matchup refinements.
- `pre_lineup_compact_40_plus_all_candidates`: adds every candidate group.
- `pre_lineup_player_team_opp`: broad 471-feature reference.

## Why This And Not More Broad Features

The previous step showed that 40 features nearly matched 471 features. Therefore, the next rational test is controlled feature swaps, not adding every available column.

Each candidate group is interpretable:

- role: likely minutes and starter-like usage
- scoring: scoring ceiling and usage movement
- ceiling: rebounds, assists, impact, efficiency
- matchup: position and opponent context

## Best Result

```text
feature_set = {best['feature_set']}
model       = {best['model']}
features    = {int(best['n_features'])}
MAE         = {best['mae']:.4f}
RMSE        = {best['rmse']:.4f}
R2          = {best['r2']:.4f}
```

## Key Comparison

```text
original compact HGB MAE = {compact_hgb['mae']:.4f}
broad reference HGB MAE  = {broad_hgb['mae']:.4f}
best refinement MAE      = {best['mae']:.4f}
```

## Interpretation

The winning variant should be judged against two goals:

1. Does it improve overall MAE?
2. Does it help the hard ceiling segments such as 35+ minutes and 50+ fantasy points?

The full numeric comparison is saved in:

```text
reports/tables/compact_refinement_summary.csv
reports/tables/compact_refinement_error_focus_segments.csv
```

## Next Step

If a small variant beats the compact baseline, promote it as the new compact candidate.

If no variant improves overall MAE, keep `pre_lineup_compact_40` as the main explainable model and move to model tuning or a cleaner target-specific ceiling approach.
"""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def run_compact_refinement(
    tables_dir: Path = TABLES_DIR,
    figures_dir: Path = FIGURES_DIR,
) -> dict[str, pd.DataFrame]:
    # Run compact refinement experiments and write report artifacts.
    loaded = load_validation_data()
    feature_sets = build_refinement_feature_sets(loaded.feature_sets)

    rows = []
    prediction_tables = []
    total_runs = len(feature_sets) * len(REFINEMENT_MODEL_NAMES)
    completed = 0
    for feature_set, features in feature_sets.items():
        for model_name in REFINEMENT_MODEL_NAMES:
            completed += 1
            print(f"[{completed}/{total_runs}] {feature_set} / {model_name}", flush=True)
            row, predictions = run_one_experiment(loaded.data, feature_set, features, model_name)
            rows.append(row)
            prediction_tables.append(predictions)

    results = pd.DataFrame(rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
    predictions = pd.concat(prediction_tables, ignore_index=True)
    segment_errors = error_by_segment(predictions)
    focus_errors = focus_segment_errors(segment_errors)
    summary = compact_refinement_summary(results, focus_errors)

    outputs = {
        "compact_refinement_feature_plan": refinement_feature_plan(feature_sets),
        "compact_refinement_model_results": results,
        "compact_refinement_predictions": predictions,
        "compact_refinement_error_by_segment": segment_errors,
        "compact_refinement_error_focus_segments": focus_errors,
        "compact_refinement_summary": summary,
        "compact_refinement_model_feature_set_inputs": feature_set_inputs(feature_sets),
    }

    tables_dir.mkdir(parents=True, exist_ok=True)
    for name, table in outputs.items():
        table.to_csv(tables_dir / f"{name}.csv", index=False)

    write_validation_plots(predictions, results, figures_dir=figures_dir)
    write_step_summary(
        tables_dir.parent / "step_summaries" / "step_08_compact_refinement.md",
        summary,
    )
    return outputs


In [25]:
compact_refinement_outputs = run_compact_refinement(tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR)
for name, table in compact_refinement_outputs.items():
    print(name, table.shape)
display(compact_refinement_outputs['compact_refinement_summary'].head(10))


[1/14] pre_lineup_compact_40 / ridge


[2/14] pre_lineup_compact_40 / hist_gradient_boosting


[3/14] pre_lineup_player_team_opp / ridge


[4/14] pre_lineup_player_team_opp / hist_gradient_boosting


[5/14] pre_lineup_compact_40_plus_role / ridge


...

[12/14] pre_lineup_compact_40_plus_matchup / hist_gradient_boosting


[13/14] pre_lineup_compact_40_plus_all_candidates / ridge


[14/14] pre_lineup_compact_40_plus_all_candidates / hist_gradient_boosting


compact_refinement_feature_plan (766, 5)
compact_refinement_model_results (14, 11)
compact_refinement_predictions (366268, 18)
compact_refinement_error_by_segment (224, 10)
compact_refinement_error_focus_segments (56, 10)
compact_refinement_summary (14, 17)
compact_refinement_model_feature_set_inputs (766, 3)


,feature_set,model,n_features,train_rows,validation_rows,fit_predict_seconds,mae,rmse,r2,mean_error,median_absolute_error,fp_bucket_40-50_mae,fp_bucket_50+_mae,minutes_bucket_35+_mae,starter_label_Starter_mae,mae_delta_vs_compact_hgb,mae_delta_vs_broad_hgb
0,pre_lineup_player_team_opp,hist_gradient_boosting,486,76937,26162,60.997,7.731509,9.900762,0.559142,0.163186,6.336169,10.636216,16.671223,10.003414,8.738365,-0.051730,0.000000
1,pre_lineup_player_team_opp,ridge,486,76937,26162,29.844,7.747664,9.912286,0.558116,-0.018882,6.387147,10.509575,16.394982,9.843303,8.699125,-0.035574,0.016155
2,pre_lineup_compact_40_plus_ceiling,hist_gradient_boosting,45,76937,26162,3.554,7.772098,9.930822,0.556461,0.048157,6.411085,10.557946,16.709017,9.970472,8.737234,-0.011140,0.040589
3,pre_lineup_compact_40_plus_role,hist_gradient_boosting,45,76937,26162,3.982,7.772410,9.934751,0.556110,0.042876,6.418030,10.575489,16.662648,9.958547,8.741745,-0.010828,0.040901
4,pre_lineup_compact_40_plus_all_candidates,hist_gradient_boosting,60,76937,26162,4.234,7.774630,9.929588,0.556571,0.051561,6.420540,10.595053,16.659914,9.978811,8.741137,-0.008608,0.043121
5,pre_lineup_compact_40_plus_matchup,hist_gradient_boosting,45,76937,26162,3.441,7.779499,9.941290,0.555526,0.028313,6.415522,10.585109,16.624856,9.966508,8.747157,-0.003740,0.047990
6,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,45,76937,26162,4.281,7.781769,9.947805,0.554943,0.045904,6.415698,10.573417,16.713693,10.008260,8.757450,-0.001469,0.050261
7,pre_lineup_compact_40,hist_gradient_boosting,40,76937,26162,3.739,7.783238,9.948584,0.554873,0.036283,6.432807,10.591078,16.673030,9.993110,8.760730,0.000000,0.051730
8,pre_lineup_compact_40_plus_all_candidates,ridge,60,76937,26162,1.081,7.790148,9.944748,0.555216,-0.103517,6.426626,10.489009,16.427584,9.860669,8.705346,0.006910,0.058639
9,pre_lineup_compact_40_plus_role,ridge,45,76937,26162,0.795,7.791165,9.945362,0.555162,-0.135200,6.436351,10.463646,16.390222,9.845023,8.700519,0.007926,0.059656


In [26]:
import os
from pathlib import Path
from time import perf_counter

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline



TUNED_FEATURE_SET = "pre_lineup_compact_40_plus_scoring"

TUNING_CONFIGS = [
    {
        "config_name": "reference_current_hgb",
        "learning_rate": 0.05,
        "max_iter": 160,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.1,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "slower_learning_more_trees",
        "learning_rate": 0.03,
        "max_iter": 260,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.1,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "slower_learning_more_trees_l2_03",
        "learning_rate": 0.03,
        "max_iter": 260,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.3,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "fast_learning_fewer_trees",
        "learning_rate": 0.08,
        "max_iter": 120,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.1,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "fast_learning_more_regularized",
        "learning_rate": 0.08,
        "max_iter": 140,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.5,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "small_trees",
        "learning_rate": 0.05,
        "max_iter": 180,
        "max_leaf_nodes": 15,
        "l2_regularization": 0.1,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "small_trees_more_trees",
        "learning_rate": 0.03,
        "max_iter": 300,
        "max_leaf_nodes": 15,
        "l2_regularization": 0.1,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "large_trees",
        "learning_rate": 0.05,
        "max_iter": 160,
        "max_leaf_nodes": 63,
        "l2_regularization": 0.1,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "large_trees_more_l2",
        "learning_rate": 0.05,
        "max_iter": 180,
        "max_leaf_nodes": 63,
        "l2_regularization": 0.5,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "large_trees_high_leaf_min",
        "learning_rate": 0.05,
        "max_iter": 180,
        "max_leaf_nodes": 63,
        "l2_regularization": 0.3,
        "min_samples_leaf": 40,
        "early_stopping": "auto",
    },
    {
        "config_name": "low_l2",
        "learning_rate": 0.05,
        "max_iter": 180,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.0,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "medium_l2",
        "learning_rate": 0.05,
        "max_iter": 180,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.3,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "high_l2",
        "learning_rate": 0.05,
        "max_iter": 180,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.8,
        "min_samples_leaf": 20,
        "early_stopping": "auto",
    },
    {
        "config_name": "small_leaf",
        "learning_rate": 0.05,
        "max_iter": 180,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.1,
        "min_samples_leaf": 10,
        "early_stopping": "auto",
    },
    {
        "config_name": "medium_leaf",
        "learning_rate": 0.05,
        "max_iter": 180,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.1,
        "min_samples_leaf": 40,
        "early_stopping": "auto",
    },
    {
        "config_name": "large_leaf",
        "learning_rate": 0.05,
        "max_iter": 180,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.1,
        "min_samples_leaf": 80,
        "early_stopping": "auto",
    },
    {
        "config_name": "no_internal_early_stop_reference",
        "learning_rate": 0.05,
        "max_iter": 160,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.1,
        "min_samples_leaf": 20,
        "early_stopping": False,
    },
    {
        "config_name": "no_early_stop_more_trees",
        "learning_rate": 0.03,
        "max_iter": 260,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.3,
        "min_samples_leaf": 40,
        "early_stopping": False,
    },
    {
        "config_name": "no_early_stop_small_trees",
        "learning_rate": 0.03,
        "max_iter": 300,
        "max_leaf_nodes": 15,
        "l2_regularization": 0.3,
        "min_samples_leaf": 40,
        "early_stopping": False,
    },
    {
        "config_name": "no_early_stop_fast_regularized",
        "learning_rate": 0.08,
        "max_iter": 120,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.5,
        "min_samples_leaf": 40,
        "early_stopping": False,
    },
]


def tuning_config_table(configs: list[dict[str, object]] | None = None) -> pd.DataFrame:
    # Return the HGB tuning config table.
    return pd.DataFrame(configs or TUNING_CONFIGS)


def make_hgb_pipeline(config: dict[str, object]) -> Pipeline:
    # Build a HistGradientBoosting pipeline from one config row.
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            (
                "model",
                HistGradientBoostingRegressor(
                    learning_rate=float(config["learning_rate"]),
                    max_iter=int(config["max_iter"]),
                    max_leaf_nodes=int(config["max_leaf_nodes"]),
                    l2_regularization=float(config["l2_regularization"]),
                    min_samples_leaf=int(config["min_samples_leaf"]),
                    early_stopping=config["early_stopping"],
                    random_state=MODEL_RANDOM_STATE,
                ),
            ),
        ]
    )


def run_tuning_config(
    data: pd.DataFrame,
    features: list[str],
    config: dict[str, object],
) -> tuple[dict[str, object], pd.DataFrame]:
    # Fit one HGB config on train and evaluate on validation only.
    train = data[data[SPLIT_COL].eq("train")].copy()
    validation = data[data[SPLIT_COL].eq("validation")].copy()
    x_train = _validate_feature_frame(train, features)
    y_train = train[TARGET_COL]
    x_validation = _validate_feature_frame(validation, features)
    y_validation = validation[TARGET_COL]

    model = make_hgb_pipeline(config)
    start = perf_counter()
    model.fit(x_train, y_train)
    predictions = model.predict(x_validation)
    elapsed = perf_counter() - start

    metrics = regression_metrics(y_validation, predictions)
    errors = y_validation.to_numpy() - predictions
    fitted_model = model.named_steps["model"]
    n_iter = getattr(fitted_model, "n_iter_", np.nan)

    row = {
        "feature_set": TUNED_FEATURE_SET,
        "model": "hist_gradient_boosting",
        "config_name": config["config_name"],
        "n_features": len(features),
        "train_rows": len(train),
        "validation_rows": len(validation),
        "fit_predict_seconds": round(elapsed, 3),
        "n_iter": n_iter,
        "mae": metrics["mae"],
        "rmse": metrics["rmse"],
        "r2": metrics["r2"],
        "mean_error": float(np.mean(errors)),
        "median_absolute_error": float(np.median(np.abs(errors))),
        **config,
    }

    prediction_cols = [
        "gameId",
        "personId",
        "game_date",
        "season_start",
        "player_name",
        "playerteamName",
        "opponentteamName",
        "starter_label",
        "primary_position",
        "minutes_bucket",
        "fp_bucket",
        "actual_numMinutes",
        TARGET_COL,
    ]
    prediction_cols = [column for column in prediction_cols if column in validation.columns]
    pred_df = validation[prediction_cols].copy()
    pred_df["feature_set"] = TUNED_FEATURE_SET
    pred_df["model"] = "hist_gradient_boosting"
    pred_df["config_name"] = config["config_name"]
    pred_df["prediction"] = predictions
    pred_df["error"] = pred_df[TARGET_COL] - pred_df["prediction"]
    pred_df["absolute_error"] = pred_df["error"].abs()
    return row, pred_df


def tuning_summary(results: pd.DataFrame, focus_errors: pd.DataFrame) -> pd.DataFrame:
    # Attach focus-segment MAE and deltas to tuning results.
    focus_pivot = focus_errors.pivot_table(
        index=["feature_set", "model", "config_name"],
        columns=["segment", "segment_value"],
        values="mae",
        aggfunc="first",
    )
    focus_pivot.columns = [f"{segment}_{value}_mae" for segment, value in focus_pivot.columns]
    focus_pivot = focus_pivot.reset_index()

    summary = results.merge(focus_pivot, on=["feature_set", "model", "config_name"], how="left")
    reference_mae = summary.loc[summary["config_name"].eq("reference_current_hgb"), "mae"].min()
    best_mae = summary["mae"].min()
    summary["mae_delta_vs_reference"] = summary["mae"] - reference_mae
    summary["mae_delta_vs_best"] = summary["mae"] - best_mae
    return summary.sort_values(["mae", "rmse"]).reset_index(drop=True)


def tuning_error_by_segment(predictions: pd.DataFrame) -> pd.DataFrame:
    # Summarize validation errors by segment while preserving tuning config.
    rows = []
    for segment in ["starter_label", "primary_position", "minutes_bucket", "fp_bucket"]:
        if segment not in predictions.columns:
            continue
        grouped = predictions.groupby(["feature_set", "model", "config_name", segment], observed=True)
        for keys, group in grouped:
            feature_set, model_name, config_name, value = keys
            rows.append(
                {
                    "feature_set": feature_set,
                    "model": model_name,
                    "config_name": config_name,
                    "segment": segment,
                    "segment_value": value,
                    "rows": len(group),
                    "mae": float(group["absolute_error"].mean()),
                    "rmse": float(np.sqrt(np.mean(group["error"] ** 2))),
                    "mean_error": float(group["error"].mean()),
                    "target_mean": float(group[TARGET_COL].mean()),
                    "prediction_mean": float(group["prediction"].mean()),
                }
            )
    return pd.DataFrame(rows)


def write_step_summary(path: Path, summary: pd.DataFrame) -> None:
    # Write a lecturer-friendly Markdown summary for HGB tuning.
    best = summary.iloc[0]
    reference = summary[summary["config_name"].eq("reference_current_hgb")].iloc[0]
    content = f"""# Step 09: Focused HGB Tuning

## Goal

This step tuned only the best current feature set:

```text
{TUNED_FEATURE_SET}
```

The goal was to improve validation MAE without adding more features or using the untouched 2025 test set.

## Why Focused Tuning

The previous step found that the 45-feature scoring compact model beat the 471-feature broad model. Therefore, the next useful question is whether HistGradientBoosting settings can extract more signal from the same 45 features.

We did not tune every model and every feature set because that would increase overfitting risk and make the experiment harder to explain.

## Search Space

The tuning varied:

- learning rate
- number of boosting iterations
- tree size through `max_leaf_nodes`
- L2 regularization
- minimum samples per leaf
- internal early stopping

The current HGB settings were included as `reference_current_hgb`.

## Best Result

```text
config_name = {best['config_name']}
features    = {int(best['n_features'])}
MAE         = {best['mae']:.4f}
RMSE        = {best['rmse']:.4f}
R2          = {best['r2']:.4f}
```

## Reference Comparison

```text
reference MAE = {reference['mae']:.4f}
best MAE      = {best['mae']:.4f}
improvement   = {reference['mae'] - best['mae']:.4f}
```

## Best Parameters

```text
learning_rate       = {best['learning_rate']}
max_iter            = {int(best['max_iter'])}
max_leaf_nodes      = {int(best['max_leaf_nodes'])}
l2_regularization   = {best['l2_regularization']}
min_samples_leaf    = {int(best['min_samples_leaf'])}
early_stopping      = {best['early_stopping']}
```

## Interpretation

This step uses validation only. The 2025 test period remains untouched.

The full tuning table and focus-segment errors are saved in:

```text
reports/tables/hgb_tuning_summary.csv
reports/tables/hgb_tuning_focus_segments.csv
```

## Next Step

If the best tuned config improves validation meaningfully, promote it as the current final validation candidate.

Then run model interpretation on the tuned 45-feature model before touching the test set.
"""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def run_hgb_tuning(
    tables_dir: Path = TABLES_DIR,
    figures_dir: Path = FIGURES_DIR,
    configs: list[dict[str, object]] | None = None,
) -> dict[str, pd.DataFrame]:
    # Run focused HGB tuning on the compact scoring feature set.
    loaded = load_validation_data()
    feature_sets = build_refinement_feature_sets(loaded.feature_sets)
    features = feature_sets[TUNED_FEATURE_SET]
    configs = configs or TUNING_CONFIGS

    rows = []
    prediction_tables = []
    total = len(configs)
    for index, config in enumerate(configs, start=1):
        print(f"[{index}/{total}] {config['config_name']}", flush=True)
        row, preds = run_tuning_config(loaded.data, features, config)
        rows.append(row)
        prediction_tables.append(preds)

    results = pd.DataFrame(rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
    predictions = pd.concat(prediction_tables, ignore_index=True)
    segment_errors = tuning_error_by_segment(predictions)
    focus_errors = focus_segment_errors(segment_errors)
    summary = tuning_summary(results, focus_errors)

    outputs = {
        "hgb_tuning_config_plan": tuning_config_table(configs),
        "hgb_tuning_model_results": results,
        "hgb_tuning_predictions": predictions,
        "hgb_tuning_error_by_segment": segment_errors,
        "hgb_tuning_focus_segments": focus_errors,
        "hgb_tuning_summary": summary,
    }

    tables_dir.mkdir(parents=True, exist_ok=True)
    for name, table in outputs.items():
        table.to_csv(tables_dir / f"{name}.csv", index=False)

    plot_results = results.copy()
    plot_predictions = predictions.copy()
    plot_results["feature_set"] = plot_results["config_name"]
    plot_predictions["feature_set"] = plot_predictions["config_name"]
    write_validation_plots(plot_predictions, plot_results, figures_dir=figures_dir)
    write_step_summary(tables_dir.parent / "step_summaries" / "step_09_hgb_tuning.md", summary)
    return outputs


In [27]:
hgb_tuning_outputs = run_hgb_tuning(tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR)
for name, table in hgb_tuning_outputs.items():
    print(name, table.shape)
display(hgb_tuning_outputs['hgb_tuning_summary'].head(10))


[1/20] reference_current_hgb


[2/20] slower_learning_more_trees


[3/20] slower_learning_more_trees_l2_03


[4/20] fast_learning_fewer_trees


[5/20] fast_learning_more_regularized


...

[18/20] no_early_stop_more_trees


[19/20] no_early_stop_small_trees


[20/20] no_early_stop_fast_regularized


hgb_tuning_config_plan (20, 7)
hgb_tuning_model_results (20, 19)
hgb_tuning_predictions (523240, 19)
hgb_tuning_error_by_segment (320, 11)
hgb_tuning_focus_segments (80, 11)
hgb_tuning_summary (20, 25)


,feature_set,model,config_name,n_features,train_rows,validation_rows,fit_predict_seconds,n_iter,mae,rmse,...,max_leaf_nodes,l2_regularization,min_samples_leaf,early_stopping,fp_bucket_40-50_mae,fp_bucket_50+_mae,minutes_bucket_35+_mae,starter_label_Starter_mae,mae_delta_vs_reference,mae_delta_vs_best
0,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,no_early_stop_more_trees,45,76937,26162,6.574,260,7.769961,9.933005,...,31,0.3,40,False,10.556069,16.679323,9.977511,8.740051,-0.011808,0.000000
1,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,large_leaf,45,76937,26162,3.352,118,7.772058,9.934851,...,31,0.1,80,auto,10.583512,16.611863,9.956611,8.741634,-0.009711,0.002097
2,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,no_internal_early_stop_reference,45,76937,26162,4.017,160,7.773356,9.936894,...,31,0.1,20,False,10.546513,16.660943,9.976498,8.743295,-0.008413,0.003395
3,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,slower_learning_more_trees,45,76937,26162,5.587,234,7.776870,9.939459,...,31,0.1,20,auto,10.575547,16.691755,9.991101,8.751086,-0.004900,0.006909
4,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,slower_learning_more_trees_l2_03,45,76937,26162,5.057,207,7.777596,9.937697,...,31,0.3,20,auto,10.589373,16.733487,9.989911,8.745845,-0.004173,0.007635
5,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,no_early_stop_small_trees,45,76937,26162,5.887,300,7.777977,9.936623,...,15,0.3,40,False,10.555661,16.637738,9.974332,8.743610,-0.003792,0.008016
6,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,medium_leaf,45,76937,26162,3.177,107,7.778246,9.934659,...,31,0.1,40,auto,10.612279,16.724873,9.978293,8.746142,-0.003524,0.008285
7,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,medium_l2,45,76937,26162,3.626,132,7.779718,9.942046,...,31,0.3,20,auto,10.635849,16.737121,10.007266,8.753487,-0.002052,0.009757
8,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,high_l2,45,76937,26162,3.599,135,7.781258,9.946000,...,31,0.8,20,auto,10.600761,16.746523,10.007718,8.754388,-0.000511,0.011297
9,pre_lineup_compact_40_plus_scoring,hist_gradient_boosting,low_l2,45,76937,26162,3.445,130,7.781689,9.942122,...,31,0.0,20,auto,10.571199,16.731025,9.993849,8.754032,-0.000081,0.011727


## 14b. Valid 11-Feature Replication

This compact replication keeps only valid shifted-history starter-minute features. Direct current-game starter minutes are excluded.


In [28]:
from pathlib import Path

import pandas as pd



SAFE_11_FEATURES = [
    "fantasy_points_roll_3",
    "fantasy_points_roll_10",
    "fantasy_points_roll_30",
    "numMinutes_roll_10",
    "home",
    "starter_minutes_roll_10",
    "pos_opp_difficulty_roll_30",
    "usagePercentage_roll_3",
    "usagePercentage_roll_10",
    "usagePercentage_roll_30",
    "usagePercentage_momentum_3_vs_10",
]

SAFE_11_RECENT_STARTER_FEATURES = [
    "fantasy_points_roll_3",
    "fantasy_points_roll_10",
    "fantasy_points_roll_30",
    "numMinutes_roll_10",
    "home",
    "starter_minutes_roll_3",
    "pos_opp_difficulty_roll_30",
    "usagePercentage_roll_3",
    "usagePercentage_roll_10",
    "usagePercentage_roll_30",
    "usagePercentage_momentum_3_vs_10",
]


REPLICATION_FEATURE_SETS = {
    "replication_11_safe": SAFE_11_FEATURES,
    "replication_11_safe_recent_starter": SAFE_11_RECENT_STARTER_FEATURES,
}

REPLICATION_MODELS = ["ridge", "hist_gradient_boosting"]


def replication_feature_plan(available_columns: list[str]) -> pd.DataFrame:
    # Return the 11-feature replication plan with leakage labels.
    available = set(available_columns)
    rows = []
    for feature_set, features in REPLICATION_FEATURE_SETS.items():
        is_valid_pre_game = True
        for position, feature in enumerate(features, start=1):
            rows.append(
                {
                    "feature_set": feature_set,
                    "position": position,
                    "feature": feature,
                    "available": feature in available,
                    "valid_pre_game": is_valid_pre_game,
                    "leakage_note": _leakage_note(feature_set, feature),
                    "source_concept": _source_concept(feature),
                }
            )
    return pd.DataFrame(rows)


def _leakage_note(feature_set: str, feature: str) -> str:
    # Describe whether a replication feature is shifted history or pre-game context.
    if feature.endswith("_roll_3") or feature.endswith("_roll_10") or feature.endswith("_roll_30") or "_momentum_" in feature:
        return "Shifted historical feature; valid before the game."
    return "Pre-game static/context feature." if feature == "home" else "Derived from shifted historical opponent-position data."


def _source_concept(feature: str) -> str:
    # Map a replication feature name to its basketball concept.
    if feature.startswith("fantasy_points"):
        return "recent fantasy production"
    if feature.startswith("numMinutes") or feature.startswith("starter_minutes"):
        return "playing time and starter role"
    if feature == "home":
        return "game location"
    if feature.startswith("usagePercentage"):
        return "offensive involvement"
    if feature.startswith("pos_opp_difficulty"):
        return "position-specific opponent defensive context"
    return "model feature"


def validate_replication_features(columns: list[str]) -> None:
    # Raise a clear error if any planned feature is missing.
    missing = sorted({feature for features in REPLICATION_FEATURE_SETS.values() for feature in features if feature not in columns})
    if missing:
        raise ValueError(f"Missing required replication features: {missing}")


def add_missing_replication_features(data: pd.DataFrame, modeling_path: Path | None = None) -> pd.DataFrame:
    # Load planned replication-only columns that are excluded from normal feature membership.
    missing = sorted({feature for features in REPLICATION_FEATURE_SETS.values() for feature in features if feature not in data.columns})
    if not missing:
        return data

    modeling_path = modeling_path or PROCESSED_DIR / "modeling_table_last4_regular.csv"
    available = pd.read_csv(modeling_path, nrows=0).columns.tolist()
    missing_available = [feature for feature in missing if feature in available]
    if not missing_available:
        return data

    extra = pd.read_csv(modeling_path, usecols=["gameId", "personId"] + missing_available, low_memory=False)
    return data.merge(extra, on=["gameId", "personId"], how="left", validate="one_to_one")



def write_step_summary(path: Path, results: pd.DataFrame, focus_errors: pd.DataFrame) -> None:
    # Write the lecturer-friendly summary for the valid 11-feature replication.
    best = results.sort_values(["mae", "rmse"]).iloc[0]
    safe_best = results[
        results["feature_set"].isin(["replication_11_safe", "replication_11_safe_recent_starter"])
        & results["model"].eq("hist_gradient_boosting")
    ].sort_values("mae").iloc[0]
    content = f"""# Step 10: 11-Feature Replication

## Goal

This step tests whether an 11-feature model can perform well using only valid pre-game features.

## Feature Sets

- `replication_11_safe`: uses shifted historical `starter_minutes_roll_10`.
- `replication_11_safe_recent_starter`: uses shifted historical `starter_minutes_roll_3`.

Direct current-game `starter_minutes` is not included in this comparison.

## Best Result

```text
feature_set = {best['feature_set']}
model       = {best['model']}
features    = {int(best['n_features'])}
MAE         = {best['mae']:.4f}
RMSE        = {best['rmse']:.4f}
R2          = {best['r2']:.4f}
```

## Best Safe HGB

```text
best safe HGB MAE = {safe_best['mae']:.4f}
```

## Outputs

```text
reports/tables/replication_11_feature_plan.csv
reports/tables/replication_11_model_results.csv
reports/tables/replication_11_focus_segments.csv
```
"""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def run_replication_11(
    tables_dir: Path = TABLES_DIR,
    figures_dir: Path = FIGURES_DIR,
    modeling_path: Path | None = None,
) -> dict[str, pd.DataFrame]:
    # Run the 11-feature replication models and write reports.
    loaded = load_validation_data()
    data = add_missing_replication_features(loaded.data, modeling_path=modeling_path)
    available_columns = data.columns.tolist()
    validate_replication_features(available_columns)

    rows = []
    prediction_tables = []
    total = len(REPLICATION_FEATURE_SETS) * len(REPLICATION_MODELS)
    completed = 0
    for feature_set, features in REPLICATION_FEATURE_SETS.items():
        for model_name in REPLICATION_MODELS:
            completed += 1
            print(f"[{completed}/{total}] {feature_set} / {model_name}", flush=True)
            row, predictions = run_one_experiment(data, feature_set, features, model_name)
            rows.append(row)
            prediction_tables.append(predictions)

    results = pd.DataFrame(rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
    predictions = pd.concat(prediction_tables, ignore_index=True)
    segment_errors = error_by_segment(predictions)
    focus_errors = focus_segment_errors(segment_errors)

    outputs = {
        "replication_11_feature_plan": replication_feature_plan(available_columns),
        "replication_11_model_results": results,
        "replication_11_predictions": predictions,
        "replication_11_error_by_segment": segment_errors,
        "replication_11_focus_segments": focus_errors,
    }

    tables_dir.mkdir(parents=True, exist_ok=True)
    for name, table in outputs.items():
        table.to_csv(tables_dir / f"{name}.csv", index=False)

    write_validation_plots(predictions, results, figures_dir=figures_dir)
    write_step_summary(tables_dir.parent / "step_summaries" / "step_10_replication_11.md", results, focus_errors)
    return outputs


In [29]:
replication_outputs = run_replication_11(tables_dir=TABLES_DIR, figures_dir=FIGURES_DIR)
for name, table in replication_outputs.items():
    print(name, table.shape)
display(replication_outputs['replication_11_model_results'])


[1/4] replication_11_safe / ridge


[2/4] replication_11_safe / hist_gradient_boosting


[3/4] replication_11_safe_recent_starter / ridge


[4/4] replication_11_safe_recent_starter / hist_gradient_boosting


replication_11_feature_plan (22, 7)
replication_11_model_results (4, 11)
replication_11_predictions (104648, 18)
replication_11_error_by_segment (64, 10)
replication_11_focus_segments (16, 10)


,feature_set,model,n_features,train_rows,validation_rows,fit_predict_seconds,mae,rmse,r2,mean_error,median_absolute_error
0,replication_11_safe_recent_starter,hist_gradient_boosting,11,76937,26162,1.474,7.817989,9.975005,0.552506,0.083607,6.460537
1,replication_11_safe,hist_gradient_boosting,11,76937,26162,1.504,7.829748,9.998769,0.550371,0.076077,6.482081
2,replication_11_safe_recent_starter,ridge,11,76937,26162,0.186,7.849748,10.020065,0.548454,-0.096245,6.490994
3,replication_11_safe,ridge,11,76937,26162,0.141,7.856017,10.031397,0.547432,-0.055369,6.501903


## 15. Final Model Evaluation And Recommendation

The final validation comparison combines broad, compact, tuned, PCA, and valid starter-focused results. The 2025 test season remains untouched.


In [30]:
best_tuned = hgb_tuning_outputs["hgb_tuning_summary"].iloc[0]
best_safe_11 = replication_outputs["replication_11_model_results"][
    replication_outputs["replication_11_model_results"]["feature_set"].isin(["replication_11_safe", "replication_11_safe_recent_starter"])
    & replication_outputs["replication_11_model_results"]["model"].eq("hist_gradient_boosting")
].sort_values("mae").iloc[0]
broad = selected_results[selected_results["feature_set"].eq("pre_lineup_player_team_opp") & selected_results["model"].eq("hist_gradient_boosting")].iloc[0]
compact_scoring = selected_results[selected_results["feature_set"].eq("pre_lineup_compact_40_plus_scoring") & selected_results["model"].eq("hist_gradient_boosting")].iloc[0]

best_valid_starter = starter_results[starter_results["validity"].isin(["pre-lineup", "lineup-aware"])].sort_values("mae").iloc[0]
starter_history_hgb = starter_results[
    starter_results["feature_set"].eq("pre_lineup_starter_history_roll10")
    & starter_results["model"].eq("hist_gradient_boosting")
].iloc[0]
lineup_starter_hgb = starter_results[
    starter_results["feature_set"].eq("lineup_aware_starter_flag_roll10")
    & starter_results["model"].eq("hist_gradient_boosting")
].iloc[0]
best_pca_valid = pca_results[pca_results["n_pca_components"].gt(0)].sort_values("validation_mae").iloc[0]
best_starter_pca = starter_pca_results.sort_values(["validation_mae", "validation_rmse"]).iloc[0]
best_isolation = isolation_forest_results.sort_values(["validation_mae", "validation_rmse"]).iloc[0]

final_comparison = pd.DataFrame(
    [
        {"model_label": "Broad valid HGB reference", "feature_set": "pre_lineup_player_team_opp", "n_features": int(broad["n_features"]), "validity": "pre-lineup", "valid_pre_game": True, "uses_current_game_minutes": False, "mae": broad["mae"], "rmse": broad["rmse"], "r2": broad["r2"], "notes": "Broad benchmark with player/team/opponent shifted rolling features."},
        {"model_label": "Best valid tuned compact model", "feature_set": "pre_lineup_compact_40_plus_scoring", "n_features": int(best_tuned["n_features"]), "validity": "pre-lineup", "valid_pre_game": True, "uses_current_game_minutes": False, "mae": best_tuned["mae"], "rmse": best_tuned["rmse"], "r2": best_tuned["r2"], "notes": f"Tuned HGB config: {best_tuned['config_name']}"},
        {"model_label": "Compact scoring before tuning", "feature_set": "pre_lineup_compact_40_plus_scoring", "n_features": int(compact_scoring["n_features"]), "validity": "pre-lineup", "valid_pre_game": True, "uses_current_game_minutes": False, "mae": compact_scoring["mae"], "rmse": compact_scoring["rmse"], "r2": compact_scoring["r2"], "notes": "Compact feature-refinement candidate before tuning."},
        {"model_label": "Best valid PCA compact model", "feature_set": str(best_pca_valid["experiment"]), "n_features": int(best_pca_valid["n_features"]), "validity": "pre-lineup", "valid_pre_game": True, "uses_current_game_minutes": False, "mae": best_pca_valid["validation_mae"], "rmse": best_pca_valid["validation_rmse"], "r2": best_pca_valid["validation_r2"], "notes": f"{int(best_pca_valid['n_pca_components'])} PCA components from unused broad features."},
        {"model_label": "Best valid starter-focused model", "feature_set": best_valid_starter["feature_set"], "n_features": int(best_valid_starter["n_features"]), "validity": best_valid_starter["validity"], "valid_pre_game": best_valid_starter["validity"] == "pre-lineup", "uses_current_game_minutes": False, "mae": best_valid_starter["mae"], "rmse": best_valid_starter["rmse"], "r2": best_valid_starter["r2"], "notes": "Starter model family included as a main comparison group."},
        {"model_label": "Best valid starter PCA model", "feature_set": str(best_starter_pca["experiment"]), "n_features": int(best_starter_pca["n_features"]), "validity": "lineup-aware" if str(best_starter_pca["base_feature_set"]).startswith("lineup_aware") else "pre-lineup", "valid_pre_game": str(best_starter_pca["base_feature_set"]).startswith("pre_lineup"), "uses_current_game_minutes": False, "mae": best_starter_pca["validation_mae"], "rmse": best_starter_pca["validation_rmse"], "r2": best_starter_pca["validation_r2"], "notes": f"Starter model plus {int(best_starter_pca['n_pca_components'])} PCA components."},
        {"model_label": "Pre-lineup starter-history HGB", "feature_set": "pre_lineup_starter_history_roll10", "n_features": int(starter_history_hgb["n_features"]), "validity": "pre-lineup", "valid_pre_game": True, "uses_current_game_minutes": False, "mae": starter_history_hgb["mae"], "rmse": starter_history_hgb["rmse"], "r2": starter_history_hgb["r2"], "notes": "Uses shifted starter history including starter_minutes_roll_10."},
        {"model_label": "Lineup-aware starter-flag HGB", "feature_set": "lineup_aware_starter_flag_roll10", "n_features": int(lineup_starter_hgb["n_features"]), "validity": "lineup-aware", "valid_pre_game": False, "uses_current_game_minutes": False, "mae": lineup_starter_hgb["mae"], "rmse": lineup_starter_hgb["rmse"], "r2": lineup_starter_hgb["r2"], "notes": "Uses current_is_starter plus starter_minutes_roll_10, not actual current-game minutes."},
        {"model_label": "Best Isolation Forest anomaly model", "feature_set": str(best_isolation["experiment"]), "n_features": int(best_isolation["n_features"]), "validity": "lineup-aware" if str(best_isolation["base_feature_set"]).startswith("lineup_aware") else "pre-lineup", "valid_pre_game": str(best_isolation["base_feature_set"]).startswith("pre_lineup"), "uses_current_game_minutes": False, "mae": best_isolation["validation_mae"], "rmse": best_isolation["validation_rmse"], "r2": best_isolation["validation_r2"], "notes": f"Isolation Forest anomaly features; variant {best_isolation['anomaly_variant']}."},
        {"model_label": "Best valid 11-feature replication", "feature_set": best_safe_11["feature_set"], "n_features": int(best_safe_11["n_features"]), "validity": "pre-lineup", "valid_pre_game": True, "uses_current_game_minutes": False, "mae": best_safe_11["mae"], "rmse": best_safe_11["rmse"], "r2": best_safe_11["r2"], "notes": "Uses shifted starter-minutes history."},
    ]
).sort_values("mae").reset_index(drop=True)
final_comparison.to_csv(TABLES_DIR / "final_model_comparison.csv", index=False)
best_valid = final_comparison[final_comparison["validity"].isin(["pre-lineup", "lineup-aware"])].iloc[0]
best_lineup_aware = final_comparison[final_comparison["validity"].eq("lineup-aware")].iloc[0]
best_pre_lineup = final_comparison[final_comparison["validity"].eq("pre-lineup")].iloc[0]
best_compact = final_comparison[final_comparison["model_label"].eq("Best valid tuned compact model")].iloc[0]
display(final_comparison)

summary_text = f"""# End-To-End Summary

This final run uses NBA seasons `2021` through `2025`.

Split policy:

- Train: full seasons `2021`, `2022`, `2023`
- Validation: full season `2024`
- Test: full season `2025`

Lowest validation MAE among valid lineup-aware or pre-lineup models: `{best_valid['model_label']}` / `{best_valid['feature_set']}`.

Best lineup-aware validation MAE: {best_lineup_aware['mae']:.4f} from `{best_lineup_aware['feature_set']}`.

Best pre-lineup validation MAE: {best_pre_lineup['mae']:.4f} from `{best_pre_lineup['feature_set']}`.

Best compact explainable candidate: `{best_compact['feature_set']}` with tuned HGB, MAE {best_compact['mae']:.4f}.

Best Isolation Forest anomaly model: `{best_isolation['experiment']}` with MAE {best_isolation['validation_mae']:.4f}.

The 2025 test split remains untouched for model choice.
"""
leakage_text = """# Leakage Summary

`starter_minutes = current_is_starter * numMinutes`. Because `numMinutes` is actual current-game minutes, direct `starter_minutes` is same-game information and is excluded from model comparisons.

Valid alternatives are shifted historical features such as `starter_minutes_roll_3`, `starter_minutes_roll_10`, and `starter_minutes_share_roll_10`.
"""
recommendation_text = f"""# Final Model Recommendation

Use `{best_pre_lineup['feature_set']}` as the main pre-lineup benchmark.

If confirmed starting lineup is available, `{best_lineup_aware['feature_set']}` is the strongest valid lineup-aware benchmark.

Keep the tuned compact scoring HistGradientBoosting model as the strongest compact explanatory candidate.

Do not touch the 2025 test split until the model choice is frozen.

Best lineup-aware validation MAE: {best_lineup_aware['mae']:.4f}.

Best pre-lineup validation MAE: {best_pre_lineup['mae']:.4f}.

Tuned compact validation MAE: {best_compact['mae']:.4f}.

Best Isolation Forest anomaly MAE: {best_isolation['validation_mae']:.4f}.
"""

(STEP_SUMMARIES_DIR / "end_to_end_summary.md").write_text(summary_text, encoding="utf-8")
(STEP_SUMMARIES_DIR / "leakage_summary.md").write_text(leakage_text, encoding="utf-8")
(STEP_SUMMARIES_DIR / "final_model_recommendation.md").write_text(recommendation_text, encoding="utf-8")
print(summary_text)


,model_label,feature_set,n_features,validity,valid_pre_game,uses_current_game_minutes,mae,rmse,r2,notes
0,Best Isolation Forest anomaly model,lineup_aware_starter_flag_roll10_best_pca_plus...,44,lineup-aware,False,False,7.579672,9.694711,0.577301,Isolation Forest anomaly features; variant lin...
1,Best valid starter PCA model,lineup_aware_starter_flag_roll10_plus_pca_30,41,lineup-aware,False,False,7.581493,9.694047,0.577359,Starter model plus 30 PCA components.
2,Best valid starter-focused model,lineup_aware_starter_flag_roll10,11,lineup-aware,False,False,7.655891,9.782516,0.569610,Starter model family included as a main compar...
3,Lineup-aware starter-flag HGB,lineup_aware_starter_flag_roll10,11,lineup-aware,False,False,7.655891,9.782516,0.569610,Uses current_is_starter plus starter_minutes_r...
4,Broad valid HGB reference,pre_lineup_player_team_opp,486,pre-lineup,True,False,7.731509,9.900762,0.559142,Broad benchmark with player/team/opponent shif...
5,Best valid PCA compact model,compact_scoring_plus_pca_30,75,pre-lineup,True,False,7.747985,9.907535,0.558539,30 PCA components from unused broad features.
6,Best valid tuned compact model,pre_lineup_compact_40_plus_scoring,45,pre-lineup,True,False,7.769961,9.933005,0.556266,Tuned HGB config: no_early_stop_more_trees
7,Compact scoring before tuning,pre_lineup_compact_40_plus_scoring,45,pre-lineup,True,False,7.781769,9.947805,0.554943,Compact feature-refinement candidate before tu...
8,Best valid 11-feature replication,replication_11_safe_recent_starter,11,pre-lineup,True,False,7.817989,9.975005,0.552506,Uses shifted starter-minutes history.
9,Pre-lineup starter-history HGB,pre_lineup_starter_history_roll10,10,pre-lineup,True,False,7.909683,10.087826,0.542326,Uses shifted starter history including starter...


# End-To-End Summary

This final run uses NBA seasons `2021` through `2025`.

Split policy:

- Train: full seasons `2021`, `2022`, `2023`
- Validation: full season `2024`
- Test: full season `2025`

Lowest validation MAE among valid lineup-aware or pre-lineup models: `Best Isolation Forest anomaly model` / `lineup_aware_starter_flag_roll10_best_pca_plus_isolation`.

Best lineup-aware validation MAE: 7.5797 from `lineup_aware_starter_flag_roll10_best_pca_plus_isolation`.

Best pre-lineup validation MAE: 7.7315 from `pre_lineup_player_team_opp`.

Best compact explainable candidate: `pre_lineup_compact_40_plus_scoring` with tuned HGB, MAE 7.7700.

Best Isolation Forest anomaly model: `lineup_aware_starter_flag_roll10_best_pca_plus_isolation` with MAE 7.5797.

The 2025 test split remains untouched for model choice.



## 17b. Integrated Final Validation Ranking And One-Time 2025 Test Evaluation

After all validation-only comparisons are complete, this section builds the final valid ranking and evaluates the selected controlled model once on the held-out 2025 test season.


In [31]:

controlled_for_ranking = combined_controlled_model_comparison.rename(
    columns={"validation_mae": "mae", "validation_rmse": "rmse", "validation_r2": "r2"}
).copy()
controlled_for_ranking["model_label"] = np.where(
    controlled_for_ranking["uses_time_decay"],
    "Controlled model with time-decay weights",
    "Controlled model",
)
controlled_for_ranking["valid_pre_game"] = controlled_for_ranking["validity"].eq("pre-lineup")
controlled_for_ranking["uses_current_game_minutes"] = False
controlled_for_ranking["notes"] = controlled_for_ranking["model"] + np.where(controlled_for_ranking["uses_time_decay"], " with train-only time-decay sample weights.", " without time-decay sample weights.")
controlled_for_ranking = controlled_for_ranking[["model_label", "feature_set", "model", "n_features", "validity", "valid_pre_game", "uses_current_game_minutes", "mae", "rmse", "r2", "notes"]]

existing_rank = final_comparison.copy()
existing_rank["model"] = existing_rank["model_label"]
existing_rank = existing_rank[["model_label", "feature_set", "model", "n_features", "validity", "valid_pre_game", "uses_current_game_minutes", "mae", "rmse", "r2", "notes"]]

final_validation_ranking = pd.concat([existing_rank, controlled_for_ranking], ignore_index=True)
final_validation_ranking = final_validation_ranking[final_validation_ranking["uses_current_game_minutes"].eq(False)]
final_validation_ranking = final_validation_ranking.sort_values(["mae", "rmse"]).reset_index(drop=True)
final_validation_ranking.to_csv(TABLES_DIR / "final_validation_ranking.csv", index=False)

selected_controlled = combined_controlled_model_comparison.sort_values(["validation_mae", "validation_rmse"]).iloc[0]
selected_feature_set = selected_controlled["feature_set"]
selected_model_name = selected_controlled["model"]
selected_uses_time_decay = bool(selected_controlled["uses_time_decay"])
selected_features = loaded.feature_sets[selected_feature_set]

development = loaded.data[loaded.data[SPLIT_COL].isin(["train", "validation"])].copy()
test = loaded.data[loaded.data[SPLIT_COL].eq("test")].copy()
if development.empty or test.empty:
    raise ValueError("Development and test splits must both be non-empty for final evaluation.")
if selected_model_name in BASELINE_MODELS:
    test_predictions = _baseline_predictions(selected_model_name, development[TARGET_COL], test)
    fitted_final_model = None
else:
    x_development = _validate_feature_frame(development, selected_features)
    x_test = _validate_feature_frame(test, selected_features)
    final_weights = time_decay_weights(development) if selected_uses_time_decay else None
    fitted_final_model, _, test_predictions = fit_predict_controlled_model(selected_model_name, x_development, development[TARGET_COL], x_test, sample_weight=final_weights)

test_metrics = regression_metrics(test[TARGET_COL], test_predictions)
final_test_evaluation = pd.DataFrame(
    [
        {
            "selected_by": "lowest_2024_validation_mae_from_controlled_valid_models",
            "feature_set": selected_feature_set,
            "model": selected_model_name,
            "uses_time_decay": selected_uses_time_decay,
            "n_features": len(selected_features),
            "development_rows_2021_2024": len(development),
            "test_rows_2025": len(test),
            "test_mae": test_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
            "test_r2": test_metrics["r2"],
        }
    ]
)
final_test_evaluation.to_csv(TABLES_DIR / "final_test_evaluation.csv", index=False)

final_test_predictions = test[["gameId", "personId", "game_date", "season_start", "player_name", "playerteamName", "opponentteamName", TARGET_COL]].copy()
final_test_predictions["feature_set"] = selected_feature_set
final_test_predictions["model"] = selected_model_name
final_test_predictions["prediction"] = test_predictions
final_test_predictions["error"] = final_test_predictions[TARGET_COL] - final_test_predictions["prediction"]
final_test_predictions["absolute_error"] = final_test_predictions["error"].abs()
final_test_predictions.to_csv(TABLES_DIR / "final_test_predictions.csv", index=False)

importance_rows = []
if fitted_final_model is not None:
    estimator = fitted_final_model.named_steps["model"]
    if hasattr(estimator, "feature_importances_"):
        importances = estimator.feature_importances_
        importance_rows = [{"feature": feature, "importance": float(value), "importance_type": "model_feature_importance"} for feature, value in zip(selected_features, importances)]
    elif hasattr(estimator, "coef_"):
        coefs = np.ravel(estimator.coef_)
        importance_rows = [{"feature": feature, "importance": float(abs(value)), "signed_coefficient": float(value), "importance_type": "absolute_coefficient"} for feature, value in zip(selected_features, coefs)]

final_feature_importance = pd.DataFrame(importance_rows).sort_values("importance", ascending=False) if importance_rows else pd.DataFrame(columns=["feature", "importance", "importance_type"])
final_feature_importance.to_csv(TABLES_DIR / "final_feature_importance.csv", index=False)
if not final_feature_importance.empty:
    plt.figure(figsize=(9, 6))
    top_imp = final_feature_importance.head(20).iloc[::-1]
    plt.barh(top_imp["feature"], top_imp["importance"])
    plt.xlabel("Importance")
    plt.title("Final Selected Model Feature Importance")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "final_feature_importance.png", dpi=160)
    plt.close()

plt.figure(figsize=(9, 5))
leak_plot = pd.concat(
    [
        leakage_diagnostic_actual_minutes[["model_family", "reported_mae"]].rename(columns={"model_family": "label", "reported_mae": "mae"}).assign(result_type="leakage diagnostic"),
        final_validation_ranking.head(5)[["model_label", "mae"]].rename(columns={"model_label": "label"}).assign(result_type="valid validation"),
    ],
    ignore_index=True,
)
leak_plot["short_label"] = leak_plot["label"].str.slice(0, 34)
colors = leak_plot["result_type"].map({"leakage diagnostic": "#d95f02", "valid validation": "#1b9e77"})
plt.barh(leak_plot["short_label"], leak_plot["mae"], color=colors)
plt.gca().invert_yaxis()
plt.xlabel("MAE")
plt.title("Leakage Diagnostic Results vs Valid Validation Results")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "leakage_vs_valid_mae.png", dpi=160)
plt.close()

display(final_validation_ranking.head(15))
display(final_test_evaluation)


,model_label,feature_set,model,n_features,validity,valid_pre_game,uses_current_game_minutes,mae,rmse,r2,notes
0,Best Isolation Forest anomaly model,lineup_aware_starter_flag_roll10_best_pca_plus...,Best Isolation Forest anomaly model,44,lineup-aware,False,False,7.579672,9.694711,0.577301,Isolation Forest anomaly features; variant lin...
1,Best valid starter PCA model,lineup_aware_starter_flag_roll10_plus_pca_30,Best valid starter PCA model,41,lineup-aware,False,False,7.581493,9.694047,0.577359,Starter model plus 30 PCA components.
2,Controlled model,lineup_aware_starter_flag_roll10,catboost,11,lineup-aware,False,False,7.643728,9.779999,0.569831,catboost without time-decay sample weights.
3,Best valid starter-focused model,lineup_aware_starter_flag_roll10,Best valid starter-focused model,11,lineup-aware,False,False,7.655891,9.782516,0.569610,Starter model family included as a main compar...
4,Lineup-aware starter-flag HGB,lineup_aware_starter_flag_roll10,Lineup-aware starter-flag HGB,11,lineup-aware,False,False,7.655891,9.782516,0.569610,Uses current_is_starter plus starter_minutes_r...
5,Controlled model,lineup_aware_starter_flag_roll10,hist_gradient_boosting,11,lineup-aware,False,False,7.655891,9.782516,0.569610,hist_gradient_boosting without time-decay samp...
6,Controlled model with time-decay weights,lineup_aware_starter_flag_roll10,catboost,11,lineup-aware,False,False,7.660687,9.794362,0.568567,catboost with train-only time-decay sample wei...
7,Controlled model with time-decay weights,lineup_aware_starter_flag_roll10,hist_gradient_boosting,11,lineup-aware,False,False,7.661765,9.785613,0.569337,hist_gradient_boosting with train-only time-de...
8,Controlled model,lineup_aware_starter_flag_roll10,lightgbm,11,lineup-aware,False,False,7.667858,9.811787,0.567030,lightgbm without time-decay sample weights.
9,Controlled model with time-decay weights,lineup_aware_starter_flag_roll10,extra_trees,11,lineup-aware,False,False,7.672109,9.800190,0.568053,extra_trees with train-only time-decay sample ...


,selected_by,feature_set,model,uses_time_decay,n_features,development_rows_2021_2024,test_rows_2025,test_mae,test_rmse,test_r2
0,lowest_2024_validation_mae_from_controlled_val...,lineup_aware_starter_flag_roll10,catboost,False,11,103099,26651,7.671208,9.790935,0.54401


## 16. Feature Importance And Interpretability

This section keeps the profile/extended-driver experiment and permutation importance as interpretability tools, not as the primary model-selection step.


In [32]:
PROFILE_EXTENDED_FEATURE_SET = "profile_extended_driver_10"
DIRECT_PLAYER_FP_SOURCES = {"fantasy_points", "points", "reboundsTotal", "assists", "steals", "blocks", "turnovers"}
TARGET_DERIVED_CONTEXT_PATTERNS = ["fantasy_points", "fp_allowed", "pos_opp_difficulty", "league_pos_fp_allowed"]
PROFILE_COLUMNS_FOR_EXPERIMENT = [
    "personId",
    "birthDate",
    "heightInches",
    "bodyWeightLbs",
    "guard",
    "forward",
    "center",
    "draftYear",
    "draftRound",
    "draftNumber",
]


def _experiment_normalize_id(series: pd.Series) -> pd.Series:
    # Normalize ids in this standalone notebook section before merges.
    numeric = pd.to_numeric(series, errors="coerce")
    normalized = numeric.astype("Int64").astype("string")
    fallback = series.astype("string").str.strip()
    return normalized.fillna(fallback)


def _experiment_add_game_date_and_season(df: pd.DataFrame, date_col: str = "gameDateTimeEst") -> pd.DataFrame:
    # Add parsed game dates and NBA season starts for the profile experiment.
    result = df.copy()
    result["game_date"] = pd.to_datetime(result[date_col], errors="coerce")
    result["season_start"] = np.where(result["game_date"].dt.month >= 7, result["game_date"].dt.year, result["game_date"].dt.year - 1)
    result["season_start"] = result["season_start"].astype("Int64")
    return result


def _load_experiment_player_profile(raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    # Load static profile fields and create age/draft/size features.
    players = pd.read_csv(raw_dir / "Players.csv", usecols=PROFILE_COLUMNS_FOR_EXPERIMENT, low_memory=False)
    players["personId"] = _experiment_normalize_id(players["personId"])
    players["birthDate"] = pd.to_datetime(players["birthDate"], errors="coerce")

    for column in ["heightInches", "bodyWeightLbs", "draftYear", "draftRound", "draftNumber", "guard", "forward", "center"]:
        if column in players.columns:
            players[column] = pd.to_numeric(players[column], errors="coerce")

    profile = players.rename(
        columns={
            "heightInches": "profile_height_inches",
            "bodyWeightLbs": "profile_weight_lbs",
            "draftYear": "profile_draft_year",
            "draftRound": "profile_draft_round",
            "draftNumber": "profile_draft_number",
        }
    )
    profile["profile_birthdate_missing"] = profile["birthDate"].isna().astype(int)
    profile["profile_height_missing"] = profile["profile_height_inches"].isna().astype(int)
    profile["profile_weight_missing"] = profile["profile_weight_lbs"].isna().astype(int)
    profile["profile_draft_missing"] = profile[["profile_draft_year", "profile_draft_round", "profile_draft_number"]].isna().all(axis=1).astype(int)
    profile["profile_draft_round_missing"] = profile["profile_draft_round"].isna().astype(int)
    profile["profile_draft_number_missing"] = profile["profile_draft_number"].isna().astype(int)
    profile["profile_bmi"] = 703 * profile["profile_weight_lbs"] / (profile["profile_height_inches"] ** 2)
    return profile


def _is_direct_player_fp_roll(feature: str) -> bool:
    # Return whether a feature is a player fantasy-points shortcut that this experiment excludes.
    return any(feature.startswith(f"{source}_") for source in DIRECT_PLAYER_FP_SOURCES)


def _is_target_derived_context(feature: str) -> bool:
    # Return whether a feature is derived from target components that this experiment excludes.
    lowered = feature.lower()
    return any(pattern in lowered for pattern in TARGET_DERIVED_CONTEXT_PATTERNS)


def _is_allowed_existing_context_feature(feature: str) -> bool:
    # Keep static/profile fields and rolling-10 non-target context; exclude FP shortcuts.
    static_keep = {
        "home",
        "guard",
        "forward",
        "center",
        "days_since_last_game",
        "is_back_to_back",
        "current_is_starter_roll_10",
        "starter_minutes_roll_10",
        "bench_minutes_roll_10",
        "starter_minutes_share_roll_10",
        "numMinutes_roll_10",
    }
    if feature in static_keep:
        return True
    if feature.startswith("profile_") or feature == "player_age_years":
        return True
    if not feature.endswith("_roll_10"):
        return False
    if _is_direct_player_fp_roll(feature):
        return False
    if _is_target_derived_context(feature):
        return False
    return True


def _load_extended_player_numeric_rolls(raw_dir: Path = RAW_DIR) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Build shifted rolling-10 features from numeric PlayerStatisticsExtended columns.
    extended = pd.read_csv(raw_dir / "PlayerStatisticsExtended.csv", low_memory=False)
    extended = _experiment_add_game_date_and_season(extended)
    extended = extended[extended["gameType"].eq("Regular Season") & extended["season_start"].isin(LAST4_SEASON_STARTS)].copy()
    for column in ["gameId", "personId"]:
        extended[column] = _experiment_normalize_id(extended[column])

    excluded_metadata = {
        "firstName", "lastName", "gameId", "personId", "gameDateTimeEst", "gameType", "gameLabel", "gameSubLabel",
        "seriesGameNumber", "comment", "startingPosition", "playerteamId", "playerteamCity", "playerteamName",
        "opponentteamId", "opponentteamCity", "opponentteamName", "game_date", "season_start",
    }
    numeric_sources = []
    decision_rows = []
    for column in extended.columns:
        if column in excluded_metadata:
            decision_rows.append({"raw_column": column, "kept": False, "drop_reason": "identifier/date/text metadata"})
            continue
        if column in DIRECT_PLAYER_FP_SOURCES:
            decision_rows.append({"raw_column": column, "kept": False, "drop_reason": "direct player fantasy-points target/component excluded"})
            continue
        numeric = pd.to_numeric(extended[column], errors="coerce")
        if numeric.notna().sum() == 0:
            decision_rows.append({"raw_column": column, "kept": False, "drop_reason": "not numeric after coercion"})
            continue
        if numeric.nunique(dropna=True) <= 1:
            decision_rows.append({"raw_column": column, "kept": False, "drop_reason": "constant or nearly empty numeric source"})
            continue
        extended[column] = numeric
        numeric_sources.append(column)
        decision_rows.append({"raw_column": column, "kept": True, "drop_reason": "kept as shifted rolling-10 extended player stat"})

    roll_input = extended[["gameId", "personId", "game_date"] + numeric_sources].sort_values(["personId", "game_date", "gameId"]).copy()
    for source_col in numeric_sources:
        shifted = roll_input.groupby("personId", observed=True)[source_col].shift(1)
        roll_input[f"{source_col}_roll_10"] = shifted.groupby(roll_input["personId"], observed=True).transform(
            lambda values: values.rolling(window=10, min_periods=1).mean()
        )

    roll_cols = [f"{source_col}_roll_10" for source_col in numeric_sources]
    return roll_input[["gameId", "personId"] + roll_cols], pd.DataFrame(decision_rows)


def build_profile_extended_driver_10_table(
    modeling_path: Path = PROCESSED_DIR / "modeling_table_last4_regular.csv",
    split_path: Path = TABLES_DIR / "modeling_rows_with_split_v1.csv",
) -> tuple[pd.DataFrame, list[str], pd.DataFrame, pd.DataFrame]:
    # Create the no-player-FP-components rolling-10 interpretability table and feature plan.
    modeling = pd.read_csv(modeling_path, low_memory=False)
    for column in ["gameId", "personId"]:
        modeling[column] = _experiment_normalize_id(modeling[column])
    modeling["game_date"] = pd.to_datetime(modeling["game_date"], errors="coerce")

    splits = pd.read_csv(split_path, usecols=["gameId", "personId", SPLIT_COL], low_memory=False)
    for column in ["gameId", "personId"]:
        splits[column] = _experiment_normalize_id(splits[column])
    if SPLIT_COL not in modeling.columns:
        modeling = modeling.merge(splits, on=["gameId", "personId"], how="left", validate="one_to_one")

    profile = _load_experiment_player_profile()
    table = modeling.merge(profile, on="personId", how="left", validate="many_to_one")
    table["player_age_years"] = ((table["game_date"] - table["birthDate"]).dt.days / 365.25).where(table["birthDate"].notna())
    table["profile_years_since_draft"] = table["game_date"].dt.year - table["profile_draft_year"]
    table.loc[table["profile_years_since_draft"] < 0, "profile_years_since_draft"] = np.nan

    extended_rolls, extended_source_decisions = _load_extended_player_numeric_rolls()
    rename_map = {}
    feature_rows = []
    existing_columns = set(table.columns)
    merge_cols = []
    for column in [c for c in extended_rolls.columns if c.endswith("_roll_10")]:
        if column in existing_columns:
            feature_rows.append(
                {
                    "feature": column,
                    "source": "existing_modeling_table",
                    "kept": _is_allowed_existing_context_feature(column),
                    "drop_reason": "duplicate rolling feature already existed; reused existing column" if _is_allowed_existing_context_feature(column) else "excluded by no-player-FP/target-derived-context policy",
                }
            )
            continue
        new_name = f"ext_{column}"
        rename_map[column] = new_name
        merge_cols.append(column)

    if merge_cols:
        table = table.merge(
            extended_rolls[["gameId", "personId"] + merge_cols].rename(columns=rename_map),
            on=["gameId", "personId"],
            how="left",
            validate="one_to_one",
        )

    static_profile_features = [
        "player_age_years",
        "profile_birthdate_missing",
        "profile_height_inches",
        "profile_weight_lbs",
        "profile_height_missing",
        "profile_weight_missing",
        "profile_bmi",
        "profile_draft_year",
        "profile_draft_round",
        "profile_draft_number",
        "profile_years_since_draft",
        "profile_draft_missing",
        "profile_draft_round_missing",
        "profile_draft_number_missing",
        "guard",
        "forward",
        "center",
    ]
    context_features = [column for column in table.columns if _is_allowed_existing_context_feature(column)]
    ext_features = [rename_map[column] for column in merge_cols if _is_allowed_existing_context_feature(rename_map[column])]

    candidate_features = list(dict.fromkeys(static_profile_features + context_features + ext_features))
    candidate_features = [feature for feature in candidate_features if feature in table.columns]

    final_features = []
    seen_value_signatures = {}
    for feature in candidate_features:
        if feature in final_features:
            continue
        if feature == TARGET_COL or feature in ["gameId", "personId", "game_date"]:
            continue
        series = pd.to_numeric(table[feature], errors="coerce")
        if series.notna().sum() == 0:
            feature_rows.append({"feature": feature, "source": "candidate_feature", "kept": False, "drop_reason": "all missing after numeric coercion"})
            continue
        # Drop exact duplicate numeric columns. This keeps the feature plan explainable without manual duplicate cleanup.
        signature = tuple(pd.util.hash_pandas_object(series.fillna(-999999999), index=False).head(50000).tolist())
        if signature in seen_value_signatures:
            feature_rows.append(
                {
                    "feature": feature,
                    "source": "candidate_feature",
                    "kept": False,
                    "drop_reason": f"duplicate values of {seen_value_signatures[signature]}",
                }
            )
            continue
        seen_value_signatures[signature] = feature
        final_features.append(feature)
        feature_rows.append({"feature": feature, "source": "profile/extended/context", "kept": True, "drop_reason": "kept for profile_extended_driver_10"})

    feature_plan = pd.DataFrame(feature_rows).drop_duplicates(subset=["feature", "source"], keep="last")
    meta_cols = [
        "gameId", "personId", "game_date", "season_start", "player_name", "playerteamName", "opponentteamName",
        "current_is_starter", "guard", "forward", "center", TARGET_COL, SPLIT_COL,
    ]
    keep_cols = [column for column in dict.fromkeys(meta_cols + final_features) if column in table.columns]
    experiment_table = table[keep_cols].copy()
    return experiment_table, final_features, feature_plan, extended_source_decisions


def run_profile_extended_driver_10_experiment() -> dict[str, pd.DataFrame]:
    # Run the profile/extended-driver experiment and write its tables, predictions, and summary.
    experiment_table, features, feature_plan, source_decisions = build_profile_extended_driver_10_table()
    table_path = PROCESSED_DIR / "profile_extended_driver_10_modeling_table.csv"
    experiment_table.to_csv(table_path, index=False)

    feature_plan.to_csv(TABLES_DIR / "profile_extended_driver_10_feature_plan.csv", index=False)
    source_decisions.to_csv(TABLES_DIR / "profile_extended_driver_10_extended_source_decisions.csv", index=False)
    pd.DataFrame({"feature_set": PROFILE_EXTENDED_FEATURE_SET, "position": range(1, len(features) + 1), "feature": features}).to_csv(
        TABLES_DIR / "profile_extended_driver_10_feature_membership.csv", index=False
    )

    data = add_validation_segments(experiment_table.copy())
    result_rows = []
    prediction_tables = []
    model_names = ["mean_baseline", "ridge", "random_forest", "hist_gradient_boosting"]
    for model_name in model_names:
        print(f"Running {PROFILE_EXTENDED_FEATURE_SET} / {model_name}", flush=True)
        row, preds = run_one_experiment(data, PROFILE_EXTENDED_FEATURE_SET, features, model_name)
        result_rows.append(row)
        prediction_tables.append(preds)

    results = pd.DataFrame(result_rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
    predictions = pd.concat(prediction_tables, ignore_index=True)
    results.to_csv(TABLES_DIR / "profile_extended_driver_10_model_results.csv", index=False)
    predictions.to_csv(TABLES_DIR / "profile_extended_driver_10_predictions.csv", index=False)
    error_by_segment(predictions).to_csv(TABLES_DIR / "profile_extended_driver_10_error_by_segment.csv", index=False)

    previous_comparison_path = TABLES_DIR / "final_model_comparison.csv"
    if previous_comparison_path.exists():
        previous = pd.read_csv(previous_comparison_path)
        comparison_rows = previous[["model_label", "feature_set", "n_features", "valid_pre_game", "mae", "rmse", "r2", "notes"]].copy()
        best_new = results.sort_values("mae").iloc[0]
        comparison_rows = pd.concat(
            [
                comparison_rows,
                pd.DataFrame(
                    [
                        {
                            "model_label": "Profile extended driver 10 interpretability",
                            "feature_set": PROFILE_EXTENDED_FEATURE_SET,
                            "n_features": int(best_new["n_features"]),
                            "valid_pre_game": True,
                            "mae": best_new["mae"],
                            "rmse": best_new["rmse"],
                            "r2": best_new["r2"],
                            "notes": f"Best model in this experiment: {best_new['model']}; excludes player FP/component rolling",
                        }
                    ]
                ),
            ],
            ignore_index=True,
        ).sort_values("mae")
        comparison_rows.to_csv(TABLES_DIR / "profile_extended_driver_10_comparison_to_prior_models.csv", index=False)

    summary = f"""# Profile Extended Driver 10 Summary

Purpose: interpret non-FP drivers of future fantasy points, not chase the best validation MAE.

Feature set: `{PROFILE_EXTENDED_FEATURE_SET}`

Rows: {len(experiment_table):,}

Features kept: {len(features):,}

Rules applied:

- One rolling window only: 10 games.
- Player history is shifted by one game before rolling.
- Excluded player-level rolling fantasy points and direct FP components: {', '.join(sorted(DIRECT_PLAYER_FP_SOURCES))}.
- Added profile fields: age at game date, height, weight, BMI, draft year, draft round, draft number, years since draft, and missingness flags.
- Kept extended player-stat rolling features and valid team/opponent/role context.

Best model in this interpretability experiment: `{results.iloc[0]['model']}`

Validation MAE: {results.iloc[0]['mae']:.4f}

Interpretation note: if this model trails the FP-rolling models, that is expected. The next analytical step is to inspect Ridge/RF importance and permutation importance to identify which non-target drivers carry the most signal.
"""
    (STEP_SUMMARIES_DIR / "profile_extended_driver_10_summary.md").write_text(summary, encoding="utf-8")
    print(summary)
    display(results)
    display(feature_plan[feature_plan["kept"]].head(40))
    return {
        "profile_extended_driver_10_table": experiment_table,
        "profile_extended_driver_10_features": pd.DataFrame({"feature": features}),
        "profile_extended_driver_10_feature_plan": feature_plan,
        "profile_extended_driver_10_model_results": results,
        "profile_extended_driver_10_predictions": predictions,
    }


In [33]:
profile_extended_outputs = run_profile_extended_driver_10_experiment()


Running profile_extended_driver_10 / mean_baseline


Running profile_extended_driver_10 / ridge


Running profile_extended_driver_10 / random_forest


Running profile_extended_driver_10 / hist_gradient_boosting


# Profile Extended Driver 10 Summary

Purpose: interpret non-FP drivers of future fantasy points, not chase the best validation MAE.

Feature set: `profile_extended_driver_10`

Rows: 129,750

Features kept: 144

Rules applied:

- One rolling window only: 10 games.
- Player history is shifted by one game before rolling.
- Excluded player-level rolling fantasy points and direct FP components: assists, blocks, fantasy_points, points, reboundsTotal, steals, turnovers.
- Added profile fields: age at game date, height, weight, BMI, draft year, draft round, draft number, years since draft, and missingness flags.
- Kept extended player-stat rolling features and valid team/opponent/role context.

Best model in this interpretability experiment: `hist_gradient_boosting`

Validation MAE: 7.8685

Interpretation note: if this model trails the FP-rolling models, that is expected. The next analytical step is to inspect Ridge/RF importance and permutation importance to identify which non-target drivers

,feature_set,model,n_features,train_rows,validation_rows,fit_predict_seconds,mae,rmse,r2,mean_error,median_absolute_error
0,profile_extended_driver_10,hist_gradient_boosting,144,76937,26162,14.587,7.868489,10.075367,0.543456,0.373838,6.440129
1,profile_extended_driver_10,ridge,144,76937,26162,4.318,7.892910,10.068858,0.544045,0.006562,6.559285
2,profile_extended_driver_10,random_forest,144,76937,26162,159.582,7.940417,10.130653,0.538432,0.207015,6.522601
3,profile_extended_driver_10,mean_baseline,144,76937,26162,0.001,12.073193,14.913770,-0.000313,0.263889,10.758609


,feature,source,kept,drop_reason
0,numMinutes_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...
1,fieldGoalsPercentage_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...
2,threePointersPercentage_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...
3,freeThrowsPercentage_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...
4,offensiveRating_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...
5,estimatedDefensiveRating_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...
6,defensiveRating_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...
7,effectiveFieldGoalPercentage_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...
8,trueShootingPercentage_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...
9,usagePercentage_roll_10,existing_modeling_table,True,duplicate rolling feature already existed; reu...


In [34]:
def run_profile_extended_driver_10_importance(
    experiment_table: pd.DataFrame | None = None,
    features: list[str] | None = None,
    max_validation_rows: int = 5000,
) -> dict[str, pd.DataFrame]:
    # Compute Ridge, random-forest, and permutation importance for the profile/extended-driver experiment.
    if experiment_table is None:
        experiment_table = pd.read_csv(PROCESSED_DIR / "profile_extended_driver_10_modeling_table.csv", low_memory=False)
    if features is None:
        features = pd.read_csv(TABLES_DIR / "profile_extended_driver_10_feature_membership.csv").sort_values("position")["feature"].tolist()

    data = add_validation_segments(experiment_table.copy())
    train = data[data[SPLIT_COL].eq("train")].copy()
    validation = data[data[SPLIT_COL].eq("validation")].copy()
    x_train = _validate_feature_frame(train, features)
    y_train = train[TARGET_COL]
    x_validation = _validate_feature_frame(validation, features)
    y_validation = validation[TARGET_COL]

    ridge = make_model("ridge")
    ridge.fit(x_train, y_train)
    ridge_coef = ridge.named_steps["model"].coef_
    ridge_importance = pd.DataFrame(
        {
            "feature": features,
            "importance_type": "ridge_abs_scaled_coef",
            "importance": np.abs(ridge_coef),
            "signed_value": ridge_coef,
        }
    ).sort_values("importance", ascending=False)
    ridge_importance.to_csv(TABLES_DIR / "profile_extended_driver_10_ridge_importance.csv", index=False)

    rf = make_model("random_forest")
    rf.fit(x_train, y_train)
    rf_importance = pd.DataFrame(
        {
            "feature": features,
            "importance_type": "random_forest_impurity",
            "importance": rf.named_steps["model"].feature_importances_,
        }
    ).sort_values("importance", ascending=False)
    rf_importance.to_csv(TABLES_DIR / "profile_extended_driver_10_rf_importance.csv", index=False)

    hgb = make_model("hist_gradient_boosting")
    hgb.fit(x_train, y_train)
    sample_n = min(max_validation_rows, len(x_validation))
    x_perm = x_validation.sample(sample_n, random_state=MODEL_RANDOM_STATE)
    y_perm = y_validation.loc[x_perm.index]
    perm = permutation_importance(
        hgb,
        x_perm,
        y_perm,
        scoring="neg_mean_absolute_error",
        n_repeats=3,
        random_state=MODEL_RANDOM_STATE,
        n_jobs=1,
    )
    permutation = pd.DataFrame(
        {
            "feature": features,
            "importance_type": "hgb_permutation_validation_mae_increase",
            "importance": perm.importances_mean,
            "importance_std": perm.importances_std,
            "validation_rows_sampled": sample_n,
        }
    ).sort_values("importance", ascending=False)
    permutation.to_csv(TABLES_DIR / "profile_extended_driver_10_permutation_importance.csv", index=False)

    combined = pd.concat(
        [
            ridge_importance.assign(rank=ridge_importance["importance"].rank(ascending=False, method="first")),
            rf_importance.assign(rank=rf_importance["importance"].rank(ascending=False, method="first")),
            permutation.assign(rank=permutation["importance"].rank(ascending=False, method="first")),
        ],
        ignore_index=True,
        sort=False,
    )
    combined.to_csv(TABLES_DIR / "profile_extended_driver_10_feature_importance_combined.csv", index=False)

    top_perm = permutation.head(25).sort_values("importance")
    plt.figure(figsize=(9, 8))
    plt.barh(top_perm["feature"], top_perm["importance"])
    plt.xlabel("Validation MAE increase when shuffled")
    plt.title("Profile Extended Driver 10: Top Permutation Importance")
    _savefig(FIGURES_DIR / "profile_extended_driver_10_permutation_importance_top25.png")

    summary = f"""# Profile Extended Driver 10 Importance Summary

Permutation importance was computed on {sample_n:,} validation rows with 3 repeats.

Top 10 permutation features:

{permutation.head(10)[['feature', 'importance', 'importance_std']].to_string(index=False)}

Use permutation importance as the main interpretation table because it measures validation MAE impact directly. Ridge and random forest importance are supporting views.
"""
    (STEP_SUMMARIES_DIR / "profile_extended_driver_10_importance_summary.md").write_text(summary, encoding="utf-8")
    print(summary)
    display(permutation.head(25))
    return {
        "ridge_importance": ridge_importance,
        "rf_importance": rf_importance,
        "permutation_importance": permutation,
        "combined_importance": combined,
    }


In [35]:
profile_extended_importance_outputs = run_profile_extended_driver_10_importance(
    experiment_table=profile_extended_outputs["profile_extended_driver_10_table"],
    features=profile_extended_outputs["profile_extended_driver_10_features"]["feature"].tolist(),
)


# Profile Extended Driver 10 Importance Summary

Permutation importance was computed on 5,000 validation rows with 3 repeats.

Top 10 permutation features:

                        feature  importance  importance_std
     ext_fieldGoalsMade_roll_10    1.071791        0.049314
        ext_possessions_roll_10    0.460222        0.024145
             numMinutes_roll_10    0.220860        0.006194
ext_fieldGoalsAttempted_roll_10    0.106316        0.013820
  ext_reboundsDefensive_roll_10    0.103037        0.009454
           days_since_last_game    0.085099        0.008783
       ext_doubleDouble_roll_10    0.038262        0.000582
   playerImpactEstimate_roll_10    0.036424        0.003627
   ext_assistPercentage_roll_10    0.031612        0.007376
       ext_foulsAgainst_roll_10    0.031159        0.001054

Use permutation importance as the main interpretation table because it measures validation MAE impact directly. Ridge and random forest importance are supporting views.



,feature,importance_type,importance,importance_std,validation_rows_sampled
74,ext_fieldGoalsMade_roll_10,hgb_permutation_validation_mae_increase,1.071791,0.049314,5000
103,ext_possessions_roll_10,hgb_permutation_validation_mae_increase,0.460222,0.024145,5000
17,numMinutes_roll_10,hgb_permutation_validation_mae_increase,0.220860,0.006194,5000
75,ext_fieldGoalsAttempted_roll_10,hgb_permutation_validation_mae_increase,0.106316,0.013820,5000
73,ext_reboundsDefensive_roll_10,hgb_permutation_validation_mae_increase,0.103037,0.009454,5000
12,days_since_last_game,hgb_permutation_validation_mae_increase,0.085099,0.008783,5000
84,ext_doubleDouble_roll_10,hgb_permutation_validation_mae_increase,0.038262,0.000582,5000
24,playerImpactEstimate_roll_10,hgb_permutation_validation_mae_increase,0.036424,0.003627,5000
91,ext_assistPercentage_roll_10,hgb_permutation_validation_mae_increase,0.031612,0.007376,5000
82,ext_foulsAgainst_roll_10,hgb_permutation_validation_mae_increase,0.031159,0.001054,5000


## 17. Artifact Inventory And Final Report Links

This final section writes the classmate-readable project summary and an inventory of notebook-generated outputs.


In [36]:
def _file_inventory(folder: Path, pattern: str) -> pd.DataFrame:
    # List generated files in one report folder with relative paths and file sizes.
    rows = []
    for path in sorted(folder.glob(pattern)):
        if path.is_file():
            rows.append({"path": str(path.relative_to(RUN_DIR)).replace("\\", "/"), "bytes": path.stat().st_size})
    return pd.DataFrame(rows)


def _markdown_table(df: pd.DataFrame) -> str:
    # Render a small Markdown table without requiring the optional tabulate package.
    if df.empty:
        return "_No rows._"
    text = df.copy()
    for column in text.columns:
        text[column] = text[column].map(lambda value: "" if pd.isna(value) else str(value).replace("|", "\\|"))
    columns = list(text.columns)
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in text[columns].to_numpy(dtype=str)]
    return "\n".join([header, separator, *rows])


processed_inventory = _file_inventory(PROCESSED_DIR, "*.csv")
table_inventory = _file_inventory(TABLES_DIR, "*.csv")
figure_inventory = _file_inventory(FIGURES_DIR, "*.png")
summary_inventory = _file_inventory(STEP_SUMMARIES_DIR, "*.md")

artifact_inventory = pd.concat(
    [
        processed_inventory.assign(category="processed_data"),
        table_inventory.assign(category="report_table"),
        figure_inventory.assign(category="report_figure"),
        summary_inventory.assign(category="step_summary"),
    ],
    ignore_index=True,
)
artifact_inventory.to_csv(TABLES_DIR / "artifact_inventory_final.csv", index=False)

split_summary_table = pd.read_csv(TABLES_DIR / "split_summary_v1.csv")
modeling_header = pd.read_csv(PROCESSED_DIR / "modeling_table_last4_regular.csv", nrows=0)
modeling_rows = sum(1 for _ in open(PROCESSED_DIR / "modeling_table_last4_regular.csv", encoding="utf-8")) - 1

key_model_cols = ["model_label", "feature_set", "validity", "n_features", "mae", "rmse", "r2", "notes"]
final_md_table = _markdown_table(final_comparison[key_model_cols])
split_md_table = _markdown_table(split_summary_table)
starter_md_table = _markdown_table(starter_results.sort_values("mae").head(8))
pca_md_table = _markdown_table(pca_results.sort_values("validation_mae").head(8))
starter_pca_md_table = _markdown_table(starter_pca_results.sort_values("validation_mae").head(8))
isolation_md_table = _markdown_table(isolation_forest_results.sort_values("validation_mae").head(8))

libraries = [
    ("pandas", "DataFrame loading, joins, groupby summaries, CSV outputs."),
    ("numpy", "Numeric arrays, missing values, vectorized target and metric calculations."),
    ("matplotlib", "Static EDA and model diagnostic figures."),
    ("scikit-learn", "Ridge, HistGradientBoostingRegressor, RandomForestRegressor, IsolationForest, SimpleImputer, StandardScaler, PCA, permutation importance."),
    ("pathlib", "Stable filesystem paths."),
    ("dataclasses", "Small ValidationData container."),
]

library_md = "\n".join([f"- `{name}`: {reason}" for name, reason in libraries])
function_names = []
for cell in globals().copy().values():
    pass

summary = f"""# Current Project Summary

## Goal

Predict player-game fantasy basketball points with a time-aware supervised learning workflow.

Target formula:

`fantasy_points = points + 1.2 * reboundsTotal + 1.5 * assists + 3 * steals + 3 * blocks - turnovers`

The prediction row grain is one player in one NBA regular-season game. Current-game box-score outcomes are not valid model features. Outcome fields can only enter through shifted historical rolling features.

## Course-Aligned Workflow

This notebook follows the course workflow with data collection, preparation, cleansing checks, EDA, feature engineering, train/validation/test split, modeling, validation comparison, dimensionality reduction, tuning, interpretability, and final recommendation.

Intentional deviations from a generic supervised-learning template:

- Chronological split replaces random split because future games must not train models for earlier games.
- Expanding time cross-validation replaces random CV for the same reason.
- The 2025 test season is held back until the model choice is frozen.
- Deployment is not included because this is an academic modeling notebook, not a production app.

## Data Scope And Split

This final run uses NBA seasons `2021`, `2022`, `2023`, `2024`, and `2025`.

Split column: `{SPLIT_COL}`

{split_md_table}

The 2021 team-context issue was repaired by filling missing `TeamStatisticsExtended.gameType` values from `Games.csv` by `gameId` before regular-season filtering.

## Main Outputs

Modeling table rows: `{modeling_rows}`  
Modeling table columns: `{len(modeling_header.columns)}`

## Final Model Comparison

{final_md_table}

## PCA Compression Results

PCA candidates came from broad legal pre-lineup features that were not already in the compact scoring feature set. Candidate ranking, missingness filtering, imputation, scaling, and PCA fitting used train rows only.

{pca_md_table}

## Starter-Focused Models

Starter models are now a main comparison family:

- `pre_lineup_starter_history_roll10`: valid before lineup news because it uses shifted starter history, including `starter_minutes_roll_10`.
- `lineup_aware_starter_flag_roll10`: valid once starting lineup is known because it uses `current_is_starter` and shifted `starter_minutes_roll_10`.
- Direct current-game `starter_minutes` is not used in model comparisons.

{starter_md_table}

## Starter PCA Results

{starter_pca_md_table}

## Isolation Forest Anomaly Results

{isolation_md_table}

## Libraries Used

{library_md}

## Important Parameters

- Seasons: `{LAST4_SEASON_STARTS}`
- Target column: `{TARGET_COL}`
- Split column: `{SPLIT_COL}`
- PCA top candidate cap: `{PCA_TOP_N_CANDIDATES}`
- PCA missingness cap: `{PCA_MISSINGNESS_MAX:.0%}`
- PCA component grid: `{PCA_COMPONENT_GRID}` plus the component count needed for 95% train explained variance
- Isolation Forest contamination: `{ISOLATION_CONTAMINATION:.0%}`
- Isolation Forest estimators: `{ISOLATION_N_ESTIMATORS}`
- Random state: `{MODEL_RANDOM_STATE}`
- Main metrics: MAE, RMSE, R2

## Thinking Process

The notebook first builds a legal modeling table with shifted rolling features because the core risk is same-game leakage. It then compares broad and compact feature sets to check whether many team/opponent features help beyond player history. Starter uncertainty is handled as its own model family because role and minutes are the main basketball-specific uncertainty. PCA is added as a controlled compression experiment for many legal unused features. Isolation Forest is added as a train-only anomaly/reliability experiment. HGB tuning is applied only after the feature families are compared. Direct current-game `starter_minutes` is excluded from model comparisons and is used only to create shifted rolling history.

## Artifact Inventory

- Processed datasets: `{len(processed_inventory)}`
- Report tables: `{len(table_inventory)}`
- Report figures: `{len(figure_inventory)}`
- Step summaries: `{len(summary_inventory)}`

See `reports/tables/artifact_inventory_final.csv` for the full file list.
"""

(STEP_SUMMARIES_DIR / "current_project_summary.md").write_text(summary, encoding="utf-8")
(STEP_SUMMARIES_DIR / "artifact_inventory.md").write_text(
    "# Artifact Inventory\n\n" + _markdown_table(artifact_inventory),
    encoding="utf-8",
)

display(artifact_inventory.groupby("category").size().reset_index(name="files"))
print("Wrote current project summary and artifact inventory.")


,category,files
0,processed_data,3
1,report_figure,68
2,report_table,90
3,step_summary,16


Wrote current project summary and artifact inventory.


## 19. Final GitHub Package And Presentation Materials

This section writes the final README, requirements file, NotebookLM briefing, and presentation outline into `runs/final/`.


In [37]:

requirements_text = """pandas>=2.2.0
numpy>=1.26.0
scikit-learn>=1.4.0
matplotlib>=3.8.0
seaborn>=0.13.0
ipykernel>=6.29.0
kagglehub>=1.0.1
xgboost>=2.0.0
lightgbm>=4.3.0
catboost>=1.2.7
tqdm>=4.66.0
pyarrow>=24.0.0
"""
(RUN_DIR / "requirements.txt").write_text(requirements_text, encoding="utf-8")

best_validation = final_validation_ranking.iloc[0]
test_row = final_test_evaluation.iloc[0]
readme_text = f"""# NBA Fantasy Points Prediction Project

This repository package contains a supervised machine-learning project for predicting NBA player-game fantasy points.

## Data

The notebook downloads the raw data directly from Kaggle using `kagglehub`. Internet access is required on the first run. If Kaggle requests authentication, configure a Kaggle API token before running the notebook.

## Objective

Predict one player's fantasy points for one regular-season NBA game before the game is played.

Target formula:

```text
fantasy_points = points + 1.2 * reboundsTotal + 1.5 * assists + 3 * steals + 3 * blocks - turnovers
```

## Split Design

The project uses a chronological split:

```text
Train:      2021, 2022, 2023 seasons
Validation: 2024 season
Test:       2025 season
```

The 2025 test season is used once after model selection.

## Workflow

The final notebook follows the course workflow: data preparation, table unification, cleansing checks, EDA, feature engineering, feature selection, modeling, tuning, final evaluation, and reporting.

## Leakage Control

Direct current-game outcome fields are not valid model features. In particular, direct `starter_minutes` is treated only as a leakage diagnostic because it includes actual minutes from the game being predicted.

## Best Validation Result

Best valid validation row:

```text
{best_validation['model_label']} / {best_validation['feature_set']} / {best_validation['model']}
MAE  = {best_validation['mae']:.4f}
RMSE = {best_validation['rmse']:.4f}
R2   = {best_validation['r2']:.4f}
```

## Final 2025 Test Result

The selected controlled final model was evaluated once on 2025:

```text
Feature set = {test_row['feature_set']}
Model       = {test_row['model']}
MAE         = {test_row['test_mae']:.4f}
RMSE        = {test_row['test_rmse']:.4f}
R2          = {test_row['test_r2']:.4f}
```

## How To Run

Create an environment from `requirements.txt`, then run the notebook. It downloads the Kaggle dataset automatically:

```text
runs/final/notebooks/NBA_ML_project.ipynb
```

All final outputs are written under `runs/final/reports/`.

## GitHub Notes

Do not upload the regenerated Kaggle cache, processed modeling tables, or large prediction-dump CSV files. They are recreated by the notebook and are ignored by `.gitignore` because several are larger than GitHub's normal file limits.
"""
(RUN_DIR / "README.md").write_text(readme_text, encoding="utf-8")

brief_text = f"""# NotebookLM Project Brief

## Project Goal
Predict NBA player-game fantasy points using a clean time-aware supervised-learning workflow.

## Key Story
The project started with broad data exploration, leakage checks, rolling feature engineering, and several model families. An alternative notebook showed below-7 MAE, but the improvement came from direct current-game `starter_minutes`, which is not available before prediction time. The final project keeps that result as a leakage diagnostic and evaluates the valid ideas under the clean split.

## Data And Split
Train seasons are 2021-2023, validation is 2024, and test is 2025. The test season is used once after model selection.

## Main Experiments
- Baselines and broad feature models
- Starter-role models
- Expanding time cross-validation
- PCA feature compression
- Isolation Forest anomaly features
- XGBoost, LightGBM, CatBoost, ExtraTrees, and RandomForest controlled comparisons

## Best Validation Result
{best_validation['model_label']} using `{best_validation['feature_set']}` with MAE {best_validation['mae']:.4f}.

## Final Test Result
Selected model `{test_row['model']}` on `{test_row['feature_set']}` reached test MAE {test_row['test_mae']:.4f}, RMSE {test_row['test_rmse']:.4f}, and R2 {test_row['test_r2']:.4f}.

## Presentation Message
The most important lesson is that sports prediction is highly sensitive to role/minutes information. Actual minutes create excellent but invalid results; shifted starter history and lineup-aware starter flags provide valid alternatives.
"""
(PRESENTATION_DIR / "notebooklm_project_brief.md").write_text(brief_text, encoding="utf-8")
(STEP_SUMMARIES_DIR / "notebooklm_project_brief.md").write_text(brief_text, encoding="utf-8")

slide_outline = f"""# Presentation Slide Outline

1. Project objective and fantasy-points target
2. Dataset and row grain: player-game regular-season rows
3. Course workflow and chronological split
4. Data preparation and 2021 team-context repair
5. EDA: target distribution, positions, starters, minutes, outliers
6. Leakage rules and why current-game minutes are invalid
7. Feature engineering: shifted rolling stats, starter history, opponent context, momentum
8. Baselines and first valid model comparisons
9. Alternative modeling approach: strong below-7 result, but leakage diagnostic only
10. Valid corrected model ideas: XGBoost, LightGBM, CatBoost, ExtraTrees, RandomForest
11. PCA and Isolation Forest experiments
12. Final validation ranking
13. One-time 2025 test result: `{test_row['model']}` / `{test_row['feature_set']}` MAE {test_row['test_mae']:.4f}
14. Feature importance and interpretation
15. Main conclusion and next improvements
"""
(PRESENTATION_DIR / "presentation_slide_outline.md").write_text(slide_outline, encoding="utf-8")
(STEP_SUMMARIES_DIR / "presentation_slide_outline.md").write_text(slide_outline, encoding="utf-8")

final_package_inventory = []
for folder in [RUN_DIR, PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, STEP_SUMMARIES_DIR, PRESENTATION_DIR, NOTEBOOKS_DIR]:
    for path in sorted(folder.glob("*")):
        if path.is_file():
            final_package_inventory.append({"path": str(path.relative_to(RUN_DIR)).replace("\\", "/"), "bytes": path.stat().st_size})
final_package_inventory = pd.DataFrame(final_package_inventory).sort_values("path").reset_index(drop=True)
final_package_inventory.to_csv(TABLES_DIR / "final_package_inventory.csv", index=False)

display(final_package_inventory.head(30))
print(f"Final package README: {RUN_DIR / 'README.md'}")
print(f"NotebookLM brief: {PRESENTATION_DIR / 'notebooklm_project_brief.md'}")
print(f"Presentation outline: {PRESENTATION_DIR / 'presentation_slide_outline.md'}")


,path,bytes
0,README.md,2082
1,data/processed/modeling_table_last4_regular.csv,968815383
2,data/processed/player_game_source_last4_regula...,138178737
3,data/processed/profile_extended_driver_10_mode...,150368946
4,notebooks/NBA_ML_project.ipynb,364365
5,presentation/notebooklm_project_brief.md,1517
6,presentation/presentation_slide_outline.md,924
7,reports/figures/eda_actual_numMinutes_vs_fp.png,379005
8,reports/figures/eda_current_is_starter_roll_10...,103216
9,reports/figures/eda_fantasy_points_momentum_5_...,392694


Final package README: C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\runs\final\README.md
NotebookLM brief: C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\runs\final\presentation\notebooklm_project_brief.md
Presentation outline: C:\Users\zivdi\fantasy_basketball_fp_starter\NBA_ML_PROJ_2\runs\final\presentation\presentation_slide_outline.md
